# 🦆 BirdCLEF+2026 | EoS9 + OOF-Gated PCEN

This notebook keeps the public EoS9 anchor as the control path and exposes optional sidecar switches for controlled experiments.

The default control state now mirrors the refreshed local EoS9(2) public notebook; BirdNET and PCEN sidecars remain opt-in switches for follow-up tests.

The anchor is a compact three-branch ensemble:

```text
p_anchor_raw = w_y * p_yukiZ
             + w_p * p_poweropt_pssm
             + w_s * p_poweropt_sz
```

The small PSSM branch is saved as an intermediate from Karnakbayev's PowerOptimization path. The dominant branch runs the PowerOptimization-style ProtoSSM + distilled SED rank pipeline:

```text
z_sz = G_prior( w_proto * R(p_proto) + w_sed * R(p_sed) )
p_sz = G_post( z_sz )
```

`R` is class-wise percentile rank. `G_prior` applies ecological site/hour priors and residual correction. `G_post` applies file-confidence scaling, rank-aware scaling, temporal smoothing, continuity gates, rare-class damping, and clipping.

The final anchor step applies light taxonomy smoothing after the direct blend:

```text
p_tax = (1 - alpha_genus) * p + alpha_genus * mean_genus(p)
p_out = (1 - alpha_class) * p_tax + alpha_class * mean_class(p_tax)
```

## Experimental Sidecars

The optional PCEN/ConvNeXt and BirdNET branches are not global ensemble members. They are used as masked rank corrections after the taxonomy-smoothed anchor is written:

```text
A = R(p_anchor)
S = R(p_sidecar)
p_final = A + W_class * M * (S - A)
```

`M` restricts the cells where a sidecar can move the anchor. `W_class` and the perturbation budget limit the total movement. This keeps the sidecar tests interpretable: the anchor logic stays fixed, and score changes come from the gated correction.

By default, both sidecar families are disabled. Enable BirdNET and PCEN separately for clean attribution. Inside the PCEN family, `dual_exp002b_exp002` can apply the 5-second and 10-second custom sidecars sequentially.

## References

| Reference | Used here as |
|---|---|
| F.A.Nina, [`birdclef-2026-eos-9`](https://www.kaggle.com/code/nina2025/birdclef-2026-eos-9) | primary EoS9 anchor structure |
| Karnakbayev Artur, [`power-optimization`](https://www.kaggle.com/code/karnakbaevarthur/power-optimization) | PowerOptimization branch and output validation style |
| yukiZ, [`Perch + ProtoSSM + ResSSM`](https://www.kaggle.com/code/hideyukizushi/bird26-reproduce-perch-protossm-resssm-inf-train/notebook) | low-weight diversity branch |
| Tucker Arrants, [`BC2026 Distilled SED`](https://www.kaggle.com/code/tuckerarrants/bc2026-distilled-sed) | distilled SED ONNX branch |
| Custom `exp002` / `exp002b` sidecar assets | optional weak-audio PCEN/logmel ConvNeXt sidecars |


In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/BirdCLEF26_sidecar3.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))

## Why Taxonomy Smoothing

The direct blend ranks each species independently. Taxonomy smoothing adds a small amount of shared evidence among related labels.

For a genus group `g`, the update is:

```text
p_c <- (1 - alpha_genus) * p_c + alpha_genus * mean_{j in g}(p_j)
```

For a broader class group `k`, the update is:

```text
p_c <- (1 - alpha_class) * p_c + alpha_class * mean_{j in k}(p_j)
```

This is a weak prior over related species, not a replacement for the model prediction. It can lift plausible sibling taxa while keeping most of the original branch ranking intact.

A reasonable reading of [F.A.Nina's EoS9](https://www.kaggle.com/code/nina2025/birdclef-2026-eos-9) structure is that taxonomy smoothing is meant to correct small rank misses among biologically related labels after the dominant EoS9 PowerOptimization branch has already produced a good per-class ordering. [Karnakbayev Artur's PowerOptimization](https://www.kaggle.com/code/karnakbaevarthur/power-optimization) branch contributes the calibrated acoustic/rank signal; the taxonomy step adds a small ecological sharing term on top of it.

The optional sidecar, if enabled, is still budgeted by rank movement:

```text
D = mean( abs(p_final - A) )
```

so it cannot move the anchor beyond the configured perturbation budget.


## Controls

Edit the first code cell. The score-changing knobs are grouped there; the overview tables printed below show the current values.

### Anchor Weights

`RUN_MODE` selects one anchor preset. Each preset assigns the outer branch weights:

```text
p_anchor_raw = YUKIZ_BLEND_WEIGHT       * p_yukiZ
             + POWEROPT_PSSM_BLEND_WEIGHT * p_poweropt_pssm
             + POWEROPT_SZ2_BLEND_WEIGHT  * p_poweropt_main
```

The notebook asserts that these outer weights sum to one. Inside the dominant Karnakbayev/PowerOptimization branch, the ProtoSSM and SED rank balance is controlled separately:

```text
z = PROTO_RANK_WEIGHT * R(p_proto) + SED_RANK_WEIGHT * R(p_sed)
SED_RANK_WEIGHT = 1 - PROTO_RANK_WEIGHT
```

Taxonomy smoothing is applied after the direct branch blend, controlled by `TAX_GENUS_ALPHA` and `TAX_CLASS_ALPHA`.

### Sidecar Gates

BirdNET and PCEN/ConvNeXt are optional post-anchor corrections. They do not change the anchor registry. When enabled, a sidecar moves only masked rank cells:

```text
A = R(p_anchor)
S = R(p_sidecar)
B = A + W_class * M * (S - A)
```

For PCEN/ConvNeXt, `W_class` comes from the packaged OOF gate CSV. A class is open only when its `gate_weight` is greater than zero. The gate CSV is built offline by comparing sidecar OOF predictions against the current anchor replay OOF. In short, a class opens only if a masked blend improves the anchor OOF AUC after class/group shrinkage checks.

The runtime knobs are:

| Knob | Meaning |
|---|---|
| `RUN_MODE` | active anchor preset |
| `YUKIZ_BLEND_WEIGHT`, `POWEROPT_PSSM_BLEND_WEIGHT`, `POWEROPT_SZ2_BLEND_WEIGHT` | outer anchor weights selected by `RUN_MODE` |
| `PROTO_RANK_WEIGHT` | dominant branch Proto/SED rank balance; SED is derived as `1 - this` |
| `POWEROPT_PSSM_PRIOR_LAMBDA`, `POWEROPT_SZ2_TEST_PRIOR_LAMBDA` | ecological prior strengths inside the Karnakbayev branches |
| `POWEROPT_PSSM_RANK_AWARE_POWER`, `POWEROPT_SZ2_RANK_AWARE_POWER` | file-level rank-aware scaling powers |
| `TAX_GENUS_ALPHA`, `TAX_CLASS_ALPHA` | final taxonomy smoothing strengths |
| `TAX_SMOOTHING_REQUIRE` | fail if `taxonomy.csv` is missing while taxonomy smoothing is selected |
| `RUN_BIRDNET_SIDECAR` | optional BirdNET rank correction |
| `RUN_EXP002_SIDECAR` | optional PCEN/ConvNeXt sidecar correction |
| `SIDECAR_EXP002_VARIANT` | active sidecar asset preset: `exp002b_5s`, `exp002_10s`, or `dual_exp002b_exp002` |
| `SIDECAR_EXP002_REQUIRE` | fail the real-test run if the sidecar is computed but rejected |
| `SIDECAR_EXP002_FOLDS` | fold checkpoints used by sidecar inference |
| `SIDECAR_EXP002_WEIGHT_CAP`, `SIDECAR_EXP002_D_BUDGET` | sidecar perturbation limits |
| `SIDECAR_EXP002_USE_OOF_GATE`, `SIDECAR_EXP002_GATE_CSV` | optional class-wise OOF reliability gate |

To reproduce the EoS9 anchor path, keep both optional sidecars disabled. To test both PCEN sidecars in one run, set `RUN_EXP002_SIDECAR = True` and `SIDECAR_EXP002_VARIANT = "dual_exp002b_exp002"`.


In [ ]:
# Front-of-notebook controls.
import time
NOTEBOOK_START_TIME = time.time()
KAGGLE_LIMIT_SEC = 90 * 60
FINAL_RESERVE_SEC = 180

RUN_MODE = "eos9_tax"
# Options:
#   "eos9_tax"           -> yukiZ + Karnakbayev PSSM + Karnakbayev EoS9 branch + taxonomy smoothing
#   "eos9_no_pssm"       -> yukiZ + Karnakbayev EoS9 branch + taxonomy smoothing
#   "poweropt_eos9_only" -> Karnakbayev EoS9 branch with taxonomy smoothing

MODEL_YUKIZ = "yukiZ_Perch_ProtoSSM_ResSSM"
MODEL_POWEROPT_PSSM = "Karnakbayev_PowerOptimization_PSSM"
MODEL_POWEROPT_SZ2 = "Karnakbayev_PowerOptimization_EoS9"

# Backward-compatible alias used by the first Karnakbayev branch cell.
MODEL_POWEROPT = MODEL_POWEROPT_PSSM

YUKIZ_SUBM = "subm_yukiz_perch_proto_res.csv"
POWEROPT_PSSM_SUBM = "subm_karnakbayev_poweropt_pssm.csv"
POWEROPT_PSSM_FULL_SUBM = "subm_karnakbayev_poweropt_model51_full_unused.csv"
POWEROPT_SZ2_SUBM = "subm_karnakbayev_poweropt_sz2.csv"
POWEROPT_SZ2_CORE_SUBM = "subm_karnakbayev_poweropt_sz2_core.csv"

# Backward-compatible alias used by the first Karnakbayev branch cell.
POWEROPT_FULL_SUBM = POWEROPT_PSSM_FULL_SUBM

if RUN_MODE == "eos9_tax":
    # Matches the active EoS9(2) public anchor registry.
    YUKIZ_BLEND_WEIGHT = 0.011
    POWEROPT_PSSM_BLEND_WEIGHT = 0.022
    POWEROPT_SZ2_BLEND_WEIGHT = 0.967
elif RUN_MODE == "eos9_no_pssm":
    # Diagnostic mode: keep yukiZ fixed and move the PSSM share into the main branch.
    YUKIZ_BLEND_WEIGHT = 0.012
    POWEROPT_PSSM_BLEND_WEIGHT = 0.0
    POWEROPT_SZ2_BLEND_WEIGHT = 0.988
elif RUN_MODE == "poweropt_eos9_only":
    YUKIZ_BLEND_WEIGHT = 0.0
    POWEROPT_PSSM_BLEND_WEIGHT = 0.0
    POWEROPT_SZ2_BLEND_WEIGHT = 1.0
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

PROTO_RANK_WEIGHT = 0.600
SED_RANK_WEIGHT = 1.0 - PROTO_RANK_WEIGHT
POWEROPT_PSSM_INTERNAL_XSED = [0.60, 0.40]

POWEROPT_PSSM_PRIOR_LAMBDA = 0.65
POWEROPT_SZ2_TEST_PRIOR_LAMBDA = 0.75
POWEROPT_SZ2_TRAIN_PRIOR_LAMBDA = 0.65
POWEROPT_FILE_CONFIDENCE_POWER = 0.40
POWEROPT_PSSM_RANK_AWARE_POWER = 0.65
POWEROPT_SZ2_RANK_AWARE_POWER = 0.60
POWEROPT_DELTA_SMOOTH_ALPHA = 0.20
CORRECTION_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

# Backward-compatible aliases used inside the first Karnakbayev branch cell.
POWEROPT_PRIOR_LAMBDA = POWEROPT_PSSM_PRIOR_LAMBDA
POWEROPT_RANK_AWARE_POWER = POWEROPT_PSSM_RANK_AWARE_POWER

TAX_GENUS_ALPHA = 0.15
TAX_CLASS_ALPHA = 0.05
TAX_SMOOTHING_REQUIRE = True

FAKE_PROTO_PULL = 0.08
PROTO_CONTINUITY_PULL = 0.15
SED_SPIKE_PULL = 0.12

RUN_SED_ONCE = True

# -----------------------------------------------------------------------------
# Sidecar 1: BirdNET v2.4
# -----------------------------------------------------------------------------
# BirdNET is not a global blend. It is a masked rank-space correction of the
# current submission. The high-impact knobs are:
#   RUN_BIRDNET_SIDECAR         -> enable/disable BirdNET
#   BIRDNET_MAPPED_WEIGHT_CAP   -> cap for covered Aves classes already mapped by Perch
#   BIRDNET_UNMAPPED_WEIGHT_CAP -> cap for covered Aves classes where Perch is weaker
#   BIRDNET_*_D_BUDGET          -> max group-wise mean rank movement
#   BIRDNET_*_TOPK / *_TAU      -> where BirdNET is allowed to move the anchor
# If the TFLite model is not attached, the cell skips and keeps the anchor unchanged.
RUN_BIRDNET_SIDECAR = True
BIRDNET_MODEL_ROOT = "/kaggle/input/models/shadiakiki1/birdnet-analyzer/tflite/birdnet_global_6k_v2.4_model_fp32-1/3"
BIRDNET_SIDECAR_SUBM = "subm_birdnet_v24.csv"
BIRDNET_DIAGNOSTICS_CSV = "birdnet_sidecar_diagnostics.csv"

# Conservative BirdNET sidecar preset. BirdNET affects only covered Aves classes.
# The caps below are automatically shrunk if the perturbation budgets are exceeded.
BIRDNET_MAPPED_WEIGHT_CAP = 0.015
BIRDNET_UNMAPPED_WEIGHT_CAP = 0.060
BIRDNET_NON_BIRD_WEIGHT_CAP = 0.000
BIRDNET_COVERAGE_MIN = 0.01

BIRDNET_MAPPED_ANCHOR_TOPK = 32
BIRDNET_MAPPED_SIDE_TOPK = 16
BIRDNET_MAPPED_TAU = 0.70

BIRDNET_UNMAPPED_ANCHOR_TOPK = 48
BIRDNET_UNMAPPED_SIDE_TOPK = 32
BIRDNET_UNMAPPED_TAU = 0.55

BIRDNET_TOTAL_D_BUDGET = 0.006
BIRDNET_MAPPED_D_BUDGET = 0.002
BIRDNET_UNMAPPED_D_BUDGET = 0.006
BIRDNET_NON_BIRD_D_BUDGET = 0.0005
BIRDNET_MIN_TOP3_OVERLAP = 0.75
BIRDNET_MIN_TOP10_OVERLAP = 0.80
BIRDNET_MAX_ACTIVE_FRACTION = 0.25
BIRDNET_DRYRUN_FILES = 1

# -----------------------------------------------------------------------------
# Sidecar 2: custom PCEN/ConvNeXt
# -----------------------------------------------------------------------------
# The active PCEN sidecar variant is selected below. It is disabled by default for the EoS9 anchor run.
# SIDECAR_EXP002_TIMEOUT_SEC is only the current sidecar subprocess watchdog.
# Raise it when increasing folds or using the 10-second variant; it does not extend Kaggle runtime.
# "dual_exp002b_exp002" applies exp002b first, then exp002, each with its own asset and OOF gate.
RUN_EXP002_SIDECAR = True
SIDECAR_EXP002_REQUIRE = True
SIDECAR_EXP002_VARIANT = "exp002b_5s"  # options: "exp002b_5s", "exp002_10s", "dual_exp002b_exp002"

SIDECAR_EXP002_VARIANT_SPECS = {
    "exp002b_5s": {
        "SIDECAR_EXP002_LABEL": "exp002b",
        "SIDECAR_EXP002_ROOT": "/kaggle/input/datasets/pilkwang/birdclef26-sidecar-exp002b-5s-weakaudio",
        "SIDECAR_EXP002_SUBM": "subm_sidecar_exp002b.csv",
        "SIDECAR_EXP002_DIAGNOSTICS_CSV": "sidecar_exp002b_diagnostics.csv",
        "SIDECAR_EXP002_ANCHOR_BEFORE_CSV": "submission_before_exp002b_sidecar.csv",
        "SIDECAR_EXP002_FOLDS": [0],
        "SIDECAR_EXP002_TIMEOUT_SEC": 600,
        "SIDECAR_EXP002_GATE_CSV": "exp002b_oof_gate.csv",
        "SIDECAR_EXP002_EXPECTED_CONTEXT_SECONDS": 5.0,
        "SIDECAR_EXP002_EXPECTED_TARGET_SECONDS": 5.0,
        "SIDECAR_EXP002_EXPECTED_IMAGE_TIME": 256,
    },
    "exp002_10s": {
        "SIDECAR_EXP002_LABEL": "exp002",
        "SIDECAR_EXP002_ROOT": "/kaggle/input/datasets/pilkwang/birdclef26-sidecar-exp002-10s-weakaudio",
        "SIDECAR_EXP002_SUBM": "subm_sidecar_exp002.csv",
        "SIDECAR_EXP002_DIAGNOSTICS_CSV": "sidecar_exp002_diagnostics.csv",
        "SIDECAR_EXP002_ANCHOR_BEFORE_CSV": "submission_before_exp002_sidecar.csv",
        "SIDECAR_EXP002_FOLDS": [0],
        "SIDECAR_EXP002_TIMEOUT_SEC": 900,
        "SIDECAR_EXP002_GATE_CSV": "exp002_oof_gate.csv",
        "SIDECAR_EXP002_EXPECTED_CONTEXT_SECONDS": 10.0,
        "SIDECAR_EXP002_EXPECTED_TARGET_SECONDS": 5.0,
        "SIDECAR_EXP002_EXPECTED_IMAGE_TIME": 384,
    },
}

if SIDECAR_EXP002_VARIANT == "dual_exp002b_exp002":
    SIDECAR_EXP002_ACTIVE_VARIANTS = ["exp002b_5s", "exp002_10s"]
elif SIDECAR_EXP002_VARIANT in SIDECAR_EXP002_VARIANT_SPECS:
    SIDECAR_EXP002_ACTIVE_VARIANTS = [SIDECAR_EXP002_VARIANT]
else:
    raise ValueError(f"Unknown SIDECAR_EXP002_VARIANT: {SIDECAR_EXP002_VARIANT}")


def _load_exp002_variant_settings(variant):
    spec = SIDECAR_EXP002_VARIANT_SPECS[str(variant)]
    globals().update(spec)
    return spec

# Initialize the public per-variant variables for overview cells and single-variant mode.
_load_exp002_variant_settings(SIDECAR_EXP002_ACTIVE_VARIANTS[0])

SIDECAR_EXP002_DEVICE = "cpu"
SIDECAR_EXP002_BATCH_SIZE = 16
SIDECAR_EXP002_FORCE_INFER = True

SIDECAR_EXP002_WEIGHT_CAP = 0.030
SIDECAR_EXP002_D_BUDGET = 0.004
SIDECAR_EXP002_ANCHOR_TOPK = 48
SIDECAR_EXP002_SIDE_TOPK = 32
SIDECAR_EXP002_TAU = 0.55
SIDECAR_EXP002_MIN_TOP3_OVERLAP = 0.75
SIDECAR_EXP002_MIN_TOP10_OVERLAP = 0.80
SIDECAR_EXP002_MAX_ACTIVE_FRACTION = 0.25

SIDECAR_EXP002_USE_OOF_GATE = True
SIDECAR_EXP002_GATE_MAX_WEIGHT = 0.030
SIDECAR_EXP002_GATE_REQUIRE_FILE = True

# Dual mode uses the same per-sidecar guards above and then checks total movement
# from the pre-sidecar anchor. Set to None to disable the final dual guard.
SIDECAR_EXP002_DUAL_TOTAL_D_BUDGET = 0.006
SIDECAR_EXP002_DUAL_DIAGNOSTICS_CSV = "sidecar_exp002_dual_diagnostics.csv"

assert 0.0 <= YUKIZ_BLEND_WEIGHT <= 1.0
assert 0.0 <= POWEROPT_PSSM_BLEND_WEIGHT <= 1.0
assert 0.0 <= POWEROPT_SZ2_BLEND_WEIGHT <= 1.0
assert abs((YUKIZ_BLEND_WEIGHT + POWEROPT_PSSM_BLEND_WEIGHT + POWEROPT_SZ2_BLEND_WEIGHT) - 1.0) < 1e-9
assert 0.0 <= PROTO_RANK_WEIGHT <= 1.0
assert abs((PROTO_RANK_WEIGHT + SED_RANK_WEIGHT) - 1.0) < 1e-9
assert POWEROPT_PSSM_PRIOR_LAMBDA >= 0.0
assert POWEROPT_SZ2_TEST_PRIOR_LAMBDA >= 0.0
assert POWEROPT_SZ2_TRAIN_PRIOR_LAMBDA >= 0.0
assert POWEROPT_FILE_CONFIDENCE_POWER >= 0.0
assert POWEROPT_PSSM_RANK_AWARE_POWER >= 0.0
assert POWEROPT_SZ2_RANK_AWARE_POWER >= 0.0
assert POWEROPT_DELTA_SMOOTH_ALPHA >= 0.0
assert len(CORRECTION_GRID) > 0 and all(w >= 0.0 for w in CORRECTION_GRID)
assert 0.0 <= TAX_GENUS_ALPHA <= 1.0
assert 0.0 <= TAX_CLASS_ALPHA <= 1.0
assert isinstance(TAX_SMOOTHING_REQUIRE, bool)
assert all(v in SIDECAR_EXP002_VARIANT_SPECS for v in SIDECAR_EXP002_ACTIVE_VARIANTS)
assert SIDECAR_EXP002_DUAL_TOTAL_D_BUDGET is None or SIDECAR_EXP002_DUAL_TOTAL_D_BUDGET >= 0.0

ACTIVE_MODELS = [name for name, weight in [
    (MODEL_YUKIZ, YUKIZ_BLEND_WEIGHT),
    (MODEL_POWEROPT_PSSM, POWEROPT_PSSM_BLEND_WEIGHT),
    (MODEL_POWEROPT_SZ2, POWEROPT_SZ2_BLEND_WEIGHT),
] if weight > 0.0]

MODEL_TABLE = [
    {
        "active": MODEL_YUKIZ in ACTIVE_MODELS,
        "name": MODEL_YUKIZ,
        "subm": YUKIZ_SUBM,
        "weight": YUKIZ_BLEND_WEIGHT,
        "xSED": [],
        "LB": "0.928",
        "source": "yukiZ / SGKF",
        "role": "low-weight pretrained Perch + ProtoSSM + ResidualSSM diversity",
    },
    {
        "active": MODEL_POWEROPT_PSSM in ACTIVE_MODELS,
        "name": MODEL_POWEROPT_PSSM,
        "subm": POWEROPT_PSSM_SUBM,
        "weight": POWEROPT_PSSM_BLEND_WEIGHT,
        "xSED": [],
        "LB": "0.949",
        "source": "Karnakbayev PowerOptimization via EoS9",
        "role": "small intermediate PSSM side branch saved before the full rank blend",
    },
    {
        "active": MODEL_POWEROPT_SZ2 in ACTIVE_MODELS,
        "name": MODEL_POWEROPT_SZ2,
        "subm": POWEROPT_SZ2_SUBM,
        "weight": POWEROPT_SZ2_BLEND_WEIGHT,
        "xSED": [PROTO_RANK_WEIGHT, SED_RANK_WEIGHT],
        "LB": "0.949",
        "source": "Karnakbayev PowerOptimization via EoS9",
        "role": "dominant ProtoSSM/SED rank branch with stronger test prior and rank-aware scaling",
    },
]

MODEL_REGISTRY = {m["name"]: m for m in MODEL_TABLE}

solutions = {
    "type_add": "TAX_SMOOTHING",
    "task1": "run SED once",
    "task2": {"save PowerOptimization PSSM as": POWEROPT_PSSM_SUBM},
    "Models": [
        {
            "Model": name,
            "subm": MODEL_REGISTRY[name]["subm"],
            "weight": MODEL_REGISTRY[name]["weight"],
            "xSED": MODEL_REGISTRY[name]["xSED"],
            "LB": MODEL_REGISTRY[name]["LB"],
        }
        for name in ACTIVE_MODELS
    ],
}


In [ ]:
import pandas as pd

MODEL_OVERVIEW = pd.DataFrame(MODEL_TABLE)[[
    "active", "name", "source", "weight", "xSED", "subm", "LB", "role"
]]
MODEL_OVERVIEW


In [ ]:
ANCHOR_OVERVIEW = pd.DataFrame([
    {"branch": "yukiZ", "weight": YUKIZ_BLEND_WEIGHT, "note": "Perch + ProtoSSM + ResidualSSM diversity"},
    {"branch": "Karnakbayev PowerOptimization PSSM", "weight": POWEROPT_PSSM_BLEND_WEIGHT, "note": "small intermediate PSSM side branch"},
    {"branch": "Karnakbayev PowerOptimization EoS9", "weight": POWEROPT_SZ2_BLEND_WEIGHT, "note": "dominant EoS9 prior/rank-aware ProtoSSM + SED branch"},
])
display(ANCHOR_OVERVIEW)

POSTPROC_OVERVIEW = pd.DataFrame([
    {"stage": "taxonomy smoothing", "genus_alpha": TAX_GENUS_ALPHA, "class_alpha": TAX_CLASS_ALPHA},
    {"stage": "PSSM branch prior", "prior_lambda": POWEROPT_PSSM_PRIOR_LAMBDA, "rank_power": POWEROPT_PSSM_RANK_AWARE_POWER},
    {"stage": "EoS9 test prior", "prior_lambda": POWEROPT_SZ2_TEST_PRIOR_LAMBDA, "rank_power": POWEROPT_SZ2_RANK_AWARE_POWER},
])
display(POSTPROC_OVERVIEW)

RECENT_WORKFLOW_OVERVIEW = pd.DataFrame([
    {"artifact": "anchor replay", "status": "prepared", "purpose": "emit train-soundscape predictions from the EoS9 anchor"},
    {"artifact": "anchor-delta OOF gates", "status": "optional", "purpose": "filter exp002/exp002b sidecar classes before masked rank correction"},
])
display(RECENT_WORKFLOW_OVERVIEW)

SIDECAR_VARIANT_OVERVIEW = pd.DataFrame([
    {
        "selected_mode": SIDECAR_EXP002_VARIANT,
        "variant": variant,
        "order": order,
        "label": SIDECAR_EXP002_VARIANT_SPECS[variant]["SIDECAR_EXP002_LABEL"],
        "enabled": RUN_EXP002_SIDECAR,
        "root": SIDECAR_EXP002_VARIANT_SPECS[variant]["SIDECAR_EXP002_ROOT"],
        "folds": str(SIDECAR_EXP002_VARIANT_SPECS[variant]["SIDECAR_EXP002_FOLDS"]),
        "timeout_sec": SIDECAR_EXP002_VARIANT_SPECS[variant]["SIDECAR_EXP002_TIMEOUT_SEC"],
        "context_sec": SIDECAR_EXP002_VARIANT_SPECS[variant]["SIDECAR_EXP002_EXPECTED_CONTEXT_SECONDS"],
        "target_sec": SIDECAR_EXP002_VARIANT_SPECS[variant]["SIDECAR_EXP002_EXPECTED_TARGET_SECONDS"],
        "image_time": SIDECAR_EXP002_VARIANT_SPECS[variant]["SIDECAR_EXP002_EXPECTED_IMAGE_TIME"],
        "gate_csv": SIDECAR_EXP002_VARIANT_SPECS[variant]["SIDECAR_EXP002_GATE_CSV"],
        "oof_gate": RUN_EXP002_SIDECAR and SIDECAR_EXP002_USE_OOF_GATE,
    }
    for order, variant in enumerate(SIDECAR_EXP002_ACTIVE_VARIANTS, start=1)
])
display(SIDECAR_VARIANT_OVERVIEW)

SIDECAR_OVERVIEW = pd.DataFrame([
    {
        "sidecar": "BirdNET v2.4",
        "enabled": RUN_BIRDNET_SIDECAR,
        "where": "covered Aves only; mapped/unmapped caps",
        "weight_cap": f"mapped={BIRDNET_MAPPED_WEIGHT_CAP}, unmapped={BIRDNET_UNMAPPED_WEIGHT_CAP}",
        "D_budget": f"total={BIRDNET_TOTAL_D_BUDGET}, mapped={BIRDNET_MAPPED_D_BUDGET}, unmapped={BIRDNET_UNMAPPED_D_BUDGET}",
        "diagnostics": BIRDNET_DIAGNOSTICS_CSV,
    },
    {
        "sidecar": "PCEN/ConvNeXt",
        "enabled": RUN_EXP002_SIDECAR,
        "where": "OOF-gated masked rank correction" if SIDECAR_EXP002_USE_OOF_GATE else "masked rank correction",
        "weight_cap": SIDECAR_EXP002_WEIGHT_CAP,
        "D_budget": SIDECAR_EXP002_D_BUDGET,
        "gate_csv": SIDECAR_EXP002_GATE_CSV,
        "diagnostics": ", ".join(SIDECAR_EXP002_VARIANT_SPECS[v]["SIDECAR_EXP002_DIAGNOSTICS_CSV"] for v in SIDECAR_EXP002_ACTIVE_VARIANTS),
        "timeout_sec": ", ".join(str(SIDECAR_EXP002_VARIANT_SPECS[v]["SIDECAR_EXP002_TIMEOUT_SEC"]) for v in SIDECAR_EXP002_ACTIVE_VARIANTS),
        "dual_total_D_budget": SIDECAR_EXP002_DUAL_TOTAL_D_BUDGET,
    },
])
SIDECAR_OVERVIEW


In [ ]:
_ensemble_models = [model['Model' ] for model in solutions['Models']]
_files_subm      = [model['subm'  ] for model in solutions['Models']]
_weights         = [model['weight'] for model in solutions['Models']]
_xsed            = [model['xSED'  ] for model in solutions['Models']]
_lbs             = [model['LB'    ] for model in solutions['Models']]

_runSED_once = RUN_SED_ONCE
print('Active models:', _ensemble_models)
print('Submission files:', _files_subm)
print('Weights:', _weights, 'sum=', sum(_weights))
print('Run SED once:', _runSED_once)
print('Karnakbayev branch knobs:', {
    'pssm_prior_lambda': POWEROPT_PSSM_PRIOR_LAMBDA,
    'sz2_test_prior_lambda': POWEROPT_SZ2_TEST_PRIOR_LAMBDA,
    'sz2_train_prior_lambda': POWEROPT_SZ2_TRAIN_PRIOR_LAMBDA,
    'file_confidence_power': POWEROPT_FILE_CONFIDENCE_POWER,
    'pssm_rank_aware_power': POWEROPT_PSSM_RANK_AWARE_POWER,
    'sz2_rank_aware_power': POWEROPT_SZ2_RANK_AWARE_POWER,
    'delta_smooth_alpha': POWEROPT_DELTA_SMOOTH_ALPHA,
    'correction_grid': CORRECTION_GRID,
    'taxonomy_genus_alpha': TAX_GENUS_ALPHA,
    'taxonomy_class_alpha': TAX_CLASS_ALPHA,
})


## yukiZ Diversity Branch

Writes `subm_yukiz_perch_proto_res.csv`. It contributes whenever its registry weight is positive: small residual diversity from a pretrained Perch/ProtoSSM/ResidualSSM path.


In [ ]:
if MODEL_YUKIZ in _ensemble_models:

    _file_name_submission = YUKIZ_SUBM
    
    # Install runtime wheels from attached datasets when missing.
    # Sidecar-only startup packages are handled outside these anchor branches.
    import subprocess
    import sys
    from pathlib import Path

    INPUT_ROOT = Path("/kaggle/input")

    def find_optional_wheel(pattern):
        hits = sorted(INPUT_ROOT.rglob(pattern))
        return hits[0] if hits else None

    def install_optional_wheel(pattern):
        whl = find_optional_wheel(pattern)
        if whl is None:
            return False
        print("Installing bundled wheel:", whl)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(whl)], check=True)
        return True

    import random
    import os
    import torch
    import numpy as np

    try:
        import onnxruntime as ort
        _ONNX_AVAILABLE = True
        print("ONNX Runtime available")
    except ImportError:
        install_optional_wheel("onnxruntime-1.24.4-*.whl")
        try:
            import onnxruntime as ort
            _ONNX_AVAILABLE = True
            print("ONNX Runtime available")
        except ImportError:
            _ONNX_AVAILABLE = False
            print("ONNX Runtime unavailable after bundled wheel search")

    try:
        import tensorflow as tf
    except ImportError:
        install_optional_wheel("tensorboard-2.20.0-*.whl")
        install_optional_wheel("tensorflow-2.20.0-*.whl")
        import tensorflow as tf
    
    def seed_everything(seed=42):
        random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        tf.random.set_seed(seed)
    
    seed_everything(1891)
    
    _ONNX_AVAILABLE
    
    # Cell 1 — Mode switch
    MODE = "submit" 
    
    assert MODE in {"train", "submit"}
    
    print("MODE =", MODE)
    
    # Cell 2 — Imports and run config
    import os
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    
    import gc
    import json
    import re
    import time
    import warnings
    from collections import defaultdict
    from pathlib import Path
    
    import numpy as np
    import pandas as pd
    import soundfile as sf
    import tensorflow as tf
    
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    
    from sklearn.decomposition import PCA
    from sklearn.linear_model import LogisticRegression
    from sklearn.neural_network import MLPClassifier
    from sklearn.metrics import roc_auc_score
    try:
        from lightgbm import LGBMClassifier
        _LGBM_AVAILABLE = True
    except ImportError:
        _LGBM_AVAILABLE = False
    from sklearn.model_selection import GroupKFold
    from sklearn.preprocessing import StandardScaler
    
    from tqdm.auto import tqdm
    
    warnings.filterwarnings("ignore")
    tf.experimental.numpy.experimental_enable_numpy_behavior()
    
    _WALL_START = time.time()
    
    def find_competition_dir():
        candidates = [
            Path("/kaggle/input/competitions/birdclef-2026"),
            Path("/kaggle/input/birdclef-2026"),
        ]
        for p in candidates:
            if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
                print("Using competition data:", p)
                return p
        for p in Path("/kaggle/input").rglob("sample_submission.csv"):
            parent = p.parent
            if (parent / "taxonomy.csv").exists() and (parent / "train_soundscapes_labels.csv").exists():
                print("Using competition data:", parent)
                return parent
        raise FileNotFoundError("BirdCLEF competition data directory not found.")

    BASE = find_competition_dir()
    MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
    
    SR = 32000
    WINDOW_SEC = 5
    WINDOW_SAMPLES = SR * WINDOW_SEC
    FILE_SAMPLES = 60 * SR
    N_WINDOWS = 12
    
    DEVICE = torch.device("cpu")  # Competition constraint
    
    LOGS = {}  # Comprehensive logging dict
    
    CFG = {
        "mode": MODE,
        "verbose": MODE == "train",
    
        # expensive research blocks
        "run_oof_baseline": MODE == "train",
        "run_probe_check": False,
        "run_probe_grid": False,
    
        # inference
        "batch_files": 16,
        "proxy_reduce_grid": ["max", "mean"],
        "proxy_reduce": "max",
        "run_proxy_reduce_grid": False,
        "dryrun_n_files": 50 if MODE == "train" else 20,
    
        # cache behavior
        "require_full_cache_in_submit": False,
        "full_cache_input_dir": Path("/kaggle/input/perch-meta"),
        "full_cache_work_dir": Path("/kaggle/working/perch_cache"),
    
        # frozen baseline fusion params
        "best_fusion": {
            "lambda_event": 0.4,
            "lambda_texture": 1.0,
            "lambda_proxy_texture": 0.8,
            "smooth_texture": 0.35,
            "smooth_event": 0.15,
        },
    
        # V17: ProtoSSM v5 — LARGER model
        "proto_ssm": {
            "d_model": 256,               # V17: increased from 128→256
            "d_state": 16,
            "n_ssm_layers": 3,            # V17: increased from 2→3
            "dropout": 0.15,
            "n_prototypes": 1,
            "n_sites": 20,
            "meta_dim": 16,
            "use_cross_attn": True,
            "cross_attn_heads": 4,
        },
    
        # ProtoSSM v5 training
        "proto_ssm_train": {
            "n_epochs": 60 if MODE == "train" else 40,   # ← was always 60,
            "lr": 1e-3,
            "weight_decay": 2e-3,
            "val_ratio": 0.15,
            "patience": 15  if MODE == "train" else 8,    # ← was always 15
            "pos_weight_cap": 30.0,
            "distill_weight": 0.1,
            "proto_margin": 0.1,
            "label_smoothing": 0.02,
            "oof_n_splits": 3,
            "mixup_alpha": 0.3,
            "focal_gamma": 2.0,
            "swa_start_frac": 0.7,
            "swa_lr": 5e-4,
        },
    
        # frozen probe params
        "frozen_best_probe": {
            "pca_dim": 64,
            "min_pos": 8,
            "C": 0.50,
            "alpha": 0.40,
        },
    
        # Residual SSM
        "residual_ssm": {
            "d_model": 64,
            "d_state": 8,
            "n_ssm_layers": 1,
            "dropout": 0.1,
            "correction_weight": 0.3,
            "n_epochs": 30,
            "lr": 1e-3,
            "patience": 8,
        },
    
        # Per-taxon temperature
        "temperature": {
            "aves": 1.10,
            "texture": 0.95,
        },
    
        # V17: Post-processing parameters
        "file_level_top_k": 2,
        "tta_shifts": [0, 1, -1],
        
        # V17 NEW: Rank-aware post-processing
        "rank_aware_scale": True,
        "rank_aware_power": 0.5,  # Power transform on file max
        
        # V17 NEW: Delta shift smoothing
        "delta_shift_alpha": 0.15,
        
        # V17 NEW: Per-class thresholds (grid search range)
        "threshold_grid": [0.3, 0.4, 0.5, 0.6, 0.7],
    
        "probe_backend": "mlp",
        "mlp_params": {
            "hidden_layer_sizes": (128,),
            "activation": "relu",
            "max_iter": 300,
            "early_stopping": True,
            "validation_fraction": 0.15,
            "n_iter_no_change": 15,
            "random_state": 42,
            "learning_rate_init": 0.001,
            "alpha": 0.01,
        },
    }
    
    CFG["full_cache_work_dir"].mkdir(parents=True, exist_ok=True)
    
    print("TensorFlow:", tf.__version__)
    print("PyTorch:", torch.__version__)
    print("Competition dir exists:", BASE.exists())
    print("Model dir exists:", MODEL_DIR.exists())
    print("V17 CFG: d_model=256, n_ssm_layers=3")
    print(json.dumps(
        {k: (str(v) if isinstance(v, Path) else v) for k, v in CFG.items()},
        indent=2
    ))
    
    # ── V18 CFG UPGRADES ──────────────────────
    CFG["proto_ssm"] = {
        "d_model": 320, "d_state": 32, "n_ssm_layers": 4,
        "dropout": 0.12, "n_prototypes": 2, "n_sites": 20,
        "meta_dim": 24, "use_cross_attn": True, "cross_attn_heads": 8,
    }
    CFG["proto_ssm_train"] = {
        "n_epochs": 80, "lr": 8e-4, "weight_decay": 1e-3,
        "val_ratio": 0.15, "patience": 20, "pos_weight_cap": 25.0,
        "distill_weight": 0.15, "proto_margin": 0.15,
        "label_smoothing": 0.03, "oof_n_splits": 5,
        "mixup_alpha": 0.4, "focal_gamma": 2.5,
        "swa_start_frac": 0.65, "swa_lr": 4e-4,
        "use_cosine_restart": True, "restart_period": 20,
    }
    CFG["residual_ssm"] = {
        "d_model": 128, "d_state": 16, "n_ssm_layers": 2,
        "dropout": 0.1, "correction_weight": 0.35,
        "n_epochs": 40, "lr": 8e-4, "patience": 12,
    }
    CFG["best_fusion"]["lambda_event"]         = 0.45
    CFG["best_fusion"]["lambda_texture"]       = 1.1
    CFG["best_fusion"]["lambda_proxy_texture"] = 0.9
    CFG["threshold_grid"] = [0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]
    CFG["tta_shifts"]        = [0, 1, -1, 2, -2]
    CFG["rank_aware_power"]  = 0.4
    CFG["delta_shift_alpha"] = 0.20
    CFG["mlp_params"] = {
        "hidden_layer_sizes": (256, 128), "activation": "relu",
        "max_iter": 500, "early_stopping": True,
        "validation_fraction": 0.15, "n_iter_no_change": 20,
        "random_state": 42, "learning_rate_init": 5e-4, "alpha": 0.005,
    }
    CFG["frozen_best_probe"] = {
        "pca_dim": 128, "min_pos": 5, "C": 0.75, "alpha": 0.45
    }
    print("✅ V18 CFG loaded")
    
    
    from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
    
    def get_cosine_restart_scheduler(optimizer, restart_period=20):
        return CosineAnnealingWarmRestarts(
            optimizer, T_0=restart_period, T_mult=1, eta_min=1e-5
        )
    
    print("✅ Cosine Restart Scheduler defined")
    
    # ── STEP 3: Mixup + CutMix Hybrid ─
    def mixup_cutmix(emb, logits, labels, alpha=0.4, cutmix_prob=0.3):
        B, T, D = emb.shape
        lam = np.random.beta(alpha, alpha)
        idx = torch.randperm(B)
    
        if np.random.rand() < cutmix_prob:
            # CutMix on time dimension
            cut_len = max(1, int(T * (1 - lam)))
            cut_start = np.random.randint(0, T - cut_len + 1)
            new_emb = emb.clone()
            new_emb[:, cut_start:cut_start+cut_len, :] = emb[idx, cut_start:cut_start+cut_len, :]
            new_logits = logits.clone()
            new_logits[:, cut_start:cut_start+cut_len, :] = logits[idx, cut_start:cut_start+cut_len, :]
            lam_actual = 1.0 - cut_len / T
            new_labels = lam_actual * labels + (1-lam_actual) * labels[idx]
        else:
            # Standard Mixup
            new_emb    = lam * emb    + (1-lam) * emb[idx]
            new_logits = lam * logits + (1-lam) * logits[idx]
            new_labels = lam * labels + (1-lam) * labels[idx]
    
        return new_emb, new_logits, new_labels
    
    print("✅ Mixup+CutMix defined")
    
    # ── STEP 4: Species-Frequency Aware Focal Loss ──
    def build_class_freq_weights(Y_FULL, cap=10.0):
        pos_count = Y_FULL.sum(axis=0).astype(np.float32) + 1.0
        total     = Y_FULL.shape[0]
        freq      = pos_count / total
        weights   = 1.0 / (freq ** 0.5)
        weights   = np.clip(weights, 1.0, cap)
        weights   = weights / weights.mean()
        return torch.tensor(weights, dtype=torch.float32)
    
    def species_focal_loss(logits, targets, class_weights, 
                           gamma=2.5, label_smoothing=0.03):
        targets_smooth = targets * (1 - label_smoothing) + label_smoothing / 2.0
        bce    = F.binary_cross_entropy_with_logits(
                     logits, targets_smooth, reduction="none")
        pt     = torch.exp(-bce)
        focal  = ((1 - pt) ** gamma) * bce
        w      = class_weights.to(logits.device).unsqueeze(0)
        return (focal * w).mean()
    
    print("✅ Species Focal Loss defined")
    
    taxonomy = pd.read_csv(BASE / "taxonomy.csv")
    sample_sub = pd.read_csv(BASE / "sample_submission.csv")
    soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
    
    PRIMARY_LABELS = sample_sub.columns[1:].tolist()
    N_CLASSES = len(PRIMARY_LABELS)
    
    taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
    soundscape_labels["primary_label"] = soundscape_labels["primary_label"].astype(str)
    
    def parse_soundscape_labels(x):
        if pd.isna(x):
            return []
        return [t.strip() for t in str(x).split(";") if t.strip()]
    
    FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")
    
    def parse_soundscape_filename(name):
        m = FNAME_RE.match(name)
        if not m:
            return {
                "file_id": None,
                "site": None,
                "date": pd.NaT,
                "time_utc": None,
                "hour_utc": -1,
                "month": -1,
            }
        file_id, site, ymd, hms = m.groups()
        dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
        return {
            "file_id": file_id,
            "site": site,
            "date": dt,
            "time_utc": hms,
            "hour_utc": int(hms[:2]),
            "month": int(dt.month) if pd.notna(dt) else -1,
        }
    
    def union_labels(series):
        return sorted(set(lbl for x in series for lbl in parse_soundscape_labels(x)))
    
    # Deduplicate duplicated rows and aggregate labels per 5s window
    sc_clean = (
        soundscape_labels
        .groupby(["filename", "start", "end"])["primary_label"]
        .apply(union_labels)
        .reset_index(name="label_list")
    )
    
    sc_clean["start_sec"] = pd.to_timedelta(sc_clean["start"]).dt.total_seconds().astype(int)
    sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
    sc_clean["row_id"] = sc_clean["filename"].str.replace(".ogg", "", regex=False) + "_" + sc_clean["end_sec"].astype(str)
    
    meta = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
    sc_clean = pd.concat([sc_clean, meta], axis=1)
    
    # Fully-labeled files
    windows_per_file = sc_clean.groupby("filename").size()
    full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
    sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)
    
    # Multi-hot label matrix aligned with sc_clean row order
    label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}
    Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
    
    for i, labels in enumerate(sc_clean["label_list"]):
        idxs = [label_to_idx[lbl] for lbl in labels if lbl in label_to_idx]
        if idxs:
            Y_SC[i, idxs] = 1
    
    full_truth = (
        sc_clean[sc_clean["file_fully_labeled"]]
        .sort_values(["filename", "end_sec"])
        .reset_index(drop=False)
    )
    
    Y_FULL_TRUTH = Y_SC[full_truth["index"].to_numpy()]
    
    print("sc_clean:", sc_clean.shape)
    print("Y_SC:", Y_SC.shape, Y_SC.dtype)
    print("Full files:", len(full_files))
    print("Trusted full windows:", len(full_truth))
    print("Active classes in full windows:", int((Y_FULL_TRUTH.sum(axis=0) > 0).sum()))
    
    CLASS_WEIGHTS = build_class_freq_weights(Y_FULL_TRUTH)
    print("✅ Class weights built")
    
    # ── STEP 5: Isotonic Calibration + Threshold Optimization ──
    from sklearn.isotonic import IsotonicRegression
    
    def calibrate_and_optimize_thresholds(oof_probs, Y_FULL, 
                                           threshold_grid, n_windows=12):
        n_samples, n_cls = oof_probs.shape
        thresholds = np.full(n_cls, 0.5, dtype=np.float32)
        n_files  = n_samples // n_windows
        file_oof = oof_probs.reshape(n_files, n_windows, n_cls).max(axis=1)
        file_y   = Y_FULL.reshape(n_files, n_windows, n_cls).max(axis=1)
    
        for c in range(n_cls):
            y_true, y_prob = file_y[:, c], file_oof[:, c]
            if y_true.sum() < 3:
                continue
            try:
                ir = IsotonicRegression(out_of_bounds="clip")
                ir.fit(y_prob, y_true)
                y_cal = ir.transform(y_prob)
            except:
                y_cal = y_prob
    
            best_f1, best_t = 0.0, 0.5
            for t in threshold_grid:
                pred = (y_cal >= t).astype(int)
                tp = ((pred==1)&(y_true==1)).sum()
                fp = ((pred==1)&(y_true==0)).sum()
                fn = ((pred==0)&(y_true==1)).sum()
                prec = tp/(tp+fp+1e-8)
                rec  = tp/(tp+fn+1e-8)
                f1   = 2*prec*rec/(prec+rec+1e-8)
                if f1 > best_f1:
                    best_f1, best_t = f1, t
            thresholds[c] = best_t
    
        print(f"Mean threshold: {thresholds.mean():.3f}")
        print(f"Range: [{thresholds.min():.2f}, {thresholds.max():.2f}]")
        return thresholds
    
    print("✅ Calibration + Threshold function defined")
    
    # ── STEP 6: Ensemble Weight Sweep ──
    def sweep_ensemble_weight(oof_proto, oof_mlp, Y_FULL, 
                              n_windows=12,
                              candidates=np.arange(0.3, 0.8, 0.05)):
        n_files = oof_proto.shape[0] // n_windows
        file_y  = Y_FULL.reshape(n_files, n_windows, -1).max(axis=1)
        best_auc, best_w = 0.0, 0.6
    
        for w in candidates:
            blended   = w * oof_proto + (1-w) * oof_mlp
            file_pred = blended.reshape(n_files, n_windows, -1).max(axis=1)
            try:
                auc = macro_auc_skip_empty(file_y, file_pred)
            except:
                continue
            if auc > best_auc:
                best_auc, best_w = auc, w
    
        print(f"Best ensemble weight (proto): {best_w:.2f}")
        print(f"Best AUC: {best_auc:.5f}")
        return best_w
    
    print("✅ Ensemble Weight Sweep defined")
    
    # Cell 3 — Load Perch, mapping, and selective frog proxies
    BEST = CFG["best_fusion"]
    
    # ONNX Perch load. TensorFlow SavedModel fallback is intentionally disabled:
    # current Kaggle TF/XLA can fail on Perch StableHLO artifacts (`vhlo.cosine_v2`).
    def _find_perch_onnx(root=Path("/kaggle/input")):
        root = Path(root)
        patterns = [
            "**/perch_v2_no_dft*.onnx",
            "**/*perch*no*dft*.onnx",
            "**/perch_v2.onnx",
            "**/*perch*.onnx",
        ]
        for pattern in patterns:
            hits = sorted(root.glob(pattern))
            if hits:
                print("Perch ONNX candidates for", pattern, ":", [str(p) for p in hits[:5]])
                return hits[0]
        all_onnx = sorted(root.glob("**/*.onnx"))
        for p in all_onnx:
            s = str(p).lower().replace("-", "_")
            if "perch" in s and "no_dft" in s:
                print("Perch ONNX selected by path tokens:", p)
                return p
        for p in all_onnx:
            s = str(p).lower()
            if "perch" in s:
                print("Perch ONNX selected by fallback path token:", p)
                return p
        print("No ONNX files found under /kaggle/input" if not all_onnx else "ONNX files found but none look like Perch: " + str([str(p) for p in all_onnx[:10]]))
        return Path("")

    ONNX_PERCH_PATH = _find_perch_onnx(Path("/kaggle/input"))
    USE_ONNX_PERCH = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()
    birdclassifier = None
    infer_fn = None

    if not _ONNX_AVAILABLE:
        raise ImportError("onnxruntime is required for Perch inference; install it before notebook execution.")
    if not ONNX_PERCH_PATH.exists():
        raise FileNotFoundError("Perch ONNX model not found. Attach the perch-v2-no-dft-onnx dataset or another Perch .onnx asset.")

    print(f"Using ONNX Perch: {ONNX_PERCH_PATH}")
    _so = ort.SessionOptions()
    _so.intra_op_num_threads = 4
    ONNX_SESSION = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so, providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUTPUT_MAP = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    
    def _find_perch_labels_path():
        preferred = MODEL_DIR / "assets" / "labels.csv"
        if preferred.exists():
            return preferred
        for p in sorted(Path("/kaggle/input").rglob("labels.csv")):
            try:
                cols = set(pd.read_csv(p, nrows=0).columns)
            except Exception:
                continue
            if {"inat2024_fsd50k", "scientific_name"} & cols:
                print("Using Perch labels:", p)
                return p
        raise FileNotFoundError("Perch labels.csv not found. Attach Perch ONNX labels or google/bird-vocalization-classifier.")
    
    def _load_perch_labels(path):
        df = pd.read_csv(path).reset_index().rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
        if "scientific_name" not in df.columns:
            for c in ["label", "labels", "name"]:
                if c in df.columns:
                    df = df.rename(columns={c: "scientific_name"})
                    break
        assert "scientific_name" in df.columns, f"No scientific_name column in {path}"
        return df
    
    bc_labels = _load_perch_labels(_find_perch_labels_path())
    
    NO_LABEL_INDEX = len(bc_labels)
    
    MANUAL_SCIENTIFIC_NAME_MAP = {
        # Optional future synonym fixes (add manual name corrections here)
    }
    
    taxonomy = taxonomy.copy()
    taxonomy["scientific_name_lookup"] = taxonomy["scientific_name"].replace(MANUAL_SCIENTIFIC_NAME_MAP)
    
    bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})
    
    mapping = taxonomy.merge(
        bc_lookup[["scientific_name_lookup", "bc_index"]],
        on="scientific_name_lookup",
        how="left"
    )
    
    mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)
    
    label_to_bc_index = mapping.set_index("primary_label")["bc_index"]
    BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
    
    MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
    MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
    UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
    MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)
    
    CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
    TEXTURE_TAXA = {"Amphibia", "Insecta"}
    
    ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]
    
    idx_active_texture = np.array(
        [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA],
        dtype=np.int32
    )
    idx_active_event = np.array(
        [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA],
        dtype=np.int32
    )
    
    idx_mapped_active_texture = idx_active_texture[MAPPED_MASK[idx_active_texture]]
    idx_mapped_active_event = idx_active_event[MAPPED_MASK[idx_active_event]]
    
    idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
    idx_unmapped_active_event = idx_active_event[~MAPPED_MASK[idx_active_event]]
    
    idx_unmapped_inactive = np.array(
        [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES],
        dtype=np.int32
    )
    
    # Build automatic genus proxies for unmapped non-sonotypes
    unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
    unmapped_non_sonotype = unmapped_df[
        ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
    ].copy()
    
    def get_genus_hits(scientific_name):
        genus = str(scientific_name).split()[0]
        hits = bc_labels[
            bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
        ].copy()
        return genus, hits
    
    proxy_map = {}
    for _, row in unmapped_non_sonotype.iterrows():
        target = row["primary_label"]
        sci = row["scientific_name"]
        genus, hits = get_genus_hits(sci)
        if len(hits) > 0:
            proxy_map[target] = {
                "target_scientific_name": sci,
                "genus": genus,
                "bc_indices": hits["bc_index"].astype(int).tolist(),
                "proxy_scientific_names": hits["scientific_name"].tolist(),
            }
    
    # Enable genus proxies for Amphibia, Insecta, and Aves (unmapped species)
    PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
    SELECTED_PROXY_TARGETS = sorted([
        t for t in proxy_map.keys()
        if CLASS_NAME_MAP.get(t) in PROXY_TAXA
    ])
    print(f"Proxy targets by class: { {cls: sum(1 for t in SELECTED_PROXY_TARGETS if CLASS_NAME_MAP.get(t)==cls) for cls in PROXY_TAXA} }")
    
    selected_proxy_pos = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)
    
    selected_proxy_pos_to_bc = {
        label_to_idx[target]: np.array(proxy_map[target]["bc_indices"], dtype=np.int32)
        for target in SELECTED_PROXY_TARGETS
    }
    
    idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
    idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
    idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)
    
    print(f"Mapped classes: {MAPPED_MASK.sum()} / {N_CLASSES}")
    print(f"Unmapped classes: {(~MAPPED_MASK).sum()}")
    print("Selected frog proxy targets:", SELECTED_PROXY_TARGETS)
    print("Active texture classes:", len(idx_active_texture))
    print("Selected proxy active texture:", len(idx_selected_proxy_active_texture))
    print("Prior-only active texture:", len(idx_selected_prioronly_active_texture))
    print("Prior-only active event:", len(idx_selected_prioronly_active_event))
    
    # Cell 4 — Metrics and helper utilities
    def macro_auc_skip_empty(y_true, y_score):
        keep = y_true.sum(axis=0) > 0
        return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")
    
    def smooth_cols_fixed12(scores, cols, alpha=0.35):
        if alpha <= 0 or len(cols) == 0:
            return scores.copy()
    
        s = scores.copy()
        assert len(s) % N_WINDOWS == 0, "Expected full-file blocks of 12 windows"
        view = s.reshape(-1, N_WINDOWS, s.shape[1])
    
        x = view[:, :, cols]
        prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
        next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    
        view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
        return s
    
    def smooth_events_fixed12(scores, cols, alpha=0.15):
        """Soft max-pool context for event birds (Aves).
        Uses local_max instead of average neighbor, preserving transient call detection."""
        if alpha <= 0 or len(cols) == 0:
            return scores.copy()
        s = scores.copy()
        assert len(s) % N_WINDOWS == 0
        view = s.reshape(-1, N_WINDOWS, s.shape[1])
        x = view[:, :, cols]
        prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
        next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
        local_max = np.maximum(x, np.maximum(prev_x, next_x))
        view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
        return s
    
    def seq_features_1d(v):
        """
        v: shape (n_rows,), ordered as full-file blocks of 12 windows
        Extended: tambah std_v untuk capture variance temporal dalam file
        """
        assert len(v) % N_WINDOWS == 0, "Expected full-file blocks of 12 windows"
        x = v.reshape(-1, N_WINDOWS)
    
        prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
        next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
        mean_v = np.repeat(x.mean(axis=1), N_WINDOWS)
        max_v  = np.repeat(x.max(axis=1),  N_WINDOWS)
        std_v  = np.repeat(x.std(axis=1),  N_WINDOWS)
    
        return prev_v, next_v, mean_v, max_v, std_v
    
    # V16/V17 NEW: Focal loss, file-level scaling, TTA, rank-aware, delta shift, per-class thresholds
    
    def focal_bce_with_logits(logits, targets, gamma=2.0, pos_weight=None, reduction="mean"):
        """Focal loss for multi-label classification.
        Reduces contribution of easy examples, focuses on hard ones."""
        if pos_weight is not None:
            bce = F.binary_cross_entropy_with_logits(
                logits, targets, pos_weight=pos_weight, reduction="none"
            )
        else:
            bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        
        p = torch.sigmoid(logits)
        pt = targets * p + (1 - targets) * (1 - p)
        focal_weight = (1 - pt) ** gamma
        loss = focal_weight * bce
        
        if reduction == "mean":
            return loss.mean()
        return loss
    
    
    def file_level_confidence_scale(preds, n_windows=12, top_k=2):
        """Rank 1/2 technique: scale each window's predictions by the file's top-K mean confidence."""
        N, C = preds.shape
        assert N % n_windows == 0
        view = preds.reshape(-1, n_windows, C)
        sorted_view = np.sort(view, axis=1)
        top_k_mean = sorted_view[:, -top_k:, :].mean(axis=1, keepdims=True)
        scaled = view * top_k_mean
        return scaled.reshape(N, C)
    
    
    # def temporal_shift_tta(emb_files, logits_files, model, site_ids, hours, shifts=[0, 1, -1]):
    #     """TTA by circular-shifting the 12-window embedding sequence."""
    #     all_preds = []
    #     model.eval()
        
    #     for shift in shifts:
    #         if shift == 0:
    #             e = emb_files
    #             l = logits_files
    #         else:
    #             e = np.roll(emb_files, shift, axis=1)
    #             l = np.roll(logits_files, shift, axis=1)
            
    #         with torch.no_grad():
    #             out, _, _ = model(
    #                 torch.tensor(e, dtype=torch.float32),
    #                 torch.tensor(l, dtype=torch.float32),
    #                 site_ids=torch.tensor(site_ids, dtype=torch.long),
    #                 hours=torch.tensor(hours, dtype=torch.long),
    #             )
    #             pred = out.numpy()
            
    #         if shift != 0:
    #             pred = np.roll(pred, -shift, axis=1)
            
    #         all_preds.append(pred)
        
    #     return np.mean(all_preds, axis=0)
    def temporal_shift_tta(emb_files, logits_files, model, site_ids, hours, shifts=[0, 1, -1], max_batch_size=512):
        """
        TTA by circular-shifting the 12-window embedding sequence.
        Batched and optimized for faster single-pass inference.
        """
        n_files = emb_files.shape[0]
        n_shifts = len(shifts)
        
        if n_shifts == 0:
            return np.zeros((n_files, emb_files.shape[1], logits_files.shape[2]), dtype=np.float32)
    
        e_list, l_list = [], []
        for shift in shifts:
            if shift == 0:
                e_list.append(emb_files)
                l_list.append(logits_files)
            else:
                e_list.append(np.roll(emb_files, shift, axis=1))
                l_list.append(np.roll(logits_files, shift, axis=1))
                
        e_batch = np.concatenate(e_list, axis=0)
        l_batch = np.concatenate(l_list, axis=0)
        
        site_batch = np.tile(site_ids, n_shifts)
        hour_batch = np.tile(hours, n_shifts)
        
        model.eval()
        pred_batch_list = []
        
        with torch.no_grad():
            total_samples = e_batch.shape[0]
            for start_idx in range(0, total_samples, max_batch_size):
                end_idx = min(start_idx + max_batch_size, total_samples)
                
                out, _, _ = model(
                    torch.tensor(e_batch[start_idx:end_idx], dtype=torch.float32),
                    torch.tensor(l_batch[start_idx:end_idx], dtype=torch.float32),
                    site_ids=torch.tensor(site_batch[start_idx:end_idx], dtype=torch.long),
                    hours=torch.tensor(hour_batch[start_idx:end_idx], dtype=torch.long),
                )
                pred_batch_list.append(out.numpy())
                
        pred_batch = np.concatenate(pred_batch_list, axis=0)
        pred_batch = pred_batch.reshape(n_shifts, n_files, pred_batch.shape[1], pred_batch.shape[2])
        
        all_preds = []
        for i, shift in enumerate(shifts):
            pred_i = pred_batch[i]
            if shift != 0:
                pred_i = np.roll(pred_i, -shift, axis=1)
            all_preds.append(pred_i)
        return np.mean(all_preds, axis=0)
    
    
    
    # V17: Post-processing utilities
    
    def rank_aware_scaling(scores, n_windows=12, power=0.5):
        """V17: 2025 Rank 3 technique. Scale each window by (file_max)^power.
        Suppresses predictions in uncertain files, boosts confident files."""
        N, C = scores.shape
        assert N % n_windows == 0
        n_files = N // n_windows
        
        view = scores.reshape(n_files, n_windows, C)
        file_max = view.max(axis=1, keepdims=True)  # (F, 1, C)
        
        # Apply power transform to file max
        scale = np.power(file_max, power)
        
        # Scale each window
        scaled = view * scale
        return scaled.reshape(N, C)
    
    
    def delta_shift_smooth(scores, n_windows=12, alpha=0.15):
        """V17: 2025 Rank 1 technique. Temporal smoothing across windows.
        new[t] = (1-alpha)*old[t] + 0.5*alpha*(old[t-1] + old[t+1])"""
        N, C = scores.shape
        assert N % n_windows == 0
        n_files = N // n_windows
        
        view = scores.reshape(n_files, n_windows, C)
        
        # Create shifted versions
        prev_view = np.concatenate([view[:, :1, :], view[:, :-1, :]], axis=1)
        next_view = np.concatenate([view[:, 1:, :], view[:, -1:, :]], axis=1)
        
        # Delta shift smoothing
        smoothed = (1 - alpha) * view + 0.5 * alpha * (prev_view + next_view)
        
        return smoothed.reshape(N, C)
    
    
    def optimize_per_class_thresholds(oof_scores, y_true, n_windows=12, thresholds=[0.3, 0.4, 0.5, 0.6, 0.7]):
        """V17: Find optimal decision threshold per class from OOF predictions.
        Optimizes F1-like metric (precision-recall balance) for each species."""
        n_classes = oof_scores.shape[1]
        best_thresholds = np.zeros(n_classes)
        best_scores = np.zeros(n_classes)
        
        for c in range(n_classes):
            y_c = y_true[:, c]
            scores_c = oof_scores[:, c]
            
            # Skip classes with no positive samples
            if y_c.sum() == 0:
                best_thresholds[c] = 0.5
                continue
                
            # Find best threshold
            best_f1 = 0
            best_t = 0.5
            
            for t in thresholds:
                pred_c = (scores_c > t).astype(int)
                tp = ((pred_c == 1) & (y_c == 1)).sum()
                fp = ((pred_c == 1) & (y_c == 0)).sum()
                fn = ((pred_c == 0) & (y_c == 1)).sum()
                
                if tp + fp == 0 or tp + fn == 0:
                    continue
                    
                precision = tp / (tp + fp)
                recall = tp / (tp + fn)
                f1 = 2 * precision * recall / (precision + recall + 1e-8)
                
                if f1 > best_f1:
                    best_f1 = f1
                    best_t = t
            
            best_thresholds[c] = best_t
            best_scores[c] = best_f1
        
        return best_thresholds, best_scores
    
    
    def apply_per_class_thresholds(scores, thresholds, n_windows=12):
        """V17: Apply per-class thresholds to convert scores to binary predictions."""
        N, C = scores.shape
        assert C == len(thresholds)
        
        # For competition, we submit probabilities but threshold for metrics
        # Apply threshold as a scaling factor that sharpens confident predictions
        scaled = np.copy(scores)
        
        for c in range(C):
            t = thresholds[c]
            # Sharpen: push above-threshold scores higher, below-threshold lower
            mask_above = scores[:, c] > t
            scaled[mask_above, c] = 0.5 + 0.5 * (scores[mask_above, c] - t) / (1 - t + 1e-8)
            scaled[~mask_above, c] = 0.5 * scores[~mask_above, c] / (t + 1e-8)
        
        return np.clip(scaled, 0, 1)
    
    
    print("V17 utilities defined: focal_bce_with_logits, file_level_confidence_scale, temporal_shift_tta,")
    print("  rank_aware_scaling, delta_shift_smooth, optimize_per_class_thresholds, apply_per_class_thresholds")
    
    # Cell 5 — Perch inference with embeddings + selective proxies
    def read_soundscape_60s(path):
        y, sr = sf.read(path, dtype="float32", always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr != SR:
            raise ValueError(f"Unexpected sample rate {sr} in {path}; expected {SR}")
        if len(y) < FILE_SAMPLES:
            y = np.pad(y, (0, FILE_SAMPLES - len(y)))
        elif len(y) > FILE_SAMPLES:
            y = y[:FILE_SAMPLES]
        return y
    
    # def infer_perch_with_embeddings(paths, batch_files=16, verbose=True, proxy_reduce="max"):
    #     paths = [Path(p) for p in paths]
    #     n_files = len(paths)
    #     n_rows = n_files * N_WINDOWS
    
    #     row_ids = np.empty(n_rows, dtype=object)
    #     filenames = np.empty(n_rows, dtype=object)
    #     sites = np.empty(n_rows, dtype=object)
    #     hours = np.empty(n_rows, dtype=np.int16)
    
    #     scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    #     embeddings = np.zeros((n_rows, 1536), dtype=np.float32)
    
    #     write_row = 0
    #     iterator = range(0, n_files, batch_files)
    #     if verbose:
    #         iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc="Perch batches")
    
    #     for start in iterator:
    #         batch_paths = paths[start:start + batch_files]
    #         batch_n = len(batch_paths)
    
    #         x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
    #         batch_row_start = write_row
    #         x_pos = 0
    
    #         for path in batch_paths:
    #             y = read_soundscape_60s(path)
    #             x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    
    #             meta = parse_soundscape_filename(path.name)
    #             stem = path.stem
    
    #             row_ids[write_row:write_row + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
    #             filenames[write_row:write_row + N_WINDOWS] = path.name
    #             sites[write_row:write_row + N_WINDOWS] = meta["site"]
    #             hours[write_row:write_row + N_WINDOWS] = int(meta["hour_utc"])
    
    #             x_pos += N_WINDOWS
    #             write_row += N_WINDOWS
    
    #         outputs = infer_fn(inputs=tf.convert_to_tensor(x))
    #         logits = outputs["label"].numpy().astype(np.float32, copy=False)
    #         emb = outputs["embedding"].numpy().astype(np.float32, copy=False)
    
    #         scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
    #         embeddings[batch_row_start:write_row] = emb
    
    #         # Selected frog proxies
    #         for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
    #             sub = logits[:, bc_idx_arr]
    #             if proxy_reduce == "max":
    #                 proxy_score = sub.max(axis=1)
    #             elif proxy_reduce == "mean":
    #                 proxy_score = sub.mean(axis=1)
    #             else:
    #                 raise ValueError("proxy_reduce must be 'max' or 'mean'")
    #             scores[batch_row_start:write_row, pos] = proxy_score.astype(np.float32)
    
    #         del x, outputs, logits, emb
    #         gc.collect()
    
    #     meta_df = pd.DataFrame({
    #         "row_id": row_ids,
    #         "filename": filenames,
    #         "site": sites,
    #         "hour_utc": hours,
    #     })
    
    #     return meta_df, scores, embeddings
    
    
    # ---------------------------------------- #
    # 2026/04/02 Update Process 
    # ---------------------------------------- #
    import concurrent.futures
    def infer_perch_with_embeddings(paths, batch_files=16, verbose=True, proxy_reduce="max"):
        paths = [Path(p) for p in paths]
        n_files = len(paths)
        n_rows = n_files * N_WINDOWS
    
        row_ids = np.empty(n_rows, dtype=object)
        filenames = np.empty(n_rows, dtype=object)
        sites = np.empty(n_rows, dtype=object)
        hours = np.empty(n_rows, dtype=np.int16)
    
        scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
        embeddings = np.zeros((n_rows, 1536), dtype=np.float32)
    
        write_row = 0
        iterator = range(0, n_files, batch_files)
        if verbose:
            iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc="Perch batches")
    
        # ─────ThreadPoolExecutor──
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
            
            # 1. Reserve the loading of the first batch in the background (start prefetching).
            next_paths = paths[0:batch_files]
            future_audio = [io_executor.submit(read_soundscape_60s, p) for p in next_paths]
    
            for start in iterator:
                batch_paths = next_paths
                batch_n = len(batch_paths)
    
                # --- Phase A: Receiving audio data ---
                batch_audio = [f.result() for f in future_audio]
    
                # --- Phase B: Immediately begin loading the "next batch". ---
                next_start = start + batch_files
                if next_start < n_files:
                    next_paths = paths[next_start:next_start + batch_files]
                    future_audio = [io_executor.submit(read_soundscape_60s, p) for p in next_paths]
    
                # --- Phase C: Data Formatting ---
                x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
                batch_row_start = write_row
                x_pos = 0
    
                for i, path in enumerate(batch_paths):
                    y = batch_audio[i]
                    x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    
                    meta = parse_soundscape_filename(path.name)
                    stem = path.stem
    
                    row_ids[write_row:write_row + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                    filenames[write_row:write_row + N_WINDOWS] = path.name
                    sites[write_row:write_row + N_WINDOWS] = meta["site"]
                    hours[write_row:write_row + N_WINDOWS] = int(meta["hour_utc"])
    
                    x_pos += N_WINDOWS
                    write_row += N_WINDOWS
    
                # --- Phase D: Heavy Inference (CPU Compute Bound) ---
                if USE_ONNX_PERCH:
                    onnx_outs = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                    logits = onnx_outs[ONNX_OUTPUT_MAP["label"]].astype(np.float32, copy=False)
                    emb = onnx_outs[ONNX_OUTPUT_MAP["embedding"]].astype(np.float32, copy=False)
                else:
                    outputs = infer_fn(inputs=tf.convert_to_tensor(x))
                    logits = outputs["label"].numpy().astype(np.float32, copy=False)
                    emb = outputs["embedding"].numpy().astype(np.float32, copy=False)
    
                scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
                embeddings[batch_row_start:write_row] = emb
    
                # Selected frog proxies
                for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
                    sub = logits[:, bc_idx_arr]
                    if proxy_reduce == "max":
                        proxy_score = sub.max(axis=1)
                    elif proxy_reduce == "mean":
                        proxy_score = sub.mean(axis=1)
                    else:
                        raise ValueError("proxy_reduce must be 'max' or 'mean'")
                    scores[batch_row_start:write_row, pos] = proxy_score.astype(np.float32)
    
                # Memory leak (OOM) prevention
                if USE_ONNX_PERCH:
                    del x, onnx_outs, logits, emb, batch_audio
                else:
                    del x, outputs, logits, emb, batch_audio
                gc.collect()
    
        meta_df = pd.DataFrame({
            "row_id": row_ids,
            "filename": filenames,
            "site": sites,
            "hour_utc": hours,
        })
    
        return meta_df, scores, embeddings
    
    # Cell 6 — Load or compute full-file Perch cache
    def resolve_full_cache_paths():
        pairs = [("full_perch_meta.parquet", "full_perch_arrays.npz")]
        roots = [
            CFG["full_cache_work_dir"],
            Path("/kaggle/working"),
            CFG["full_cache_input_dir"],
            Path("/kaggle/input"),
        ]
        seen = set()
        for root in roots:
            if not root.exists():
                continue
            key = str(root)
            if key in seen:
                continue
            seen.add(key)
            for meta_name, npz_name in pairs:
                meta = root / meta_name
                npz = root / npz_name
                if meta.exists() and npz.exists():
                    return meta, npz
            for meta_name, npz_name in pairs:
                for meta in sorted(root.rglob(meta_name)):
                    npz = meta.parent / npz_name
                    if npz.exists():
                        return meta, npz
        return None, None

    def pick_cache_array(arr, candidates, shape_hint_cols):
        for key in candidates:
            if key in arr.files:
                value = arr[key]
                if getattr(value, "ndim", 0) == 2 and value.shape[1] == shape_hint_cols:
                    return value, key
                print(f"Skipping cache key {key!r}: shape={getattr(value, 'shape', None)}, expected second dim={shape_hint_cols}")
        for key in arr.files:
            value = arr[key]
            if getattr(value, "ndim", 0) == 2 and value.shape[1] == shape_hint_cols:
                return value, key
        raise KeyError(f"Could not find cache array. candidates={candidates}, available={arr.files}")

    def normalize_perch_cache_rows(meta_df, scores, emb, n_windows=N_WINDOWS):
        assert len(meta_df) == scores.shape[0] == emb.shape[0], (
            f"cache row count mismatch: meta={len(meta_df)}, scores={scores.shape}, emb={emb.shape}"
        )
        meta_df = meta_df.copy()
        if "filename" not in meta_df.columns:
            if "row_id" in meta_df.columns:
                meta_df["filename"] = meta_df["row_id"].str.rsplit("_", n=1).str[0] + ".ogg"
            else:
                raise KeyError("cache metadata must contain filename or row_id")
        if "end_sec" not in meta_df.columns:
            if "window_idx" in meta_df.columns:
                meta_df["end_sec"] = (meta_df["window_idx"].astype(int) + 1) * WINDOW_SEC
            elif "row_id" in meta_df.columns:
                meta_df["end_sec"] = meta_df["row_id"].str.rsplit("_", n=1).str[-1].astype(int)
            else:
                assert len(meta_df) % n_windows == 0, "cannot infer end_sec from cache row count"
                meta_df["end_sec"] = np.tile(np.arange(WINDOW_SEC, WINDOW_SEC * n_windows + 1, WINDOW_SEC), len(meta_df) // n_windows)
        if "row_id" not in meta_df.columns:
            stem = meta_df["filename"].astype(str).str.replace(".ogg", "", regex=False)
            meta_df["row_id"] = stem + "_" + meta_df["end_sec"].astype(int).astype(str)
        assert meta_df["row_id"].is_unique, "duplicate row_id in Perch cache metadata"
        meta_df["_cache_pos"] = np.arange(len(meta_df))
        order = meta_df.sort_values(["filename", "end_sec"])["_cache_pos"].to_numpy()
        meta_df = meta_df.iloc[order].drop(columns=["_cache_pos"]).reset_index(drop=True)
        return meta_df, scores[order], emb[order]

    cache_meta, cache_npz = resolve_full_cache_paths()
    
    if cache_meta is not None and cache_npz is not None:
        print("Loading cached full-file Perch outputs from:")
        print("  ", cache_meta)
        print("  ", cache_npz)
    
        meta_full = pd.read_parquet(cache_meta)
        arr = np.load(cache_npz)
        scores_full_raw, score_key = pick_cache_array(
            arr,
            ["scores_full_raw", "scores", "sc", "logits", "perch_scores", "preds", "arr_0"],
            N_CLASSES,
        )
        emb_full, emb_key = pick_cache_array(
            arr,
            ["emb_full", "embs", "emb", "embeddings", "features", "perch_embs", "arr_1"],
            1536,
        )
        scores_full_raw = scores_full_raw.astype(np.float32)
        emb_full = emb_full.astype(np.float32)
        print("Loaded cache arrays:", score_key, scores_full_raw.shape, emb_key, emb_full.shape)
    
    else:
        if CFG["mode"] == "submit" and CFG["require_full_cache_in_submit"]:
            raise FileNotFoundError(
                "Submit mode requires cached full-file Perch outputs. "
                "Attach the cache dataset or place full_perch_meta.parquet/full_perch_arrays.npz in working dir."
            )
    
        print("No cache found. Running Perch on trusted full files...")
        full_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    
        # Use CFG["proxy_reduce"] for consistency with grid search
        meta_full, scores_full_raw, emb_full = infer_perch_with_embeddings(
            full_paths,
            batch_files=CFG["batch_files"],
            verbose=CFG["verbose"],
            proxy_reduce=CFG["proxy_reduce"],
        )
    
        out_meta = CFG["full_cache_work_dir"] / "full_perch_meta.parquet"
        out_npz = CFG["full_cache_work_dir"] / "full_perch_arrays.npz"
    
        meta_full.to_parquet(out_meta, index=False)
        np.savez_compressed(
            out_npz,
            scores_full_raw=scores_full_raw,
            emb_full=emb_full,
        )
    
        print("Saved cache to:")
        print("  ", out_meta)
        print("  ", out_npz)
    
    meta_full, scores_full_raw, emb_full = normalize_perch_cache_rows(meta_full, scores_full_raw, emb_full)

    # Align truth to cached order
    full_truth_aligned = full_truth.set_index("row_id").loc[meta_full["row_id"]].reset_index()
    Y_FULL = Y_SC[full_truth_aligned["index"].to_numpy()]
    
    assert np.all(full_truth_aligned["filename"].values == meta_full["filename"].values)
    assert np.all(full_truth_aligned["row_id"].values == meta_full["row_id"].values)
    
    print("meta_full:", meta_full.shape)
    print("scores_full_raw:", scores_full_raw.shape, scores_full_raw.dtype)
    print("emb_full:", emb_full.shape, emb_full.dtype)
    print("Y_FULL:", Y_FULL.shape, Y_FULL.dtype)
    
    # [MODIFIED - Opsi 3] Grid search proxy_reduce: evaluasi "max" vs "mean" via OOF AUC
    # Dilakukan hanya saat train mode; hasilnya di-freeze ke CFG["proxy_reduce"] untuk submit
    PROXY_REDUCE_CACHE = CFG["full_cache_work_dir"] / "proxy_reduce_grid.json"
    
    if CFG.get("run_proxy_reduce_grid", False):
        print("\n[Opsi 3] Running proxy_reduce grid search: max vs mean...")
        proxy_reduce_results = {}
    
        for pr in CFG["proxy_reduce_grid"]:
            full_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
            _meta, _scores, _emb = infer_perch_with_embeddings(
                full_paths,
                batch_files=CFG["batch_files"],
                verbose=False,
                proxy_reduce=pr,
            )
    
            # OOF baseline AUC untuk proxy_reduce ini (tanpa probe)
            _oof_b, _oof_p, _ = build_oof_base_prior(
                scores_full_raw=_scores,
                meta_full=_meta,
                sc_clean=sc_clean,
                Y_SC=Y_SC,
                n_splits=5,
                verbose=False,
            )
            auc = macro_auc_skip_empty(Y_FULL, _oof_b)
            proxy_reduce_results[pr] = float(auc)
            print(f"  proxy_reduce={pr!r:6s} → OOF baseline AUC = {auc:.6f}")
    
        best_pr = max(proxy_reduce_results, key=proxy_reduce_results.get)
        CFG["proxy_reduce"] = best_pr
        print(f"\n  Best proxy_reduce = {best_pr!r} (AUC={proxy_reduce_results[best_pr]:.6f})")
    
        PROXY_REDUCE_CACHE.write_text(json.dumps({
            "results": proxy_reduce_results,
            "best_proxy_reduce": best_pr,
        }, indent=2))
        print("  Saved to:", PROXY_REDUCE_CACHE)
    
    elif PROXY_REDUCE_CACHE.exists():
        _pr_data = json.loads(PROXY_REDUCE_CACHE.read_text())
        CFG["proxy_reduce"] = _pr_data["best_proxy_reduce"]
        print(f"[Opsi 3] Loaded proxy_reduce from cache: {CFG['proxy_reduce']!r}")
        print("  Grid results:", _pr_data["results"])
    
    else:
        print(f"[Opsi 3] Using default proxy_reduce={CFG['proxy_reduce']!r} (submit mode or no cache)")
    
    # Cell 7 — Fold-safe metadata prior tables
    def fit_prior_tables(prior_df, Y_prior):
        prior_df = prior_df.reset_index(drop=True)
    
        global_p = Y_prior.mean(axis=0).astype(np.float32)
    
        # Site
        site_keys = sorted(prior_df["site"].dropna().astype(str).unique().tolist())
        site_to_i = {k: i for i, k in enumerate(site_keys)}
        site_n = np.zeros(len(site_keys), dtype=np.float32)
        site_p = np.zeros((len(site_keys), Y_prior.shape[1]), dtype=np.float32)
    
        for s in site_keys:
            i = site_to_i[s]
            mask = prior_df["site"].astype(str).values == s
            site_n[i] = mask.sum()
            site_p[i] = Y_prior[mask].mean(axis=0)
    
        # Hour
        hour_keys = sorted(prior_df["hour_utc"].dropna().astype(int).unique().tolist())
        hour_to_i = {h: i for i, h in enumerate(hour_keys)}
        hour_n = np.zeros(len(hour_keys), dtype=np.float32)
        hour_p = np.zeros((len(hour_keys), Y_prior.shape[1]), dtype=np.float32)
    
        for h in hour_keys:
            i = hour_to_i[h]
            mask = prior_df["hour_utc"].astype(int).values == h
            hour_n[i] = mask.sum()
            hour_p[i] = Y_prior[mask].mean(axis=0)
    
        # Site-hour
        sh_to_i = {}
        sh_n_list = []
        sh_p_list = []
    
        for (s, h), idx in prior_df.groupby(["site", "hour_utc"]).groups.items():
            sh_to_i[(str(s), int(h))] = len(sh_n_list)
            idx = np.array(list(idx))
            sh_n_list.append(len(idx))
            sh_p_list.append(Y_prior[idx].mean(axis=0))
    
        sh_n = np.array(sh_n_list, dtype=np.float32)
        sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, Y_prior.shape[1]), dtype=np.float32)
    
        return {
            "global_p": global_p,
            "site_to_i": site_to_i,
            "site_n": site_n,
            "site_p": site_p,
            "hour_to_i": hour_to_i,
            "hour_n": hour_n,
            "hour_p": hour_p,
            "sh_to_i": sh_to_i,
            "sh_n": sh_n,
            "sh_p": sh_p,
        }
    
    def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
        n = len(sites)
        p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32, copy=True)
    
        site_idx = np.fromiter(
            (tables["site_to_i"].get(str(s), -1) for s in sites),
            dtype=np.int32,
            count=n
        )
        hour_idx = np.fromiter(
            (tables["hour_to_i"].get(int(h), -1) if int(h) >= 0 else -1 for h in hours),
            dtype=np.int32,
            count=n
        )
        sh_idx = np.fromiter(
            (tables["sh_to_i"].get((str(s), int(h)), -1) if int(h) >= 0 else -1 for s, h in zip(sites, hours)),
            dtype=np.int32,
            count=n
        )
    
        valid = hour_idx >= 0
        if valid.any():
            nh = tables["hour_n"][hour_idx[valid]][:, None]
            wh = nh / (nh + 8.0)
            p[valid] = wh * tables["hour_p"][hour_idx[valid]] + (1.0 - wh) * p[valid]
    
        valid = site_idx >= 0
        if valid.any():
            ns = tables["site_n"][site_idx[valid]][:, None]
            ws = ns / (ns + 8.0)
            p[valid] = ws * tables["site_p"][site_idx[valid]] + (1.0 - ws) * p[valid]
    
        valid = sh_idx >= 0
        if valid.any():
            nsh = tables["sh_n"][sh_idx[valid]][:, None]
            wsh = nsh / (nsh + 4.0)
            p[valid] = wsh * tables["sh_p"][sh_idx[valid]] + (1.0 - wsh) * p[valid]
    
        np.clip(p, eps, 1.0 - eps, out=p)
        return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)
    
    def fuse_scores_with_tables(base_scores, sites, hours, tables,
                                lambda_event=BEST["lambda_event"],
                                lambda_texture=BEST["lambda_texture"],
                                lambda_proxy_texture=BEST["lambda_proxy_texture"],
                                smooth_texture=BEST["smooth_texture"],
                                smooth_event=BEST["smooth_event"]):
        scores = base_scores.copy()
        prior = prior_logits_from_tables(sites, hours, tables)
    
        # mapped active
        if len(idx_mapped_active_event):
            scores[:, idx_mapped_active_event] += lambda_event * prior[:, idx_mapped_active_event]
    
        if len(idx_mapped_active_texture):
            scores[:, idx_mapped_active_texture] += lambda_texture * prior[:, idx_mapped_active_texture]
    
        # selected frog proxies
        if len(idx_selected_proxy_active_texture):
            scores[:, idx_selected_proxy_active_texture] += lambda_proxy_texture * prior[:, idx_selected_proxy_active_texture]
    
        # prior-only active unmapped
        if len(idx_selected_prioronly_active_event):
            scores[:, idx_selected_prioronly_active_event] = lambda_event * prior[:, idx_selected_prioronly_active_event]
    
        if len(idx_selected_prioronly_active_texture):
            scores[:, idx_selected_prioronly_active_texture] = lambda_texture * prior[:, idx_selected_prioronly_active_texture]
    
        # inactive unmapped
        if len(idx_unmapped_inactive):
            scores[:, idx_unmapped_inactive] = -8.0
    
        scores = smooth_cols_fixed12(scores, idx_active_texture, alpha=smooth_texture)
        scores = smooth_events_fixed12(scores, idx_active_event, alpha=smooth_event)
        return scores.astype(np.float32, copy=False), prior
    
    # Cell 8 — Honest OOF base/prior meta-features (required for final stacker fit)
    # def build_oof_base_prior(scores_full_raw, meta_full, sc_clean, Y_SC, n_splits=5, verbose=True):
    #     groups_full = meta_full["filename"].to_numpy()
    #     gkf = GroupKFold(n_splits=n_splits)
    
    #     oof_base = np.zeros_like(scores_full_raw, dtype=np.float32)
    #     oof_prior = np.zeros_like(scores_full_raw, dtype=np.float32)
    #     fold_id = np.full(len(meta_full), -1, dtype=np.int16)
    
    #     splits = list(gkf.split(scores_full_raw, groups=groups_full))
    #     iterator = tqdm(splits, desc="OOF base/prior folds", disable=not verbose)
    
    #     for fold, (tr_idx, va_idx) in enumerate(iterator, 1):
    #         tr_idx = np.sort(tr_idx)
    #         va_idx = np.sort(va_idx)
    
    #         val_files = set(meta_full.iloc[va_idx]["filename"].tolist())
    
    #         # Fold-safe prior tables: exclude all validation files
    #         prior_mask = ~sc_clean["filename"].isin(val_files).values
    #         prior_df_fold = sc_clean.loc[prior_mask].reset_index(drop=True)
    #         Y_prior_fold = Y_SC[prior_mask]
    
    #         tables = fit_prior_tables(prior_df_fold, Y_prior_fold)
    
    #         va_base, va_prior = fuse_scores_with_tables(
    #             scores_full_raw[va_idx],
    #             sites=meta_full.iloc[va_idx]["site"].to_numpy(),
    #             hours=meta_full.iloc[va_idx]["hour_utc"].to_numpy(),
    #             tables=tables,
    #         )
    
    #         oof_base[va_idx] = va_base
    #         oof_prior[va_idx] = va_prior
    #         fold_id[va_idx] = fold
    
    #     assert (fold_id >= 0).all()
    #     return oof_base, oof_prior, fold_id
    from sklearn.model_selection import StratifiedGroupKFold
    def build_oof_base_prior(scores_full_raw, meta_full, sc_clean, Y_SC, n_splits=5, verbose=True):
        groups_full = meta_full["filename"].to_numpy()
        
        row_id_to_idx = {r: i for i, r in enumerate(sc_clean["row_id"])}
        aligned_indices = [row_id_to_idx[r] for r in meta_full["row_id"]]
        Y_ALIGNED = Y_SC[aligned_indices]  # これで長さが 708 になります
        
        y_strat = np.argmax(Y_ALIGNED, axis=1)
        unique_classes, counts = np.unique(y_strat, return_counts=True)
        rare_classes = unique_classes[counts < n_splits]
        y_strat[np.isin(y_strat, rare_classes)] = -1
        
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=91)
    
        oof_base = np.zeros_like(scores_full_raw, dtype=np.float32)
        oof_prior = np.zeros_like(scores_full_raw, dtype=np.float32)
        fold_id = np.full(len(meta_full), -1, dtype=np.int16)
    
        splits = list(sgkf.split(scores_full_raw, y_strat, groups=groups_full))
        iterator = tqdm(splits, desc="OOF base/prior folds", disable=not verbose)
    
        for fold, (tr_idx, va_idx) in enumerate(iterator, 1):
            tr_idx = np.sort(tr_idx)
            va_idx = np.sort(va_idx)
    
            val_files = set(meta_full.iloc[va_idx]["filename"].tolist())
    
            # Fold-safe prior tables: exclude all validation files
            prior_mask = ~sc_clean["filename"].isin(val_files).values
            prior_df_fold = sc_clean.loc[prior_mask].reset_index(drop=True)
            Y_prior_fold = Y_SC[prior_mask]
    
            tables = fit_prior_tables(prior_df_fold, Y_prior_fold)
    
            va_base, va_prior = fuse_scores_with_tables(
                scores_full_raw[va_idx],
                sites=meta_full.iloc[va_idx]["site"].to_numpy(),
                hours=meta_full.iloc[va_idx]["hour_utc"].to_numpy(),
                tables=tables,
            )
    
            oof_base[va_idx] = va_base
            oof_prior[va_idx] = va_prior
            fold_id[va_idx] = fold
    
        assert (fold_id >= 0).all()
        return oof_base, oof_prior, fold_id
    
    
    OOF_META_CACHE = CFG["full_cache_work_dir"] / "full_oof_meta_features.npz"

    def resolve_oof_meta_cache():
        if OOF_META_CACHE.exists():
            return OOF_META_CACHE
        hits = sorted(Path("/kaggle/input").rglob("full_oof_meta_features.npz"))
        if hits:
            sgkf_hits = [p for p in hits if "sgkfk" in str(p).lower()]
            chosen = sgkf_hits[0] if sgkf_hits else hits[0]
            print("Using external OOF meta cache:", chosen)
            return chosen
        return None

    OOF_META_READ = resolve_oof_meta_cache()

    if OOF_META_READ is not None:
        print("Loading cached OOF meta-features from:", OOF_META_READ)
        arr = np.load(OOF_META_READ)
        oof_base = arr["oof_base"].astype(np.float32)
        oof_prior = arr["oof_prior"].astype(np.float32)
        oof_fold_id = arr["fold_id"].astype(np.int16)
    else:
        print("Building OOF meta-features...")
        oof_base, oof_prior, oof_fold_id = build_oof_base_prior(
            scores_full_raw=scores_full_raw,
            meta_full=meta_full,
            sc_clean=sc_clean,
            Y_SC=Y_SC,
            n_splits=5,
            verbose=CFG["verbose"],
        )

        np.savez_compressed(
            OOF_META_CACHE,
            oof_base=oof_base,
            oof_prior=oof_prior,
            fold_id=oof_fold_id,
        )
        print("Saved OOF meta-features to:", OOF_META_CACHE)

    baseline_oof_auc = macro_auc_skip_empty(Y_FULL, oof_base)
    
    if MODE == "train":
        raw_local_auc = macro_auc_skip_empty(Y_FULL, scores_full_raw)
        print(f"Raw local AUC (not OOF-dependent): {raw_local_auc:.6f}")
        print(f"Honest OOF baseline AUC: {baseline_oof_auc:.6f}")
    
    import torch
    import torch.nn as nn
    import numpy as np
    
    def build_all_class_features_vectorized(Z, raw_scores, prior_scores, base_scores, valid_classes, n_windows=12):
        """
        A function that constructs all 14 types of features for all classes in one go, without using a for loop.
        Output tensor shape: (V: number of effective classes, N: number of samples, D+14)
        """
        N, D = Z.shape
        V = len(valid_classes)
        
        # (V, N)
        raw = raw_scores[:, valid_classes].T
        prior = prior_scores[:, valid_classes].T
        base = base_scores[:, valid_classes].T
        
        n_files = N // n_windows
        base_view = base.reshape(V, n_files, n_windows)
        
        # Batch calculation of time series features
        prev_base = np.concatenate([base_view[:, :, :1], base_view[:, :, :-1]], axis=2).reshape(V, N)
        next_base = np.concatenate([base_view[:, :, 1:], base_view[:, :, -1:]], axis=2).reshape(V, N)
        mean_base = np.repeat(base_view.mean(axis=2), n_windows, axis=1)
        max_base = np.repeat(base_view.max(axis=2), n_windows, axis=1)
        std_base = np.repeat(base_view.std(axis=2), n_windows, axis=1)
        
        diff_mean = base - mean_base
        diff_prev = base - prev_base
        diff_next = base - next_base
        
        interact_rp = raw * prior
        interact_rb = raw * base
        interact_pb = prior * base
        
        # Stack 14 scalar features in the last dimension -> (V, N, 14)
        scalar_feats = np.stack([
            raw, prior, base, prev_base, next_base, 
            mean_base, max_base, std_base, 
            diff_mean, diff_prev, diff_next, 
            interact_rp, interact_rb, interact_pb
        ], axis=-1)
        
        # Z (N, D) -> (V, N, D) 
        Z_expanded = np.broadcast_to(Z, (V, N, D))
        
        # features -> (V, N, D+14)
        X_all = np.concatenate([Z_expanded, scalar_feats], axis=-1)
        return X_all.astype(np.float32)
    
    class VectorizedMLPProbes(nn.Module):
        """
        A class that combines multiple scikit-learn MLPClassifier classes into a single PyTorch model.
        """
        def __init__(self, probe_models, device="cpu"):
            super().__init__()
            self.valid_classes = sorted(list(probe_models.keys()))
            self.V = len(self.valid_classes)
            
            if self.V == 0:
                return
                
            sample_clf = probe_models[self.valid_classes[0]]
            self.n_layers = len(sample_clf.coefs_)
            
            self.weights = nn.ParameterList()
            self.biases = nn.ParameterList()
            
            # (V, in_dim, out_dim)
            for layer_idx in range(self.n_layers):
                W = np.stack([probe_models[c].coefs_[layer_idx] for c in self.valid_classes], axis=0)
                b = np.stack([probe_models[c].intercepts_[layer_idx] for c in self.valid_classes], axis=0)
                
                self.weights.append(nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False))
                self.biases.append(nn.Parameter(torch.tensor(b, dtype=torch.float32), requires_grad=False))
                
            self.to(device)
    
        def forward(self, x):
            # x shape: (V, N, in_dim)
            h = x
            for i in range(self.n_layers):
                h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
                if i < self.n_layers - 1:
                    h = torch.relu(h)
            
            return h.squeeze(-1) # (V, N)
    
    def get_vectorized_mlp_scores(Z, raw, prior, base, probe_models, alpha_p, n_windows=12, device="cpu"):
        """
        A wrapper function that wraps all of the above vectorization processes
        """
        mlp_scores = base.copy()
        if len(probe_models) == 0:
            return mlp_scores
            
        valid_classes = sorted(list(probe_models.keys()))
        
        # 1. Building a tensor
        X_all = build_all_class_features_vectorized(Z, raw, prior, base, valid_classes, n_windows)
        
        # 2. Batch inference using PyTorch
        vec_probe = VectorizedMLPProbes(probe_models, device=device)
        vec_probe.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X_all, dtype=torch.float32, device=device)
            preds = vec_probe(X_tensor).cpu().numpy() # (V, N)
            
        # 3. Blending
        preds_t = preds.T # (N, V)
        base_valid = base[:, valid_classes]
        
        mlp_scores[:, valid_classes] = (1.0 - alpha_p) * base_valid + alpha_p * preds_t
        return mlp_scores
    
    # Cell 9 — Classwise embedding-probe helpers
    def build_class_features(emb_proj, raw_col, prior_col, base_col):
        """
        emb_proj: (n, d)
        raw_col, prior_col, base_col: (n,)
        returns: (n, d + 13)
    
        Fitur: embedding + 7 sequential + 3 interaction + std + 3 diff
        """
        prev_base, next_base, mean_base, max_base, std_base = seq_features_1d(base_col)
    
        # Diff features: posisi window relatif terhadap konteks file
        diff_mean = base_col - mean_base   # apakah window ini lebih tinggi dari rata2 file?
        diff_prev = base_col - prev_base   # onset: naik dari window sebelumnya?
        diff_next = base_col - next_base   # offset: turun ke window berikutnya?
    
        feats = np.concatenate([
            emb_proj,
            raw_col[:, None],
            prior_col[:, None],
            base_col[:, None],
            prev_base[:, None],
            next_base[:, None],
            mean_base[:, None],
            max_base[:, None],
            std_base[:, None],             # variance temporal dalam file
            diff_mean[:, None],            # deviasi dari mean file
            diff_prev[:, None],            # deteksi onset
            diff_next[:, None],            # deteksi offset
            # interaction terms
            (raw_col * prior_col)[:, None],
            (raw_col * base_col)[:, None],
            (prior_col * base_col)[:, None],
        ], axis=1)
    
        return feats.astype(np.float32, copy=False)
    
    from sklearn.model_selection import StratifiedGroupKFold
    
    def run_oof_embedding_probe(
        scores_raw,
        emb,
        meta_df,
        y_true,
        pca_dim=64,
        min_pos=8,
        C=0.25,
        alpha=0.5,
    ):
        groups = meta_df["filename"].to_numpy()
        
        y_strat = np.argmax(Y_SC, axis=1) 
        
        unique_classes, counts = np.unique(y_strat, return_counts=True)
        rare_classes = unique_classes[counts < n_splits]
        y_strat[np.isin(y_strat, rare_classes)] = -1
        
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=91)
    
        oof_base_local = np.zeros_like(scores_raw, dtype=np.float32)
        oof_final = np.zeros_like(scores_raw, dtype=np.float32)
        modeled_counts = np.zeros(scores_raw.shape[1], dtype=np.int32)
        oof_models = {}
    
        split_list = list(sgkf.split(scores_raw, y_strat, groups=groups))
    
        for fold, (tr_idx, va_idx) in enumerate(tqdm(split_list, desc="Embedding-probe folds", disable=not CFG.get("verbose", True)), 1):
            tr_idx = np.sort(tr_idx)
            va_idx = np.sort(va_idx)
    
            val_files = set(meta_df.iloc[va_idx]["filename"].tolist())
    
            # Fold-safe priors
            prior_mask = ~sc_clean["filename"].isin(val_files).values
            prior_df_fold = sc_clean.loc[prior_mask].reset_index(drop=True)
            Y_prior_fold = Y_SC[prior_mask]
            tables = fit_prior_tables(prior_df_fold, Y_prior_fold)
    
            base_tr, prior_tr = fuse_scores_with_tables(
                scores_raw[tr_idx],
                sites=meta_df.iloc[tr_idx]["site"].to_numpy(),
                hours=meta_df.iloc[tr_idx]["hour_utc"].to_numpy(),
                tables=tables,
            )
            base_va, prior_va = fuse_scores_with_tables(
                scores_raw[va_idx],
                sites=meta_df.iloc[va_idx]["site"].to_numpy(),
                hours=meta_df.iloc[va_idx]["hour_utc"].to_numpy(),
                tables=tables,
            )
    
            oof_base_local[va_idx] = base_va
            oof_final[va_idx] = base_va
    
            # Embedding preprocessing on train fold only
            scaler = StandardScaler()
            emb_tr_s = scaler.fit_transform(emb[tr_idx])
            emb_va_s = scaler.transform(emb[va_idx])
    
            n_comp = min(pca_dim, emb_tr_s.shape[0] - 1, emb_tr_s.shape[1])
            pca = PCA(n_components=n_comp)
            Z_tr = pca.fit_transform(emb_tr_s).astype(np.float32)
            Z_va = pca.transform(emb_va_s).astype(np.float32)
    
            class_iterator = np.where(y_true[tr_idx].sum(axis=0) >= min_pos)[0].tolist()
    
            for cls_idx in tqdm(class_iterator, desc=f"Fold {fold} classes", leave=False, disable=not CFG["verbose"]):
            # for cls_idx in tqdm(class_iterator, desc=f"Fold {fold} classes", leave=False):
                y_tr = y_true[tr_idx, cls_idx]
    
                if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                    continue
    
                X_tr_cls = build_class_features(
                    Z_tr,
                    raw_col=scores_raw[tr_idx, cls_idx],
                    prior_col=prior_tr[:, cls_idx],
                    base_col=base_tr[:, cls_idx],
                )
                X_va_cls = build_class_features(
                    Z_va,
                    raw_col=scores_raw[va_idx, cls_idx],
                    prior_col=prior_va[:, cls_idx],
                    base_col=base_va[:, cls_idx],
                )
    
                # Pilih backend probe: mlp | lgbm | logreg
                backend = CFG.get("probe_backend", "mlp")
                n_pos = int(y_tr.sum())
                n_neg = len(y_tr) - n_pos
    
                if backend == "mlp":
                    # MLPClassifier tidak support sample_weight
                    # Gunakan oversampling: duplikasi positif agar balance
                    if n_pos > 0 and n_neg > n_pos:
                        repeat = max(1, n_neg // n_pos)
                        pos_idx = np.where(y_tr == 1)[0]
                        X_bal = np.vstack([X_tr_cls, np.tile(X_tr_cls[pos_idx], (repeat, 1))])
                        y_bal = np.concatenate([y_tr, np.ones(len(pos_idx) * repeat, dtype=y_tr.dtype)])
                    else:
                        X_bal, y_bal = X_tr_cls, y_tr
                    clf = MLPClassifier(**CFG["mlp_params"])
                    clf.fit(X_bal, y_bal)
                    pred_va = clf.predict_proba(X_va_cls)[:, 1].astype(np.float32)
                    pred_va = np.log(pred_va + 1e-7) - np.log(1 - pred_va + 1e-7)
                elif backend == "lgbm" and _LGBM_AVAILABLE:
                    scale_pos = max(1.0, n_neg / max(n_pos, 1))
                    clf = LGBMClassifier(
                        **CFG["lgbm_params"],
                        scale_pos_weight=scale_pos,
                    )
                    clf.fit(X_tr_cls, y_tr)
                    pred_va = clf.predict_proba(X_va_cls)[:, 1].astype(np.float32)
                    pred_va = np.log(pred_va + 1e-7) - np.log(1 - pred_va + 1e-7)
                else:
                    clf = LogisticRegression(
                        C=C, max_iter=400, solver="liblinear",
                        class_weight="balanced",
                    )
                    clf.fit(X_tr_cls, y_tr)
                    pred_va = clf.decision_function(X_va_cls).astype(np.float32)
    
                oof_final[va_idx, cls_idx] = (
                    (1.0 - alpha) * base_va[:, cls_idx] +
                    alpha * pred_va
                )
    
                modeled_counts[cls_idx] += 1
    
        score_base = macro_auc_skip_empty(y_true, oof_base_local)
        score_final = macro_auc_skip_empty(y_true, oof_final)
    
        return {
            "oof_base": oof_base_local,
            "oof_final": oof_final,
            "modeled_counts": modeled_counts,
            "score_base": score_base,
            "score_final": score_final,
        }
    
    # ProtoSSM v4 — Enhanced with Cross-Attention Layer
    
    class SelectiveSSM(nn.Module):
        # Simplified Mamba-style selective state space model.
        # Input-dependent (selective) discretization of continuous-time SSM.
        # For T=12 bioacoustic windows, the sequential scan is efficient on CPU.
    
        def __init__(self, d_model, d_state=16, d_conv=4):
            super().__init__()
            self.d_model = d_model
            self.d_state = d_state
    
            self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
            self.conv1d = nn.Conv1d(
                d_model, d_model, d_conv,
                padding=d_conv - 1, groups=d_model
            )
            self.dt_proj = nn.Linear(d_model, d_model, bias=True)
    
            A = torch.arange(1, d_state + 1, dtype=torch.float32)
            A = A.unsqueeze(0).expand(d_model, -1)
            self.A_log = nn.Parameter(torch.log(A))
            self.D = nn.Parameter(torch.ones(d_model))
            self.B_proj = nn.Linear(d_model, d_state, bias=False)
            self.C_proj = nn.Linear(d_model, d_state, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
    
        def forward(self, x):
            B_size, T, D = x.shape
            xz = self.in_proj(x)
            x_ssm, z = xz.chunk(2, dim=-1)
    
            x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
            x_conv = F.silu(x_conv)
    
            dt = F.softplus(self.dt_proj(x_conv))
            A = -torch.exp(self.A_log)
            B = self.B_proj(x_conv)
            C = self.C_proj(x_conv)
    
            h = torch.zeros(B_size, D, self.d_state, device=x.device)
            ys = []
            for t in range(T):
                dt_t = dt[:, t, :]
                dA = torch.exp(A[None, :, :] * dt_t[:, :, None])
                dB = dt_t[:, :, None] * B[:, t, None, :]
                h = h * dA + x[:, t, :, None] * dB
                y_t = (h * C[:, t, None, :]).sum(-1)
                ys.append(y_t)
    
            y = torch.stack(ys, dim=1)
            return y + x * self.D[None, None, :]
    
    
    class TemporalCrossAttention(nn.Module):
        """Multi-head cross-attention between temporal windows.
        Captures non-local patterns (e.g., dawn chorus onset, counter-singing)
        that sequential SSM may miss."""
        
        def __init__(self, d_model, n_heads=4, dropout=0.1):
            super().__init__()
            self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(d_model)
            self.ffn = nn.Sequential(
                nn.Linear(d_model, d_model * 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 2, d_model),
                nn.Dropout(dropout),
            )
            self.norm2 = nn.LayerNorm(d_model)
        
        def forward(self, x):
            # x: (B, T, D)
            residual = x
            x = self.norm(x)
            attn_out, _ = self.attn(x, x, x)
            x = residual + attn_out
            
            residual = x
            x = self.norm2(x)
            x = residual + self.ffn(x)
            return x
    
    
    class ProtoSSMv2(nn.Module):
        # Prototypical State Space Model v4 with cross-attention and metadata awareness.
        #
        # V16 additions:
        # - Cross-attention layer after SSM for non-local temporal patterns
        # - All other v2 features preserved (metadata, prototypes, gated fusion)
        
        def __init__(self, d_input=1536, d_model=192, d_state=16,
                     n_ssm_layers=2, n_classes=234, n_windows=12,
                     dropout=0.2, n_sites=20, meta_dim=16,
                     use_cross_attn=True, cross_attn_heads=4):
            super().__init__()
            self.d_model = d_model
            self.n_classes = n_classes
            self.n_windows = n_windows
    
            # 1. Feature projection
            self.input_proj = nn.Sequential(
                nn.Linear(d_input, d_model),
                nn.LayerNorm(d_model),
                nn.GELU(),
                nn.Dropout(dropout),
            )
    
            # 2. Learnable positional encoding
            self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
    
            # 3. Metadata embeddings
            self.site_emb = nn.Embedding(n_sites, meta_dim)
            self.hour_emb = nn.Embedding(24, meta_dim)
            self.meta_proj = nn.Linear(2 * meta_dim, d_model)
    
            # 4. Bidirectional SSM layers
            self.ssm_fwd = nn.ModuleList()
            self.ssm_bwd = nn.ModuleList()
            self.ssm_merge = nn.ModuleList()
            self.ssm_norm = nn.ModuleList()
            for _ in range(n_ssm_layers):
                self.ssm_fwd.append(SelectiveSSM(d_model, d_state))
                self.ssm_bwd.append(SelectiveSSM(d_model, d_state))
                self.ssm_merge.append(nn.Linear(2 * d_model, d_model))
                self.ssm_norm.append(nn.LayerNorm(d_model))
            self.ssm_drop = nn.Dropout(dropout)
    
            # 4b. NEW: Cross-attention after SSM
            self.use_cross_attn = use_cross_attn
            if use_cross_attn:
                self.cross_attn = TemporalCrossAttention(d_model, n_heads=cross_attn_heads, dropout=dropout)
    
            # 5. Learnable class prototypes
            self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
            self.proto_temp = nn.Parameter(torch.tensor(5.0))
    
            # 6. Per-class calibration bias
            self.class_bias = nn.Parameter(torch.zeros(n_classes))
    
            # 7. Per-class gated fusion with Perch logits
            self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))
    
            # 8. Taxonomic auxiliary head
            self.n_families = 0
            self.family_head = None
    
        def init_prototypes_from_data(self, embeddings, labels):
            with torch.no_grad():
                h = self.input_proj(embeddings)
                for c in range(self.n_classes):
                    mask = labels[:, c] > 0.5
                    if mask.sum() > 0:
                        self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)
    
        def init_family_head(self, n_families, class_to_family):
            self.n_families = n_families
            self.family_head = nn.Linear(self.d_model, n_families)
            self.register_buffer('class_to_family', torch.tensor(class_to_family, dtype=torch.long))
    
        def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
            B, T, _ = emb.shape
    
            # Project embeddings
            h = self.input_proj(emb)
            h = h + self.pos_enc[:, :T, :]
    
            # Add metadata embeddings
            if site_ids is not None and hours is not None:
                s_emb = self.site_emb(site_ids)
                h_emb = self.hour_emb(hours)
                meta = self.meta_proj(torch.cat([s_emb, h_emb], dim=-1))
                h = h + meta[:, None, :]
    
            # Bidirectional SSM
            for fwd, bwd, merge, norm in zip(
                self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm
            ):
                residual = h
                h_f = fwd(h)
                h_b = bwd(h.flip(1)).flip(1)
                h = merge(torch.cat([h_f, h_b], dim=-1))
                h = self.ssm_drop(h)
                h = norm(h + residual)
    
            # NEW: Cross-attention for non-local temporal patterns
            if self.use_cross_attn:
                h = self.cross_attn(h)
    
            h_temporal = h
    
            # Prototypical cosine similarity + class bias
            h_norm = F.normalize(h, dim=-1)
            p_norm = F.normalize(self.prototypes, dim=-1)
            temp = F.softplus(self.proto_temp)
            sim = torch.matmul(h_norm, p_norm.T) * temp + self.class_bias[None, None, :]
    
            # Gated fusion with Perch logits
            if perch_logits is not None:
                alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
                species_logits = alpha * sim + (1 - alpha) * perch_logits
            else:
                species_logits = sim
    
            # Taxonomic auxiliary prediction
            family_logits = None
            if self.family_head is not None:
                h_pool = h.mean(dim=1)
                family_logits = self.family_head(h_pool)
    
            return species_logits, family_logits, h_temporal
    
        def count_parameters(self):
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
    
    ssm_cfg = CFG["proto_ssm"]
    print("ProtoSSMv4 architecture defined (with cross-attention).")
    test_model = ProtoSSMv2(
        d_model=ssm_cfg["d_model"], n_ssm_layers=2,
        n_sites=ssm_cfg["n_sites"], meta_dim=ssm_cfg["meta_dim"],
        use_cross_attn=ssm_cfg.get("use_cross_attn", True),
        cross_attn_heads=ssm_cfg.get("cross_attn_heads", 4),
    )
    print(f"Parameter count: {test_model.count_parameters():,}")
    del test_model
    
    # ProtoSSM v4 Training Loop — with Mixup, Focal Loss, SWA
    
    def build_taxonomy_groups(taxonomy_df, primary_labels):
        for col in ["family", "order", "class_name"]:
            if col in taxonomy_df.columns:
                group_map = taxonomy_df.set_index("primary_label")[col].to_dict()
                break
        else:
            group_map = {label: "Unknown" for label in primary_labels}
    
        groups = sorted(set(group_map.values()))
        grp_to_idx = {g: i for i, g in enumerate(groups)}
        class_to_group = []
        for label in primary_labels:
            grp = group_map.get(label, "Unknown")
            class_to_group.append(grp_to_idx.get(grp, 0))
        return len(groups), class_to_group, grp_to_idx
    
    
    def build_site_mapping(meta_df):
        sites = meta_df["site"].unique().tolist()
        site_to_idx = {s: i + 1 for i, s in enumerate(sites)}
        n_sites = len(sites) + 1
        return site_to_idx, n_sites
    
    
    def reshape_to_files(flat_array, meta_df, n_windows=N_WINDOWS):
        filenames = meta_df["filename"].to_numpy()
        unique_files = []
        seen = set()
        for f in filenames:
            if f not in seen:
                unique_files.append(f)
                seen.add(f)
    
        n_files = len(unique_files)
        assert len(flat_array) == n_files * n_windows, \
            f"Expected {n_files * n_windows} rows, got {len(flat_array)}"
    
        new_shape = (n_files, n_windows) + flat_array.shape[1:]
        return flat_array.reshape(new_shape), unique_files
    
    
    def get_file_metadata(meta_df, file_list, site_to_idx, n_sites_max):
        file_to_row = {}
        filenames = meta_df["filename"].to_numpy()
        sites = meta_df["site"].to_numpy()
        hours = meta_df["hour_utc"].to_numpy()
        for i, f in enumerate(filenames):
            if f not in file_to_row:
                file_to_row[f] = i
    
        site_ids = np.zeros(len(file_list), dtype=np.int64)
        hour_ids = np.zeros(len(file_list), dtype=np.int64)
        for fi, fname in enumerate(file_list):
            row = file_to_row.get(fname)
            if row is not None:
                sid = site_to_idx.get(sites[row], 0)
                site_ids[fi] = min(sid, n_sites_max - 1)
                hour_ids[fi] = int(hours[row]) % 24
        return site_ids, hour_ids
    
    
    def mixup_files(emb, logits, labels, site_ids, hours, families, alpha=0.3):
        """File-level mixup augmentation for ProtoSSM training.
        Mixes pairs of files with random lambda from Beta(alpha, alpha).
        Returns augmented versions of all inputs."""
        n = len(emb)
        if alpha <= 0 or n < 2:
            return emb, logits, labels, site_ids, hours, families
        
        lam = np.random.beta(alpha, alpha)
        lam = max(lam, 1.0 - lam)  # Ensure lam >= 0.5 (dominant sample stays dominant)
        
        perm = np.random.permutation(n)
        
        emb_mix = lam * emb + (1 - lam) * emb[perm]
        logits_mix = lam * logits + (1 - lam) * logits[perm]
        labels_mix = lam * labels + (1 - lam) * labels[perm]
        
        # For discrete features (site, hour), keep the dominant sample's values
        families_mix = lam * families + (1 - lam) * families[perm] if families is not None else None
        
        return emb_mix, logits_mix, labels_mix, site_ids, hours, families_mix
    
    # ─────────────────────────────────────────────────────────────────────────────
    # Locate pretrained ProtoSSM weights in attached inputs. Prefer the SGKF package when present.
    def _prefer_sgkf(paths):
        paths = sorted(Path(p) for p in paths)
        sgkf = [p for p in paths if "sgkfk" in str(p).lower()]
        return sgkf[0] if sgkf else (paths[0] if paths else None)

    ProtoSSM_PATH_OBJ = _prefer_sgkf(Path("/kaggle/input").rglob("proto_ssm_best.pt"))
    ProtoSSM_JSON_OBJ = _prefer_sgkf(Path("/kaggle/input").rglob("proto_ssm_history.json"))
    ProtoSSM_PATH = str(ProtoSSM_PATH_OBJ) if ProtoSSM_PATH_OBJ is not None else None
    ProtoSSM_JSON = str(ProtoSSM_JSON_OBJ) if ProtoSSM_JSON_OBJ is not None else None
    print("ProtoSSM weights:", ProtoSSM_PATH)

    def train_proto_ssm_single(model, emb_train, logits_train, labels_train,
                               site_ids_train=None, hours_train=None,
                               emb_val=None, logits_val=None, labels_val=None,
                               site_ids_val=None, hours_val=None,
                               file_families_train=None, file_families_val=None,
                               cfg=None, verbose=True):
        """Train a single ProtoSSM v4 model with mixup, focal loss, and SWA."""
        print("────────────────────────────────────────────────────────")
        print("──▶▶▶ProtoSSM Train...:")
        print("────────────────────────────────────────────────────────")
        if ProtoSSM_PATH is not None and ProtoSSM_JSON is not None:
            print("────────────────────────────────────────────────────────")
            print("──▶▶▶ProtoSSM Load Model(TrainSkip)...:")
            print("────────────────────────────────────────────────────────")
            load_model_path = CFG.get("pretrained_proto_path", ProtoSSM_PATH)
            load_hist_path = CFG.get("pretrained_hist_path", ProtoSSM_JSON)
            
            # Model Load
            if os.path.exists(load_model_path):
                model.load_state_dict(torch.load(load_model_path, map_location=DEVICE))
                model.eval()
                if verbose:
                    print(f"▶ [Load] Loaded pre-trained ProtoSSM from {load_model_path}")
            else:
                print(f"⚠️ WARNING: Pre-trained model not found at {load_model_path}!")
                
            # History Load
            history = {"train_loss": [], "val_loss": [], "val_auc": []}
            if os.path.exists(load_hist_path):
                import json
                with open(load_hist_path, "r") as f:
                    history = json.load(f)
                    
            return model, history
        
    
        if cfg is None:
            cfg = CFG["proto_ssm_train"]
    
        label_smoothing = cfg.get("label_smoothing", 0.0)
        mixup_alpha = cfg.get("mixup_alpha", 0.0)
        focal_gamma = cfg.get("focal_gamma", 0.0)
        swa_start_frac = cfg.get("swa_start_frac", 1.0)  # 1.0 = disabled
        n_epochs = cfg["n_epochs"]
        swa_start_epoch = int(n_epochs * swa_start_frac)
    
        # Convert to tensors (base — unmixed)
        labels_np = labels_train.copy()
        
        # Apply label smoothing
        if label_smoothing > 0:
            labels_np = labels_np * (1.0 - label_smoothing) + label_smoothing / 2.0
    
        has_val = emb_val is not None
        if has_val:
            emb_v = torch.tensor(emb_val, dtype=torch.float32)
            logits_v = torch.tensor(logits_val, dtype=torch.float32)
            labels_v = torch.tensor(labels_val, dtype=torch.float32)
            site_v = torch.tensor(site_ids_val, dtype=torch.long) if site_ids_val is not None else None
            hour_v = torch.tensor(hours_val, dtype=torch.long) if hours_val is not None else None
    
        fam_v = torch.tensor(file_families_val, dtype=torch.float32) if (has_val and file_families_val is not None) else None
    
        # Class weights for imbalanced data
        labels_tr_t = torch.tensor(labels_np, dtype=torch.float32)
        pos_counts = labels_tr_t.sum(dim=(0, 1))
        total = labels_tr_t.shape[0] * labels_tr_t.shape[1]
        pos_weight = ((total - pos_counts) / (pos_counts + 1)).clamp(max=cfg["pos_weight_cap"])
    
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
        )
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=cfg["lr"],
            epochs=n_epochs, steps_per_epoch=1,
            pct_start=0.1, anneal_strategy='cos'
        )
    
        best_val_loss = float('inf')
        best_state = None
        wait = 0
        history = {"train_loss": [], "val_loss": [], "val_auc": []}
    
        # SWA state accumulator
        swa_state = None
        swa_count = 0
    
        for epoch in range(n_epochs):
            # === Mixup augmentation (per-epoch re-sampling) ===
            if mixup_alpha > 0 and epoch > 5:  # Skip mixup for first 5 epochs (warmup)
                emb_mix, logits_mix, labels_mix, _, _, fam_mix = mixup_files(
                    emb_train, logits_train, labels_np,
                    site_ids_train, hours_train, file_families_train,
                    alpha=mixup_alpha,
                )
            else:
                emb_mix, logits_mix, labels_mix = emb_train, logits_train, labels_np
                fam_mix = file_families_train
    
            emb_tr = torch.tensor(emb_mix, dtype=torch.float32)
            logits_tr = torch.tensor(logits_mix, dtype=torch.float32)
            labels_tr = torch.tensor(labels_mix, dtype=torch.float32)
            site_tr = torch.tensor(site_ids_train, dtype=torch.long) if site_ids_train is not None else None
            hour_tr = torch.tensor(hours_train, dtype=torch.long) if hours_train is not None else None
            fam_tr = torch.tensor(fam_mix, dtype=torch.float32) if fam_mix is not None else None
    
            # === Train ===
            model.train()
            species_out, family_out, _ = model(emb_tr, logits_tr, site_ids=site_tr, hours=hour_tr)
    
            # Primary loss: focal BCE or weighted BCE
            if focal_gamma > 0:
                loss_main = focal_bce_with_logits(
                    species_out, labels_tr,
                    gamma=focal_gamma,
                    pos_weight=pos_weight[None, None, :],
                )
            else:
                loss_main = F.binary_cross_entropy_with_logits(
                    species_out, labels_tr,
                    pos_weight=pos_weight[None, None, :]
                )
    
            # Knowledge distillation loss
            loss_distill = F.mse_loss(species_out, logits_tr)
    
            # Total loss
            loss = loss_main + cfg["distill_weight"] * loss_distill
    
            # Taxonomic auxiliary loss
            if family_out is not None and fam_tr is not None:
                loss_family = F.binary_cross_entropy_with_logits(family_out, fam_tr)
                loss = loss + 0.1 * loss_family
    
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
    
            # === SWA accumulation ===
            if epoch >= swa_start_epoch:
                if swa_state is None:
                    swa_state = {k: v.clone() for k, v in model.state_dict().items()}
                    swa_count = 1
                else:
                    for k in swa_state:
                        swa_state[k] += model.state_dict()[k]
                    swa_count += 1
    
            # === Validate ===
            model.eval()
            with torch.no_grad():
                if has_val:
                    val_out, val_fam, _ = model(emb_v, logits_v, site_ids=site_v, hours=hour_v)
                    val_loss = F.binary_cross_entropy_with_logits(
                        val_out, labels_v,
                        pos_weight=pos_weight[None, None, :]
                    )
    
                    val_pred = val_out.reshape(-1, val_out.shape[-1]).numpy()
                    val_true = labels_v.reshape(-1, labels_v.shape[-1]).numpy()
                    try:
                        val_auc = macro_auc_skip_empty(val_true, val_pred)
                    except Exception:
                        val_auc = 0.0
                else:
                    val_loss = loss
                    val_auc = 0.0
    
            history["train_loss"].append(loss.item())
            history["val_loss"].append(val_loss.item())
            history["val_auc"].append(val_auc)
    
            if val_loss.item() < best_val_loss:
                best_val_loss = val_loss.item()
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
    
            if verbose and (epoch + 1) % 20 == 0:
                lr_now = optimizer.param_groups[0]['lr']
                swa_info = f" swa={swa_count}" if swa_count > 0 else ""
                print(f"  Epoch {epoch+1:3d}: train={loss.item():.4f} val={val_loss.item():.4f} "
                      f"auc={val_auc:.4f} lr={lr_now:.6f} wait={wait}{swa_info}")
    
            if wait >= cfg["patience"]:
                if verbose:
                    print(f"  Early stopping at epoch {epoch+1} (best val_loss={best_val_loss:.4f})")
                break
    
        # Apply SWA if we accumulated enough checkpoints
        if swa_state is not None and swa_count >= 3:
            if verbose:
                print(f"  Applying SWA (averaged {swa_count} checkpoints)")
            avg_state = {k: v / swa_count for k, v in swa_state.items()}
            model.load_state_dict(avg_state)
        elif best_state is not None:
            model.load_state_dict(best_state)
    
        if verbose:
            print(f"  Training complete. Best val_loss={best_val_loss:.4f}")
            with torch.no_grad():
                alphas = torch.sigmoid(model.fusion_alpha).numpy()
                print(f"  Fusion alpha: mean={alphas.mean():.3f} min={alphas.min():.3f} max={alphas.max():.3f}")
                print(f"  Proto temperature: {F.softplus(model.proto_temp).item():.3f}")
        
        # ─────── Fix 2: Save Model & History───────
        PROC_MODE = "DoTrain"
        if PROC_MODE == "DoTrain":
            save_model_path = CFG.get("proto_model_path", "train_proto_ssm_single/models/proto_ssm_best.pt")
            save_hist_path = CFG.get("proto_hist_path", "train_proto_ssm_single/models/proto_ssm_history.json")
            
            os.makedirs(os.path.dirname(save_model_path) or ".", exist_ok=True)
            
            torch.save(model.state_dict(), save_model_path)
            
            import json
            with open(save_hist_path, "w") as f:
                json.dump(history, f, indent=4)
                
            if verbose:
                print(f"▶ [Save] Model successfully saved to {save_model_path}")
                print(f"▶ [Save] History successfully saved to {save_hist_path}")
        # ──────────────────────────────────────────────────────────────────────
        
        return model, history
    
    from sklearn.model_selection import StratifiedGroupKFold
    
    def run_proto_ssm_oof(emb_files, logits_files, labels_files,
                          site_ids_all, hours_all,
                          file_families, file_groups,
                          n_families, class_to_family,
                          cfg=None, verbose=True):
        """Run StratifiedGroupKFold OOF cross-validation for ProtoSSM v4."""
        if cfg is None:
            cfg = CFG["proto_ssm_train"]
    
        n_splits = cfg.get("oof_n_splits", 5)
        n_files = len(emb_files)
        ssm_cfg = CFG["proto_ssm"]
    
        oof_preds = np.zeros((n_files, N_WINDOWS, N_CLASSES), dtype=np.float32)
        fold_histories = []
        fold_alphas = []
    
        n_unique_groups = len(set(file_groups))
        if n_unique_groups < n_splits:
            print(f"  WARNING: Only {n_unique_groups} groups, reducing n_splits from {n_splits} to {n_unique_groups}")
            n_splits = n_unique_groups
    
        
        file_level_labels = labels_files.max(axis=1) # (n_files, N_CLASSES)
        
        y_strat = np.argmax(Y_SC, axis=1) 
        
        
        unique_classes, counts = np.unique(y_strat, return_counts=True)
        rare_classes = unique_classes[counts < n_splits]
        y_strat[np.isin(y_strat, rare_classes)] = -1
        
        
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=91)
        for fold_i, (train_idx, val_idx) in enumerate(sgkf.split(emb_files, y_strat, groups=file_groups)):
            if verbose:
                print(f"\n--- Fold {fold_i+1}/{n_splits} (train={len(train_idx)}, val={len(val_idx)}) ---")
    
            fold_model = ProtoSSMv2(
                d_input=emb_files.shape[2],
                d_model=ssm_cfg["d_model"],
                d_state=ssm_cfg["d_state"],
                n_ssm_layers=ssm_cfg["n_ssm_layers"],
                n_classes=N_CLASSES,
                n_windows=N_WINDOWS,
                dropout=ssm_cfg["dropout"],
                n_sites=ssm_cfg["n_sites"],
                meta_dim=ssm_cfg["meta_dim"],
                use_cross_attn=ssm_cfg.get("use_cross_attn", True),
                cross_attn_heads=ssm_cfg.get("cross_attn_heads", 4),
            ).to(DEVICE)
    
            # Initialize prototypes
            emb_flat_fold = emb_files[train_idx].reshape(-1, emb_files.shape[2])
            labels_flat_fold = labels_files[train_idx].reshape(-1, N_CLASSES)
            fold_model.init_prototypes_from_data(
                torch.tensor(emb_flat_fold, dtype=torch.float32),
                torch.tensor(labels_flat_fold, dtype=torch.float32)
            )
            fold_model.init_family_head(n_families, class_to_family)
    
            # Train on fold
            fold_model, fold_hist = train_proto_ssm_single(
                fold_model,
                emb_files[train_idx], logits_files[train_idx], labels_files[train_idx].astype(np.float32),
                site_ids_train=site_ids_all[train_idx], hours_train=hours_all[train_idx],
                emb_val=emb_files[val_idx], logits_val=logits_files[val_idx],
                labels_val=labels_files[val_idx].astype(np.float32),
                site_ids_val=site_ids_all[val_idx], hours_val=hours_all[val_idx],
                file_families_train=file_families[train_idx],
                file_families_val=file_families[val_idx],
                cfg=cfg, verbose=verbose,
            )
    
            # OOF predictions with TTA
            fold_model.eval()
            tta_shifts = CFG.get("tta_shifts", [0])
            if len(tta_shifts) > 1:
                oof_preds[val_idx] = temporal_shift_tta(
                    emb_files[val_idx], logits_files[val_idx], fold_model,
                    site_ids_all[val_idx], hours_all[val_idx], shifts=tta_shifts
                )
            else:
                with torch.no_grad():
                    val_emb = torch.tensor(emb_files[val_idx], dtype=torch.float32)
                    val_logits = torch.tensor(logits_files[val_idx], dtype=torch.float32)
                    val_sites = torch.tensor(site_ids_all[val_idx], dtype=torch.long)
                    val_hours = torch.tensor(hours_all[val_idx], dtype=torch.long)
                    val_out, _, _ = fold_model(val_emb, val_logits, site_ids=val_sites, hours=val_hours)
                    oof_preds[val_idx] = val_out.numpy()
    
            fold_alphas.append(torch.sigmoid(fold_model.fusion_alpha).detach().numpy().copy())
            fold_histories.append(fold_hist)
    
        return oof_preds, fold_histories, fold_alphas
    
    
    def optimize_ensemble_weight(oof_proto_flat, oof_mlp_flat, y_true_flat):
        """Grid search over blend weights to find optimal ProtoSSM ensemble weight."""
        weights = np.arange(0.0, 1.05, 0.05)
        results = []
    
        for w in weights:
            blended = w * oof_proto_flat + (1.0 - w) * oof_mlp_flat
            try:
                auc = macro_auc_skip_empty(y_true_flat, blended)
            except Exception:
                auc = 0.0
            results.append((w, auc))
    
        best_w, best_auc = max(results, key=lambda x: x[1])
        return best_w, best_auc, results
    
    
    print("ProtoSSM v4 training functions defined (with mixup, focal loss, SWA, TTA).")
    
    # Cell 10 — Probe tuning (train mode only)
    grid_results = None
    BEST_PROBE = None
    
    if CFG["run_probe_check"]:
        probe_result = run_oof_embedding_probe(
            scores_raw=scores_full_raw,
            emb=emb_full,
            meta_df=meta_full,
            y_true=Y_FULL,
            pca_dim=64,
            min_pos=8,
            C=0.25,
            alpha=0.5,
        )
    
        print(f"Honest OOF baseline AUC: {probe_result['score_base']:.6f}")
        print(f"Honest OOF embedding-probe AUC: {probe_result['score_final']:.6f}")
        print(f"Delta: {probe_result['score_final'] - probe_result['score_base']:.6f}")
    
        modeled_classes = np.where(probe_result["modeled_counts"] > 0)[0]
        print("Modeled classes:", len(modeled_classes))
        print([PRIMARY_LABELS[i] for i in modeled_classes[:20]])
    
    if CFG["run_probe_grid"]:
        param_grid = [
            {"pca_dim": 32, "min_pos": 8,  "C": 0.25, "alpha": 0.4},
            {"pca_dim": 64, "min_pos": 8,  "C": 0.25, "alpha": 0.4},
            {"pca_dim": 64, "min_pos": 8,  "C": 0.25, "alpha": 0.5},
            {"pca_dim": 64, "min_pos": 12, "C": 0.25, "alpha": 0.4},
            {"pca_dim": 96, "min_pos": 8,  "C": 0.25, "alpha": 0.4},
            {"pca_dim": 64, "min_pos": 8,  "C": 0.50, "alpha": 0.4},
        ]
    
        results = []
        for params in tqdm(param_grid, desc="Probe grid", disable=not CFG["verbose"]):
            out = run_oof_embedding_probe(
                scores_raw=scores_full_raw,
                emb=emb_full,
                meta_df=meta_full,
                y_true=Y_FULL,
                pca_dim=params["pca_dim"],
                min_pos=params["min_pos"],
                C=params["C"],
                alpha=params["alpha"],
            )
            results.append({
                **params,
                "baseline_oof_auc": out["score_base"],
                "probe_oof_auc": out["score_final"],
                "delta": out["score_final"] - out["score_base"],
                "n_modeled_classes": int((out["modeled_counts"] > 0).sum()),
            })
    
        grid_results = pd.DataFrame(results).sort_values("probe_oof_auc", ascending=False).reset_index(drop=True)
        display(grid_results)
    
        BEST_PROBE = {
            "pca_dim": int(grid_results.iloc[0]["pca_dim"]),
            "min_pos": int(grid_results.iloc[0]["min_pos"]),
            "C": float(grid_results.iloc[0]["C"]),
            "alpha": float(grid_results.iloc[0]["alpha"]),
        }
    
        # Save best params for future freezing
        best_probe_path = CFG["full_cache_work_dir"] / "best_probe_params.json"
        best_probe_path.write_text(json.dumps(BEST_PROBE, indent=2))
        print("Saved best probe params to:", best_probe_path)
    
    else:
        BEST_PROBE = CFG["frozen_best_probe"]
        print("Using frozen BEST_PROBE in submit mode:")
        print(BEST_PROBE)
    
    if grid_results is not None:
        grid_results.to_csv(CFG["full_cache_work_dir"] / "probe_grid_results.csv", index=False)
    
    # Cell 11 — Freeze final probe params
    if BEST_PROBE is None:
        BEST_PROBE = CFG["frozen_best_probe"]
    
    print("Final BEST_PROBE =", BEST_PROBE)
    
    # Optional — rerun best OOF probe once for diagnostics / caching
    BEST_OOF_RESULT = None
    
    if MODE == "train":
        BEST_OOF_RESULT = run_oof_embedding_probe(
            scores_raw=scores_full_raw,
            emb=emb_full,
            meta_df=meta_full,
            y_true=Y_FULL,
            pca_dim=int(BEST_PROBE["pca_dim"]),
            min_pos=int(BEST_PROBE["min_pos"]),
            C=float(BEST_PROBE["C"]),
            alpha=float(BEST_PROBE["alpha"]),
        )
    
        print(f"Honest OOF baseline AUC (BEST_PROBE rerun): {BEST_OOF_RESULT['score_base']:.6f}")
        print(f"Honest OOF probe AUC   (BEST_PROBE rerun): {BEST_OOF_RESULT['score_final']:.6f}")
    
    # Cell 12 — Fit final prior tables on all labeled soundscapes
    final_prior_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)
    
    print("Built final prior tables for inference.")
    print("OOF baseline AUC used for stacker training:", baseline_oof_auc)
    
    # Cell 13 — Fit embedding scaler + PCA on all trusted full windows
    emb_scaler = StandardScaler()
    emb_full_scaled = emb_scaler.fit_transform(emb_full)
    
    n_comp = min(
        int(BEST_PROBE["pca_dim"]),
        emb_full_scaled.shape[0] - 1,
        emb_full_scaled.shape[1]
    )
    
    emb_pca = PCA(n_components=n_comp)
    Z_FULL = emb_pca.fit_transform(emb_full_scaled).astype(np.float32)
    
    print("emb_full:", emb_full.shape)
    print("Z_FULL:", Z_FULL.shape)
    print("Explained variance ratio sum:", emb_pca.explained_variance_ratio_.sum())
    
    # Instantiate and train ProtoSSM v4
    
    # --- Step 1: Reshape to file-level ---
    emb_files, file_list = reshape_to_files(emb_full, meta_full)
    logits_files, _ = reshape_to_files(scores_full_raw, meta_full)
    labels_files, _ = reshape_to_files(Y_FULL, meta_full)
    
    print(f"Reshaped to file-level: emb={emb_files.shape}, logits={logits_files.shape}, labels={labels_files.shape}")
    print(f"Files: {len(file_list)}")
    
    # --- Step 2: Build taxonomy groups, site mapping, file metadata ---
    n_families, class_to_family, fam_to_idx = build_taxonomy_groups(taxonomy, PRIMARY_LABELS)
    print(f"Taxonomic groups: {n_families}")
    
    site_to_idx, n_sites_mapped = build_site_mapping(meta_full)
    n_sites_cfg = CFG["proto_ssm"]["n_sites"]
    print(f"Sites mapped: {n_sites_mapped} (capped to {n_sites_cfg})")
    
    site_ids_all, hours_all = get_file_metadata(meta_full, file_list, site_to_idx, n_sites_cfg)
    
    # Build per-file family labels (multi-hot)
    file_families = np.zeros((len(file_list), n_families), dtype=np.float32)
    for fi in range(len(file_list)):
        active_classes = np.where(labels_files[fi].sum(axis=0) > 0)[0]
        for ci in active_classes:
            file_families[fi, class_to_family[ci]] = 1.0
    
    # --- OOF Cross-Validation (TRAIN MODE ONLY) ---
    ENSEMBLE_WEIGHT_PROTO = 0.5  # default, overridden by OOF in train mode
    oof_proto_flat = None
    fold_alphas = []
    
    if MODE == "train":
        file_groups = np.array([f.split("_")[3] if len(f.split("_")) > 3 else f for f in file_list])
        print(f"File groups for OOF: {len(set(file_groups))} unique groups: {sorted(set(file_groups))}")
    
        t0_oof = time.time()
        oof_proto_preds, fold_histories, fold_alphas = run_proto_ssm_oof(
            emb_files, logits_files, labels_files,
            site_ids_all, hours_all,
            file_families, file_groups,
            n_families, class_to_family,
            cfg=CFG["proto_ssm_train"],
            verbose=CFG["verbose"],
        )
        oof_time = time.time() - t0_oof
        print(f"\nOOF cross-validation time: {oof_time:.1f}s")
    
        oof_proto_flat = oof_proto_preds.reshape(-1, N_CLASSES)
        y_flat = labels_files.reshape(-1, N_CLASSES).astype(np.float32)
    
        per_class_auc_proto = {}
        for ci in range(N_CLASSES):
            if y_flat[:, ci].sum() > 0 and y_flat[:, ci].sum() < len(y_flat):
                try:
                    per_class_auc_proto[ci] = roc_auc_score(y_flat[:, ci], oof_proto_flat[:, ci])
                except Exception:
                    pass
    
        overall_oof_auc_proto = macro_auc_skip_empty(y_flat, oof_proto_flat)
        print(f"ProtoSSM OOF macro AUC: {overall_oof_auc_proto:.4f}")
    
        LOGS["oof_auc_proto"] = overall_oof_auc_proto
        LOGS["per_class_auc_proto"] = {PRIMARY_LABELS[k]: v for k, v in per_class_auc_proto.items()}
        LOGS["oof_time"] = oof_time
    else:
        print("Submit mode: skipping OOF cross-validation")
    
    # --- Train final model on ALL data ---
    ssm_cfg = CFG["proto_ssm"]
    model = ProtoSSMv2(
        d_input=emb_full.shape[1],
        d_model=ssm_cfg["d_model"],
        d_state=ssm_cfg["d_state"],
        n_ssm_layers=ssm_cfg["n_ssm_layers"],
        n_classes=N_CLASSES,
        n_windows=N_WINDOWS,
        dropout=ssm_cfg["dropout"],
        n_sites=ssm_cfg["n_sites"],
        meta_dim=ssm_cfg["meta_dim"],
        use_cross_attn=ssm_cfg.get("use_cross_attn", True),
        cross_attn_heads=ssm_cfg.get("cross_attn_heads", 4),
    ).to(DEVICE)
    
    emb_flat_tensor = torch.tensor(emb_full, dtype=torch.float32)
    labels_flat_tensor = torch.tensor(Y_FULL, dtype=torch.float32)
    model.init_prototypes_from_data(emb_flat_tensor, labels_flat_tensor)
    model.init_family_head(n_families, class_to_family)
    
    print(f"\nProtoSSM v4 parameters: {model.count_parameters():,}")
    
    t0_final = time.time()
    model, train_history = train_proto_ssm_single(
        model,
        emb_files, logits_files, labels_files.astype(np.float32),
        site_ids_train=site_ids_all, hours_train=hours_all,
        cfg=CFG["proto_ssm_train"],
        verbose=True,
    )
    train_time = time.time() - t0_final
    print(f"Final model training time: {train_time:.1f}s")
    
    with torch.no_grad():
        final_alphas = torch.sigmoid(model.fusion_alpha).numpy()
        print(f"Fusion alpha: mean={final_alphas.mean():.4f} min={final_alphas.min():.4f} max={final_alphas.max():.4f}")
    
    # --- Train MLP probes ---
    PROBE_CLASS_IDX = np.where(Y_FULL.sum(axis=0) >= int(CFG["frozen_best_probe"]["min_pos"]))[0].astype(np.int32)
    
    probe_models = {}
    for cls_idx in tqdm(PROBE_CLASS_IDX, desc="Training MLP probes", disable=not CFG["verbose"]):
        y = Y_FULL[:, cls_idx]
        if y.sum() == 0 or y.sum() == len(y):
            continue
        X_cls = build_class_features(
            Z_FULL,
            raw_col=scores_full_raw[:, cls_idx],
            prior_col=oof_prior[:, cls_idx],
            base_col=oof_base[:, cls_idx],
        )
        n_pos = int(y.sum())
        n_neg = len(y) - n_pos
        if n_pos > 0 and n_neg > n_pos:
            repeat = max(1, n_neg // n_pos)
            pos_idx = np.where(y == 1)[0]
            X_bal = np.vstack([X_cls, np.tile(X_cls[pos_idx], (repeat, 1))])
            y_bal = np.concatenate([y, np.ones(len(pos_idx) * repeat, dtype=y.dtype)])
        else:
            X_bal, y_bal = X_cls, y
        clf = MLPClassifier(**CFG["mlp_params"])
        clf.fit(X_bal, y_bal)
        probe_models[cls_idx] = clf
    
    print(f"MLP probes trained: {len(probe_models)}")
    
    # --- Optimize ensemble weight (TRAIN MODE ONLY) ---
    if MODE == "train" and oof_proto_flat is not None:
        oof_mlp_flat = oof_base.copy()
        for cls_idx, clf in probe_models.items():
            X_cls = build_class_features(
                Z_FULL,
                raw_col=scores_full_raw[:, cls_idx],
                prior_col=oof_prior[:, cls_idx],
                base_col=oof_base[:, cls_idx],
            )
            if hasattr(clf, "predict_proba"):
                prob = clf.predict_proba(X_cls)[:, 1].astype(np.float32)
                pred = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
            else:
                pred = clf.decision_function(X_cls).astype(np.float32)
            alpha_probe = float(CFG["frozen_best_probe"]["alpha"])
            oof_mlp_flat[:, cls_idx] = (1.0 - alpha_probe) * oof_base[:, cls_idx] + alpha_probe * pred
    
        y_flat = labels_files.reshape(-1, N_CLASSES).astype(np.float32)
        best_w, best_auc, weight_results = optimize_ensemble_weight(oof_proto_flat, oof_mlp_flat, y_flat)
        ENSEMBLE_WEIGHT_PROTO = best_w
    
        mlp_only_auc = macro_auc_skip_empty(y_flat, oof_mlp_flat)
        print(f"\n=== Ensemble Optimization ===")
        print(f"Best ProtoSSM weight: {ENSEMBLE_WEIGHT_PROTO:.2f}")
        print(f"Best ensemble OOF AUC: {best_auc:.4f}")
        print(f"MLP-only OOF AUC: {mlp_only_auc:.4f}")
    
        for w, auc in weight_results:
            marker = " <-- best" if abs(w - best_w) < 0.01 else ""
            print(f"  w={w:.2f}: AUC={auc:.4f}{marker}")
    
        LOGS["ensemble_weight"] = ENSEMBLE_WEIGHT_PROTO
        LOGS["ensemble_auc"] = best_auc
        LOGS["mlp_only_auc"] = mlp_only_auc
    else:
        print(f"\nUsing default ensemble weight: ProtoSSM={ENSEMBLE_WEIGHT_PROTO:.2f}")
    
    LOGS["train_time_final"] = train_time
    LOGS["n_probe_models"] = len(probe_models)
    
    if fold_alphas:
        mean_alphas = np.stack(fold_alphas).mean(axis=0)
        print(f"\nFusion alpha (mean across folds):")
        print(f"  ProtoSSM-dominant (alpha>0.5): {(mean_alphas > 0.5).sum()} classes")
        print(f"  Perch-dominant (alpha<=0.5): {(mean_alphas <= 0.5).sum()} classes")
    
    # ─────────────────────────────────────────────────────────────────────────────
    # Locate pretrained ResidualSSM weights in attached inputs. Prefer the SGKF package when present.
    ResidualSSM_PATH_OBJ = _prefer_sgkf(Path("/kaggle/input").rglob("residual_ssm_best.pt"))
    ResidualSSM_PATH = str(ResidualSSM_PATH_OBJ) if ResidualSSM_PATH_OBJ is not None else None
    print("ResidualSSM weights:", ResidualSSM_PATH)

    # Residual SSM: second-pass boosting on first-pass errors
    # Wall-time safety: skip if > 4 min elapsed (leave max time for test inference)
    _wall_min = (time.time() - _WALL_START) / 60.0
    print(f"Wall time: {_wall_min:.1f} min")
    
    res_model = None
    CORRECTION_WEIGHT = 0.0
    
    # ─────── Fix 1: クラス定義を外に出す（Submit時にもインスタンス化できるように） ───────
    class ResidualSSM(nn.Module):
        # Lightweight SSM that takes first-pass scores + embeddings and predicts corrections.
        # Architecture: project(concat(emb, first_pass)) -> 1-layer BiSSM -> linear head
    
        def __init__(self, d_input=1536, d_scores=234, d_model=64, d_state=8,
                     n_classes=234, n_windows=12, dropout=0.1, n_sites=20, meta_dim=8):
            super().__init__()
            self.d_model = d_model
            self.n_classes = n_classes
    
            # Project embeddings + first-pass scores
            self.input_proj = nn.Sequential(
                nn.Linear(d_input + d_scores, d_model),
                nn.LayerNorm(d_model),
                nn.GELU(),
                nn.Dropout(dropout),
            )
    
            # Metadata
            self.site_emb = nn.Embedding(n_sites, meta_dim)
            self.hour_emb = nn.Embedding(24, meta_dim)
            self.meta_proj = nn.Linear(2 * meta_dim, d_model)
    
            # Positional encoding
            self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
    
            # Single bidirectional SSM layer (lightweight)
            self.ssm_fwd = SelectiveSSM(d_model, d_state)
            self.ssm_bwd = SelectiveSSM(d_model, d_state)
            self.ssm_merge = nn.Linear(2 * d_model, d_model)
            self.ssm_norm = nn.LayerNorm(d_model)
            self.ssm_drop = nn.Dropout(dropout)
    
            # Output: per-class correction (additive)
            self.output_head = nn.Linear(d_model, n_classes)
    
            # Initialize output near zero (corrections start small)
            nn.init.zeros_(self.output_head.weight)
            nn.init.zeros_(self.output_head.bias)
    
        def forward(self, emb, first_pass_scores, site_ids=None, hours=None):
            # emb: (B, T, d_input), first_pass_scores: (B, T, n_classes)
            B, T, _ = emb.shape
    
            # Concatenate embeddings with first-pass scores
            x = torch.cat([emb, first_pass_scores], dim=-1)  # (B, T, d_input + d_scores)
            h = self.input_proj(x)
    
            # Add metadata
            if site_ids is not None and hours is not None:
                site_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
                hour_e = self.hour_emb(hours.clamp(0, 23))
                meta = self.meta_proj(torch.cat([site_e, hour_e], dim=-1))
                h = h + meta.unsqueeze(1)
    
            h = h + self.pos_enc[:, :T, :]
    
            # Bidirectional SSM
            residual = h
            h_f = self.ssm_fwd(h)
            h_b = self.ssm_bwd(h.flip(1)).flip(1)
            h = self.ssm_merge(torch.cat([h_f, h_b], dim=-1))
            h = self.ssm_drop(h)
            h = self.ssm_norm(h + residual)
    
            # Output correction
            correction = self.output_head(h)  # (B, T, n_classes)
            return correction
    
        def count_parameters(self):
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
    # ────────────────────────────────────────────────────────────────────────
    
    if ResidualSSM_PATH is not None:
        print("Loading pretrained ResidualSSM...")
        load_res_path = CFG.get("pretrained_residual_path", ResidualSSM_PATH)
        
        if os.path.exists(load_res_path):
            res_cfg = CFG["residual_ssm"]
            res_model = ResidualSSM(
                d_input=emb_full.shape[1],
                d_scores=N_CLASSES,
                d_model=res_cfg["d_model"],
                d_state=res_cfg["d_state"],
                n_classes=N_CLASSES,
                n_windows=N_WINDOWS,
                dropout=res_cfg["dropout"],
                n_sites=CFG["proto_ssm"]["n_sites"],
                meta_dim=8,
            ).to(DEVICE)
            
            res_model.load_state_dict(torch.load(load_res_path, map_location=DEVICE))
            res_model.eval()
            CORRECTION_WEIGHT = res_cfg["correction_weight"]
            print(f"▶ [Load] Loaded ResidualSSM from {load_res_path}")
            LOGS["residual_ssm"] = {"skipped": False, "mode": "submit", "loaded_from": load_res_path}
        else:
            print(f"⚠️ WARNING: Pre-trained ResidualSSM not found at {load_res_path}. Skipping correction.")
            LOGS["residual_ssm"] = {"skipped": True, "mode": "submit", "reason": "weights_not_found"}
        # ────────────────────────────────────────────────────────────────────────
    
    elif _wall_min < 120.0:
        print("───────────────────────────────────")
        print("────▶▶▶Training ResidualSSM...")
        print("───────────────────────────────────")
        
        # --- Train ResidualSSM on first-pass errors ---
        
        # Step 1: Compute first-pass scores on training data
        model.eval()
        with torch.no_grad():
            emb_train_t = torch.tensor(emb_files, dtype=torch.float32)
            logits_train_t = torch.tensor(logits_files, dtype=torch.float32)
            site_train_t = torch.tensor(site_ids_all, dtype=torch.long)
            hour_train_t = torch.tensor(hours_all, dtype=torch.long)
        
            proto_train_out, _, _ = model(emb_train_t, logits_train_t,
                                           site_ids=site_train_t, hours=hour_train_t)
            proto_train_scores = proto_train_out.numpy()  # (n_files, 12, 234)
        
        # MLP probe scores on training data (flat)
        mlp_train_scores_flat = np.zeros_like(scores_full_raw, dtype=np.float32)
        
        # Get prior-fused base for MLP
        train_base_scores, train_prior_scores = fuse_scores_with_tables(
            scores_full_raw,
            sites=meta_full["site"].to_numpy(),
            hours=meta_full["hour_utc"].to_numpy(),
            tables=final_prior_tables,
        )
        mlp_train_scores_flat = train_base_scores.copy()
        
        # for cls_idx, clf in probe_models.items():
        #     X_cls = build_class_features(
        #         Z_FULL,
        #         raw_col=scores_full_raw[:, cls_idx],
        #         prior_col=train_prior_scores[:, cls_idx],
        #         base_col=train_base_scores[:, cls_idx],
        #     )
        #     if hasattr(clf, "predict_proba"):
        #         prob = clf.predict_proba(X_cls)[:, 1].astype(np.float32)
        #         pred = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        #     else:
        #         pred = clf.decision_function(X_cls).astype(np.float32)
        #     alpha_p = float(CFG["frozen_best_probe"]["alpha"])
        #     mlp_train_scores_flat[:, cls_idx] = (1 - alpha_p) * train_base_scores[:, cls_idx] + alpha_p * pred
    
        # === [Update]Processing in one line using a tensorization function ===
        alpha_p = float(CFG["frozen_best_probe"]["alpha"])
        mlp_train_scores_flat = get_vectorized_mlp_scores(
            Z_FULL, scores_full_raw, train_prior_scores, train_base_scores, 
            probe_models, alpha_p, n_windows=N_WINDOWS, device=DEVICE
        )
        
        # Reshape MLP scores to file-level
        mlp_train_scores_files, _ = reshape_to_files(mlp_train_scores_flat, meta_full)
        
        # First-pass ensemble (same formula as test-time)
        first_pass_files = (
            ENSEMBLE_WEIGHT_PROTO * proto_train_scores +
            (1 - ENSEMBLE_WEIGHT_PROTO) * mlp_train_scores_files
        ).astype(np.float32)
        
        # Step 2: Compute residuals (what the first pass got wrong)
        # Target: Y_FULL reshaped to files. Residual = target - sigmoid(first_pass)
        labels_float = labels_files.astype(np.float32)
        first_pass_probs = 1.0 / (1.0 + np.exp(-first_pass_files))
        residuals = labels_float - first_pass_probs  # in [-1, 1]
        
        print(f"First-pass training scores: {first_pass_files.shape}")
        print(f"Residuals: mean={residuals.mean():.4f}, std={residuals.std():.4f}, "
              f"abs_mean={np.abs(residuals).mean():.4f}")
        
        # Step 3: Train ResidualSSM
        res_cfg = CFG["residual_ssm"]
        res_model = ResidualSSM(
            d_input=emb_full.shape[1],
            d_scores=N_CLASSES,
            d_model=res_cfg["d_model"],
            d_state=res_cfg["d_state"],
            n_classes=N_CLASSES,
            n_windows=N_WINDOWS,
            dropout=res_cfg["dropout"],
            n_sites=CFG["proto_ssm"]["n_sites"],
            meta_dim=8,
        ).to(DEVICE)
        
        print(f"ResidualSSM parameters: {res_model.count_parameters():,}")
        
        # Train with MSE loss on residuals
        n_files = len(file_list)
        n_val = max(1, int(n_files * 0.15))
        perm = torch.randperm(n_files, generator=torch.Generator().manual_seed(123))
        val_i = perm[:n_val].numpy()
        train_i = perm[n_val:].numpy()
        
        emb_tr = torch.tensor(emb_files[train_i], dtype=torch.float32)
        fp_tr = torch.tensor(first_pass_files[train_i], dtype=torch.float32)
        res_tr = torch.tensor(residuals[train_i], dtype=torch.float32)
        site_tr = torch.tensor(site_ids_all[train_i], dtype=torch.long)
        hour_tr = torch.tensor(hours_all[train_i], dtype=torch.long)
        
        emb_va = torch.tensor(emb_files[val_i], dtype=torch.float32)
        fp_va = torch.tensor(first_pass_files[val_i], dtype=torch.float32)
        res_va = torch.tensor(residuals[val_i], dtype=torch.float32)
        site_va = torch.tensor(site_ids_all[val_i], dtype=torch.long)
        hour_va = torch.tensor(hours_all[val_i], dtype=torch.long)
        
        optimizer = torch.optim.AdamW(res_model.parameters(), lr=res_cfg["lr"], weight_decay=1e-3)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=res_cfg["lr"],
            epochs=res_cfg["n_epochs"], steps_per_epoch=1,
            pct_start=0.1, anneal_strategy='cos'
        )
        
        best_val_loss = float('inf')
        best_state = None
        wait = 0
        
        t0_res = time.time()
        for epoch in range(res_cfg["n_epochs"]):
            res_model.train()
            correction = res_model(emb_tr, fp_tr, site_ids=site_tr, hours=hour_tr)
            loss = F.mse_loss(correction, res_tr)
        
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(res_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        
            res_model.eval()
            with torch.no_grad():
                val_corr = res_model(emb_va, fp_va, site_ids=site_va, hours=hour_va)
                val_loss = F.mse_loss(val_corr, res_va)
        
            if val_loss.item() < best_val_loss:
                best_val_loss = val_loss.item()
                best_state = {k: v.clone() for k, v in res_model.state_dict().items()}
                wait = 0
            else:
                wait += 1
        
            if (epoch + 1) % 20 == 0:
                print(f"  ResidualSSM epoch {epoch+1}: train={loss.item():.6f} val={val_loss.item():.6f} wait={wait}")
        
            if wait >= res_cfg["patience"]:
                print(f"  ResidualSSM early stop at epoch {epoch+1}")
                break
        
        if best_state is not None:
            res_model.load_state_dict(best_state)
        
        res_time = time.time() - t0_res
        print(f"ResidualSSM training time: {res_time:.1f}s")
        print(f"Best val MSE: {best_val_loss:.6f}")
        
        # ─────── Fix 3: Save処理 (学習完了・最適重みロード直後に配置) ───────
        save_res_path = CFG.get("residual_model_path", "ResidualSSM/models/residual_ssm_best.pt")
        os.makedirs(os.path.dirname(save_res_path) or ".", exist_ok=True)
        torch.save(res_model.state_dict(), save_res_path)
        print(f"▶ [Save] Saved best ResidualSSM model to {save_res_path}")
        # ────────────────────────────────────────────────────────────────────
        
        # Verify correction magnitude
        res_model.eval()
        with torch.no_grad():
            all_corr = res_model(emb_train_t, torch.tensor(first_pass_files, dtype=torch.float32),
                                 site_ids=site_train_t, hours=hour_train_t)
            corr_np = all_corr.numpy()
            print(f"Correction magnitude: mean_abs={np.abs(corr_np).mean():.4f}, max={np.abs(corr_np).max():.4f}")
        
        CORRECTION_WEIGHT = res_cfg["correction_weight"]
        print(f"Correction weight: {CORRECTION_WEIGHT}")
        LOGS["residual_ssm"] = {
            "params": res_model.count_parameters(),
            "train_time": res_time,
            "best_val_mse": best_val_loss,
            "correction_mean_abs": float(np.abs(corr_np).mean()),
            "correction_weight": CORRECTION_WEIGHT,
        }
        
    else:
        print("SKIPPED ResidualSSM (wall time safety)")
        LOGS["residual_ssm"] = {"skipped": True, "wall_min": _wall_min}
    
    # Cell 15 — Diagnostics
    if MODE == "train":
        if grid_results is not None:
            best_row = grid_results.iloc[0]
            print(f"Best honest OOF probe AUC: {best_row['probe_oof_auc']:.6f}")
            print(f"Delta over honest OOF baseline: {best_row['delta']:.6f}")
    else:
        print("Skipping train diagnostics in submit mode.")
    
    # Cell 16 — Infer Perch on hidden test with embeddings
    test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
    IS_DRY_RUN = len(test_paths) == 0
    
    if IS_DRY_RUN:
        print(f"Hidden test not mounted. Dry-run on first {CFG['dryrun_n_files']} train soundscapes.")
        test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:CFG["dryrun_n_files"]]
    else:
        print(f"Hidden test files: {len(test_paths)}")
    
    # [MODIFIED - Opsi 3] Gunakan proxy_reduce terbaik dari grid search (bukan hardcode "max")
    meta_test, scores_test_raw, emb_test = infer_perch_with_embeddings(
        test_paths,
        batch_files=CFG["batch_files"],
        verbose=CFG["verbose"],
        proxy_reduce=CFG["proxy_reduce"],  # hasil grid search, default "max"
    )
    print(f"proxy_reduce used for test inference: {CFG['proxy_reduce']!r}")
    
    print("meta_test:", meta_test.shape)
    print("scores_test_raw:", scores_test_raw.shape)
    print("emb_test:", emb_test.shape)
    
    
    # Score Fusion: ProtoSSM v4 + MLP Probes + Priors + TTA (OOF-optimized weight)
    
    # --- Step 1: ProtoSSM v4 inference on test with TTA ---
    emb_test_files, test_file_list = reshape_to_files(emb_test, meta_test)
    logits_test_files, _ = reshape_to_files(scores_test_raw, meta_test)
    
    # Build test metadata
    test_site_ids, test_hours = get_file_metadata(meta_test, test_file_list, site_to_idx, CFG["proto_ssm"]["n_sites"])
    
    emb_test_tensor = torch.tensor(emb_test_files, dtype=torch.float32)
    logits_test_tensor = torch.tensor(logits_test_files, dtype=torch.float32)
    test_site_tensor = torch.tensor(test_site_ids, dtype=torch.long)
    test_hour_tensor = torch.tensor(test_hours, dtype=torch.long)
    
    # V16: TTA — average predictions from shifted temporal sequences
    model.eval()
    tta_shifts = CFG.get("tta_shifts", [0])
    if len(tta_shifts) > 1:
        print(f"Running TTA with shifts: {tta_shifts}")
        proto_scores = temporal_shift_tta(
            emb_test_files, logits_test_files, model,
            test_site_ids, test_hours, shifts=tta_shifts
        )
    else:
        with torch.no_grad():
            proto_out, _, h_test = model(emb_test_tensor, logits_test_tensor,
                                          site_ids=test_site_tensor, hours=test_hour_tensor)
            proto_scores = proto_out.numpy()
    
    # Flatten back to (n_rows, n_classes)
    proto_scores_flat = proto_scores.reshape(-1, N_CLASSES).astype(np.float32)
    
    print(f"ProtoSSM v4 test scores: {proto_scores_flat.shape}")
    print(f"Score range: {proto_scores_flat.min():.3f} to {proto_scores_flat.max():.3f}")
    
    # --- Step 2: Prior-fused base scores ---
    test_base_scores, test_prior_scores = fuse_scores_with_tables(
        scores_test_raw,
        sites=meta_test["site"].to_numpy(),
        hours=meta_test["hour_utc"].to_numpy(),
        tables=final_prior_tables,
    )
    
    # --- Step 3: MLP probe scores ---
    emb_test_scaled = emb_scaler.transform(emb_test)
    Z_TEST = emb_pca.transform(emb_test_scaled).astype(np.float32)
    
    mlp_scores = test_base_scores.copy()
    
    # for cls_idx, clf in probe_models.items():
    #     X_cls_test = build_class_features(
    #         Z_TEST,
    #         raw_col=scores_test_raw[:, cls_idx],
    #         prior_col=test_prior_scores[:, cls_idx],
    #         base_col=test_base_scores[:, cls_idx],
    #     )
    
    #     if hasattr(clf, "predict_proba"):
    #         prob = clf.predict_proba(X_cls_test)[:, 1].astype(np.float32)
    #         pred = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
    #     else:
    #         pred = clf.decision_function(X_cls_test).astype(np.float32)
    
    #     alpha = float(CFG["frozen_best_probe"]["alpha"])
    #     mlp_scores[:, cls_idx] = (1.0 - alpha) * test_base_scores[:, cls_idx] + alpha * pred
    
    # === Processing in one line using a tensorization function ===
    alpha_p = float(CFG["frozen_best_probe"]["alpha"])
    mlp_scores = get_vectorized_mlp_scores(
        Z_TEST, scores_test_raw, test_prior_scores, test_base_scores, 
        probe_models, alpha_p, n_windows=N_WINDOWS, device=DEVICE
    )
    
    # --- Step 4: Ensemble fusion with OOF-optimized weight ---
    print(f"\nUsing OOF-optimized ensemble weight: {ENSEMBLE_WEIGHT_PROTO:.2f}")
    
    final_test_scores = (
        ENSEMBLE_WEIGHT_PROTO * proto_scores_flat +
        (1.0 - ENSEMBLE_WEIGHT_PROTO) * mlp_scores
    ).astype(np.float32)
    
    # --- Step 5: Residual SSM correction (second pass) ---
    if res_model is not None and CORRECTION_WEIGHT > 0:
        first_pass_test_files, _ = reshape_to_files(final_test_scores, meta_test)
        first_pass_test_t = torch.tensor(first_pass_test_files, dtype=torch.float32)
    
        res_model.eval()
        with torch.no_grad():
            test_correction = res_model(
                emb_test_tensor, first_pass_test_t,
                site_ids=test_site_tensor, hours=test_hour_tensor
            ).numpy()
    
        test_correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)
    
        print(f"\nResidual correction: mean_abs={np.abs(test_correction_flat).mean():.4f}, "
              f"max={np.abs(test_correction_flat).max():.4f}")
    
        final_test_scores = final_test_scores + CORRECTION_WEIGHT * test_correction_flat
        print(f"Final scores (after residual): range [{final_test_scores.min():.3f}, {final_test_scores.max():.3f}]")
    else:
        print("\nResidual correction: SKIPPED")
    
    print(f"Final scores: {final_test_scores.shape}")
    
    # --- Logging ---
    test_logs = {}
    window_scores = proto_scores.reshape(-1, N_WINDOWS, N_CLASSES).mean(axis=(0, 2))
    test_logs["window_position_scores"] = window_scores.tolist()
    print(f"\nWindow position mean scores: {[f'{s:.3f}' for s in window_scores]}")
    
    if hasattr(model, 'class_to_family'):
        taxon_scores = defaultdict(list)
        idx_to_fam = {v: k for k, v in fam_to_idx.items()}
        for ci in range(N_CLASSES):
            fam_idx = class_to_family[ci]
            fam_name = idx_to_fam.get(fam_idx, f"group_{fam_idx}")
            taxon_scores[fam_name].append(float(proto_scores_flat[:, ci].mean()))
    
        test_logs["taxon_mean_scores"] = {k: float(np.mean(v)) for k, v in taxon_scores.items()}
        for k, v in sorted(taxon_scores.items(), key=lambda x: -np.mean(x[1]))[:5]:
            print(f"  {k}: mean_score={np.mean(v):.4f} (n_classes={len(v)})")
    
    with torch.no_grad():
        p_norm = F.normalize(model.prototypes, dim=-1)
        cos_sim = torch.matmul(p_norm, p_norm.T)
        cos_sim.fill_diagonal_(0)
        top_sims = cos_sim.max(dim=1)[0].numpy()
        test_logs["prototype_max_similarity"] = {
            "mean": float(top_sims.mean()),
            "max": float(top_sims.max()),
            "min": float(top_sims.min()),
        }
        print(f"\nPrototype nearest-neighbor similarity: mean={top_sims.mean():.3f}, max={top_sims.max():.3f}")
    
    
    LOGS["test_inference"] = test_logs
    
    
    
    ## Submission
    ## Temperature scaling and CSV generation.
    
    
    
    # Cell 18 — V17: Full post-processing pipeline
    
    # V17: Optimize per-class thresholds from OOF (train mode only)
    PER_CLASS_THRESHOLDS = np.full(N_CLASSES, 0.5, dtype=np.float32)
    if MODE == "train" and oof_proto_flat is not None:
        print("Optimizing per-class thresholds from OOF...")
        best_thresholds, best_scores = optimize_per_class_thresholds(
            oof_proto_flat, Y_FULL, n_windows=N_WINDOWS, thresholds=CFG["threshold_grid"]
        )
        PER_CLASS_THRESHOLDS = best_thresholds.astype(np.float32)
        print(f"  Mean threshold: {best_thresholds.mean():.3f}")
        print(f"  Threshold range: [{best_thresholds.min():.2f}, {best_thresholds.max():.2f}]")
        print(f"  Mean F1 (proxy): {best_scores.mean():.3f}")
        
        # Show classes with extreme thresholds
        high_t = np.where(best_thresholds > 0.6)[0]
        low_t = np.where(best_thresholds < 0.4)[0]
        if len(high_t) > 0:
            print(f"  High threshold classes (>0.6): {len(high_t)}")
        if len(low_t) > 0:
            print(f"  Low threshold classes (<0.4): {len(low_t)}")
    else:
        # Submit mode: use default 0.5 thresholds for all classes
        print("Using default per-class thresholds (0.5) for submit mode")
    
    
    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))
    
    # --- Step 1: Per-taxon temperature scaling ---
    temp_cfg = CFG["temperature"]
    T_AVES = temp_cfg["aves"]
    T_TEXTURE = temp_cfg["texture"]
    
    class_temperatures = np.ones(N_CLASSES, dtype=np.float32) * T_AVES
    for ci, label in enumerate(PRIMARY_LABELS):
        cn = CLASS_NAME_MAP.get(label, "Aves")
        if cn in TEXTURE_TAXA:
            class_temperatures[ci] = T_TEXTURE
    
    print(f"\nPer-taxon temperature: Aves={T_AVES}, Texture={T_TEXTURE}")
    
    scaled_scores = final_test_scores / class_temperatures[None, :]
    probs = sigmoid(scaled_scores)
    
    # --- Step 2: File-level confidence scaling ---
    top_k = CFG.get("file_level_top_k", 0)
    if top_k > 0:
        print(f"Applying file-level confidence scaling (top_k={top_k})")
        probs = file_level_confidence_scale(probs, n_windows=N_WINDOWS, top_k=top_k)
        probs = np.clip(probs, 0.0, 1.0)
    
    # --- Step 3: V17 Rank-aware post-processing ---
    if CFG.get("rank_aware_scale", False):
        power = CFG.get("rank_aware_power", 0.5)
        print(f"Applying rank-aware scaling (power={power})")
        probs = rank_aware_scaling(probs, n_windows=N_WINDOWS, power=power)
        probs = np.clip(probs, 0.0, 1.0)
    
    # --- Step 4: V17 Delta shift smoothing ---
    def adaptive_delta_smooth(probs, n_windows, base_alpha=0.20):
        n_files = probs.shape[0] // n_windows
        result = probs.copy()
        view = result.reshape(n_files, n_windows, -1)
        p_view = probs.reshape(n_files, n_windows, -1)
        for i in range(1, n_windows - 1):
            conf = p_view[:, i, :].max(axis=-1, keepdims=True)
            a = base_alpha * (1.0 - conf)
            neighbor_avg = (p_view[:, i-1, :] + p_view[:, i+1, :]) / 2.0
            view[:, i, :] = (1.0 - a) * p_view[:, i, :] + a * neighbor_avg
        return result.reshape(probs.shape)
    
    alpha = CFG.get("delta_shift_alpha", 0.0)
    if alpha > 0:
        print(f"Applying delta shift smoothing (alpha={alpha})")
        probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=alpha)
        probs = np.clip(probs, 0.0, 1.0)
    # --- Step 5: V17 Per-class threshold sharpening ---
    print(f"Applying per-class threshold sharpening...")
    probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS, n_windows=N_WINDOWS)
    
    # --- Build submission ---
    submission = pd.DataFrame(probs, columns=PRIMARY_LABELS)
    submission.insert(0, "row_id", meta_test["row_id"].values)
    submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)
    
    expected_rows = len(test_paths) * N_WINDOWS
    assert len(submission) == expected_rows, f"Expected {expected_rows}, got {len(submission)}"
    assert submission.columns.tolist() == ["row_id"] + PRIMARY_LABELS
    assert not submission.isna().any().any()
    
    if IS_DRY_RUN:
        print("Dry-run detected: Aligning yukiZ branch rows with sample_submission.csv")
        template = submission[PRIMARY_LABELS].mean(axis=0).astype(np.float32)
        submission = sample_sub.copy()
        for label in PRIMARY_LABELS:
            submission[label] = template[label]
    
    submission.to_csv(_file_name_submission, index=False)
    
    print(f"\nSaved {_file_name_submission}")
    print("Submission shape:", submission.shape)
    print(f"Final score range: {probs.min():.6f} to {probs.max():.6f}")
    print(f"Final mean: {probs.mean():.4f}")
    print(submission.iloc[:3, :8])
    
    # Cell 19 — Final Diagnostics and Logging
    
    # Save comprehensive logs
    wall_time = time.time() - _WALL_START
    LOGS["wall_time_seconds"] = wall_time
    LOGS["temperature"] = CFG["temperature"]
    LOGS["ensemble_weight_proto"] = ENSEMBLE_WEIGHT_PROTO
    LOGS["n_classes"] = N_CLASSES
    LOGS["n_windows"] = N_WINDOWS
    LOGS["cfg_proto_ssm"] = CFG["proto_ssm"]
    LOGS["cfg_proto_ssm_train"] = {k: v for k, v in CFG["proto_ssm_train"].items() if not isinstance(v, (np.ndarray,))}
    LOGS["v17_improvements"] = [
        "d_model_256", "n_ssm_layers_3", "cross_attention", "mixup", "focal_loss", "swa",
        "per_taxon_temperature", "file_level_scaling", "tta", "rank_aware_scaling",
        "delta_shift_smooth", "per_class_thresholds"
    ]
    LOGS["per_class_thresholds"] = PER_CLASS_THRESHOLDS.tolist()
    
    try:
        with open("/kaggle/working/v17_logs.json", "w") as f:
            json.dump(LOGS, f, indent=2, default=str)
        print("Saved /kaggle/working/v17_logs.json")
    except Exception as e:
        print(f"Warning: could not save logs: {e}")
    
    if MODE == "train":
        print("=== ProtoSSM v5 Training Summary ===")
        print(f"Parameters: {model.count_parameters():,}")
        print(f"d_model: {CFG['proto_ssm']['d_model']}, n_ssm_layers: {CFG['proto_ssm']['n_ssm_layers']}")
        print(f"Wall time: {wall_time:.1f}s")
        print(f"OOF CV time: {LOGS.get('oof_time', 0):.1f}s")
        print(f"Final model training time: {LOGS.get('train_time_final', 0):.1f}s")
        print(f"Final train loss: {train_history['train_loss'][-1]:.4f}")
        print(f"Best val loss: {min(train_history['val_loss']):.4f}")
        print(f"Best val AUC: {max(train_history['val_auc']):.4f}")
    
        print(f"\n=== OOF Results ===")
        print(f"ProtoSSM OOF AUC: {LOGS.get('oof_auc_proto', 0):.4f}")
        print(f"MLP-only OOF AUC: {LOGS.get('mlp_only_auc', 0):.4f}")
        print(f"Ensemble OOF AUC: {LOGS.get('ensemble_auc', 0):.4f}")
        print(f"Optimized ProtoSSM weight: {ENSEMBLE_WEIGHT_PROTO:.2f}")
    
        with torch.no_grad():
            alphas = torch.sigmoid(model.fusion_alpha).numpy()
            high_proto = (alphas > 0.5).sum()
            high_perch = (alphas <= 0.5).sum()
            print(f"\nFusion alpha distribution (final model):")
            print(f"  ProtoSSM-dominant (alpha>0.5): {high_proto} classes")
            print(f"  Perch-dominant (alpha<=0.5): {high_perch} classes")
    
        print(f"\nPer-class calibration bias stats:")
        with torch.no_grad():
            cb = model.class_bias.numpy()
            print(f"  mean={cb.mean():.4f} std={cb.std():.4f} min={cb.min():.4f} max={cb.max():.4f}")
    
        print(f"\nMLP probes: {len(probe_models)} classes")
    
        if "per_class_auc_proto" in LOGS and LOGS["per_class_auc_proto"]:
            sorted_aucs = sorted(LOGS["per_class_auc_proto"].items(), key=lambda x: x[1], reverse=True)
            print(f"\nTop 10 classes by ProtoSSM OOF AUC:")
            for label, auc in sorted_aucs[:10]:
                print(f"  {label}: {auc:.4f}")
            print(f"\nBottom 10 classes by ProtoSSM OOF AUC:")
            for label, auc in sorted_aucs[-10:]:
                print(f"  {label}: {auc:.4f}")
    
        print("\nSubmission probability stats:")
        print(submission.iloc[:, 1:].stack().describe())
    else:
        print("Submit mode completed.")
        print(f"ProtoSSM v5 parameters: {model.count_parameters():,}")
        print(f"Ensemble weight: {ENSEMBLE_WEIGHT_PROTO:.2f}")
        print(f"Wall time: {wall_time:.1f}s")
        print(f"V17 improvements: {LOGS['v17_improvements']}")
    


## Karnakbayev PowerOptimization PSSM Branch

This branch writes the small intermediate PSSM side prediction consumed by the EoS9 outer blend. The full Model-51-style output is still written for diagnostics, but the active outer blend uses `POWEROPT_PSSM_SUBM`.


In [ ]:
if MODEL_POWEROPT_PSSM in _ensemble_models:
            
    _file_name_submission = POWEROPT_PSSM_FULL_SUBM

    # EoS9 small PSSM branch. The outer blend consumes the intermediate
    # PSSM CSV saved by task2, while the full branch output remains diagnostic.

    solut = {
     'type_add'  : 'direct',
     'Models'    : [
        {
         'Model' : 'Karnakbayev_PowerOptimization_PSSM_Core',
         'subm'  : POWEROPT_PSSM_FULL_SUBM, 
         'weight': 1.0,
         'xSED'  : POWEROPT_PSSM_INTERNAL_XSED,
         'LB'    : '0.948'
        }]
    }
    
    __ensemble_models = [model['Model' ] for model in solut['Models']]
    __files_subm      = [model['subm'  ] for model in solut['Models']]
    __weights         = [model['weight'] for model in solut['Models']]
    __xsed            = [model['xSED'  ] for model in solut['Models']]
    __lbs             = [model['LB'    ] for model in solut['Models']]
    
    if 'Karnakbayev_PowerOptimization_PSSM_Core' in __ensemble_models:
        
        __file_name_submission = POWEROPT_PSSM_FULL_SUBM
    
        import os
        import subprocess
        import sys
        from pathlib import Path
        import random
        import numpy as np
        import torch

        INPUT_ROOT = Path("/kaggle/input")

        def find_optional_wheel(pattern):
            hits = sorted(INPUT_ROOT.rglob(pattern))
            return hits[0] if hits else None

        def install_optional_wheel(pattern):
            whl = find_optional_wheel(pattern)
            if whl is None:
                return False
            print("Installing bundled wheel:", whl)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(whl)], check=True)
            return True

        try:
            import onnxruntime as ort
            _ONNX_AVAILABLE = True
            print("ONNX Runtime available")
        except ImportError:
            install_optional_wheel("onnxruntime-1.24.4-*.whl")
            try:
                import onnxruntime as ort
                _ONNX_AVAILABLE = True
                print("ONNX Runtime available")
            except ImportError:
                _ONNX_AVAILABLE = False
                print("ONNX Runtime unavailable after bundled wheel search")

        try:
            import tensorflow as tf
        except ImportError:
            install_optional_wheel("tensorboard-2.20.0-*.whl")
            install_optional_wheel("tensorflow-2.20.0-*.whl")
            import tensorflow as tf

        def seed_everything(seed=42):
            random.seed(seed)
            os.environ['PYTHONHASHSEED'] = str(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        
        seed_everything(4)
        print("Global random seed set to 4")
        
        MODE = "submit"
        assert MODE in {"train", "submit"}
        print("MODE =", MODE)
        
        import os, re, gc, time, warnings
        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        warnings.filterwarnings("ignore")
        
        import pandas as pd
        import soundfile as sf
        import tensorflow as tf
        from sklearn.metrics import roc_auc_score
        from sklearn.model_selection import GroupKFold
        from scipy.ndimage import gaussian_filter1d
        from tqdm.auto import tqdm
        
        tf.experimental.numpy.experimental_enable_numpy_behavior()
        try: tf.config.set_visible_devices([], "GPU")
        except: pass
        
        _WALL_START = time.time()
        
        def find_competition_dir():
            candidates = [
                Path("/kaggle/input/competitions/birdclef-2026"),
                Path("/kaggle/input/birdclef-2026"),
            ]
            for p in candidates:
                if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
                    print("Using competition data:", p)
                    return p
            for p in Path("/kaggle/input").rglob("sample_submission.csv"):
                parent = p.parent
                if (parent / "taxonomy.csv").exists() and (parent / "train_soundscapes_labels.csv").exists():
                    print("Using competition data:", parent)
                    return parent
            raise FileNotFoundError("BirdCLEF competition data directory not found.")
    
        BASE      = find_competition_dir()
        MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
        WORK_DIR  = Path("/kaggle/working/cache")
        WORK_DIR.mkdir(parents=True, exist_ok=True)
        
        SR             = 32_000
        WINDOW_SEC     = 5
        WINDOW_SAMPLES = SR * WINDOW_SEC
        FILE_SAMPLES   = 60 * SR
        N_WINDOWS      = 12
        
        CFG = {
            "batch_files": 16,
            "oof_n_splits": 5   if MODE == "train" else 3,
            "dryrun_n_files": 20 if MODE == "train" else 0,
            "run_oof": MODE == "train",
            "verbose": MODE == "train",
            "proto_ssm_train": {
                "n_epochs":        80  if MODE == "train" else 40,
                "lr":              8e-4,
                "weight_decay":    1e-3,
                "val_ratio":       0.15,
                "patience":        20  if MODE == "train" else 8,
                "pos_weight_cap":  25.0,
                "distill_weight":  0.15,
                "proto_margin":    0.15,
                "label_smoothing": 0.03,
                "oof_n_splits":    5   if MODE == "train" else 3,
                "mixup_alpha":     0.4,
                "focal_gamma":     2.5,
                "swa_start_frac":  0.65,
                "swa_lr":          4e-4,
                "use_cosine_restart": True,
                "restart_period":  20,
            },
            "residual_ssm": {
                "d_model": 128, "d_state": 16, "n_ssm_layers": 2,
                "dropout": 0.1, "correction_weight": 0.35,
                "n_epochs": 40  if MODE == "train" else 20,
                "lr": 8e-4,
                "patience": 12  if MODE == "train" else 6,
            },
            "mlp_params": {
                "hidden_layer_sizes": (256, 128), "activation": "relu",
                "max_iter": 500  if MODE == "train" else 200,
                "early_stopping": True,
                "validation_fraction": 0.15,
                "n_iter_no_change": 20  if MODE == "train" else 10,
                "random_state": 42,
                "learning_rate_init": 5e-4,
                "alpha": 0.005,
            },
        }
        print("CFG loaded")
        
        
        # ── Data ──────────────────────────────────────────────────────────────────────
        taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
        sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
        soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
        
        PRIMARY_LABELS = sample_sub.columns[1:].tolist()
        N_CLASSES      = len(PRIMARY_LABELS)
        label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}
        
        FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")
        
        def parse_fname(name):
            m = FNAME_RE.match(name)
            if not m: return {"site": "unknown", "hour_utc": -1}
            _, site, _, hms = m.groups()
            return {"site": site, "hour_utc": int(hms[:2])}
        
        def union_labels(series):
            out = set()
            for x in series:
                if pd.notna(x):
                    for t in str(x).split(";"):
                        t = t.strip()
                        if t: out.add(t)
            return sorted(out)
        
        sc = (soundscape_labels
              .groupby(["filename", "start", "end"])["primary_label"]
              .apply(union_labels)
              .reset_index(name="label_list"))
        
        sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
        sc["row_id"]  = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)
        
        _meta = sc["filename"].apply(parse_fname).apply(pd.Series)
        sc = pd.concat([sc, _meta], axis=1)
        
        Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
        for i, lbls in enumerate(sc["label_list"]):
            for lbl in lbls:
                if lbl in label_to_idx:
                    Y_SC[i, label_to_idx[lbl]] = 1
        
        windows_per_file = sc.groupby("filename").size()
        full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
        sc["fully_labeled"] = sc["filename"].isin(full_files)
        
        full_rows = (sc[sc["fully_labeled"]]
                     .sort_values(["filename", "end_sec"])
                     .reset_index(drop=False))
        Y_FULL = Y_SC[full_rows["index"].to_numpy()]
        
        print(f"Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}")
        print(f"Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0) > 0).sum())}")
        
        
        # ── Perch backbone ────────────────────────────────────────────────────────────
        # Prefer no-DFT ONNX variant, fallback to standard ONNX. TensorFlow SavedModel
        # fallback is intentionally disabled because Kaggle TF/XLA can fail on Perch
        # StableHLO artifacts (`vhlo.cosine_v2`).
        def _find_perch_onnx(root=INPUT_ROOT):
            root = Path(root)
            patterns = [
                "**/perch_v2_no_dft*.onnx",
                "**/*perch*no*dft*.onnx",
                "**/perch_v2.onnx",
                "**/*perch*.onnx",
            ]
            for pattern in patterns:
                hits = sorted(root.glob(pattern))
                if hits:
                    print("Perch ONNX candidates for", pattern, ":", [str(p) for p in hits[:5]])
                    return hits[0]
            all_onnx = sorted(root.glob("**/*.onnx"))
            for p in all_onnx:
                s = str(p).lower().replace("-", "_")
                if "perch" in s and "no_dft" in s:
                    print("Perch ONNX selected by path tokens:", p)
                    return p
            for p in all_onnx:
                s = str(p).lower()
                if "perch" in s:
                    print("Perch ONNX selected by fallback path token:", p)
                    return p
            print("No ONNX files found under /kaggle/input" if not all_onnx else "ONNX files found but none look like Perch: " + str([str(p) for p in all_onnx[:10]]))
            return Path("")

        ONNX_PERCH_PATH = _find_perch_onnx(INPUT_ROOT)
        USE_ONNX = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()
        birdclassifier = None
        infer_fn       = None

        if not _ONNX_AVAILABLE:
            raise ImportError("onnxruntime is required for Perch inference; install it before notebook execution.")
        if not ONNX_PERCH_PATH.exists():
            raise FileNotFoundError("Perch ONNX model not found. Attach the perch-v2-no-dft-onnx dataset or another Perch .onnx asset.")

        _so = ort.SessionOptions()
        _so.intra_op_num_threads = 4
        ONNX_SESSION    = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                                providers=["CPUExecutionProvider"])
        ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
        ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
        print(f"Using ONNX Perch: {ONNX_PERCH_PATH}")
        
        def _find_perch_labels_path():
            preferred = MODEL_DIR / "assets" / "labels.csv"
            if preferred.exists():
                return preferred
            for p in sorted(Path("/kaggle/input").rglob("labels.csv")):
                try:
                    cols = set(pd.read_csv(p, nrows=0).columns)
                except Exception:
                    continue
                if {"inat2024_fsd50k", "scientific_name"} & cols:
                    print("Using Perch labels:", p)
                    return p
            raise FileNotFoundError("Perch labels.csv not found. Attach Perch ONNX labels or google/bird-vocalization-classifier.")
        
        def _load_perch_labels(path):
            df = pd.read_csv(path).reset_index().rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
            if "scientific_name" not in df.columns:
                for c in ["label", "labels", "name"]:
                    if c in df.columns:
                        df = df.rename(columns={c: "scientific_name"})
                        break
            assert "scientific_name" in df.columns, f"No scientific_name column in {path}"
            return df
        
        bc_labels = _load_perch_labels(_find_perch_labels_path())
        NO_LABEL = len(bc_labels)
        
        mapping = (taxonomy
                   .merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}),
                          on="scientific_name", how="left"))
        mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
        lbl2bc = mapping.set_index("primary_label")["bc_index"]
        
        BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
        MAPPED_MASK   = BC_INDICES != NO_LABEL
        MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
        MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)
        
        print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")
        
        import re as _re
        UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)
        CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
        TEXTURE_TAXA   = {"Amphibia", "Insecta"}
        
        proxy_map = {}
        unmapped_df = (taxonomy[taxonomy["primary_label"]
                       .isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].copy())
        
        for _, row in unmapped_df.iterrows():
            target = row["primary_label"]
            sci    = str(row["scientific_name"])
            genus  = sci.split()[0]
            hits = bc_labels[
                bc_labels["scientific_name"]
                .astype(str)
                .str.match(rf"^{_re.escape(genus)}\s", na=False)
            ]
            if len(hits) > 0:
                proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()
        
        PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
        proxy_map  = {
            idx: bc_idxs
            for idx, bc_idxs in proxy_map.items()
            if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
        }
        
        print(f"Unmapped: {len(UNMAPPED_POS)} | Proxy: {len(proxy_map)} | No signal: {len(UNMAPPED_POS)-len(proxy_map)}")
        
        
        # ── Per-taxon temperatures ────────────────────────────────────────────────────
        temperatures = np.ones(N_CLASSES, dtype=np.float32)
        for ci, label in enumerate(PRIMARY_LABELS):
            cls = CLASS_NAME_MAP.get(label, "Aves")
            temperatures[ci] = 0.95 if cls in TEXTURE_TAXA else 1.10
        
        
        # ── Perch inference engine ────────────────────────────────────────────────────
        import concurrent.futures
        
        def read_60s(path):
            y, sr = sf.read(path, dtype="float32", always_2d=False)
            if y.ndim == 2: y = y.mean(axis=1)
            if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
            else:                      y = y[:FILE_SAMPLES]
            return y
        
        def run_perch(paths, batch_files=16, verbose=True):
            paths  = [Path(p) for p in paths]
            n_rows = len(paths) * N_WINDOWS
            row_ids   = np.empty(n_rows, dtype=object)
            filenames = np.empty(n_rows, dtype=object)
            sites     = np.empty(n_rows, dtype=object)
            hours     = np.zeros(n_rows, dtype=np.int16)
            scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
            embs      = np.zeros((n_rows, 1536),      dtype=np.float32)
            wr  = 0
            itr = tqdm(range(0, len(paths), batch_files), desc="Perch") if verbose else range(0, len(paths), batch_files)
        
            with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
                next_paths   = paths[0:batch_files]
                future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
                for start in itr:
                    batch_paths  = next_paths
                    batch_n      = len(batch_paths)
                    batch_audio  = [f.result() for f in future_audio]
                    next_start = start + batch_files
                    if next_start < len(paths):
                        next_paths   = paths[next_start:next_start + batch_files]
                        future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
                    x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
                    br = wr
                    for bi, path in enumerate(batch_paths):
                        y    = batch_audio[bi]
                        meta = parse_fname(path.name)
                        stem = path.stem
                        x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                        row_ids  [wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                        filenames[wr:wr + N_WINDOWS] = path.name
                        sites    [wr:wr + N_WINDOWS] = meta["site"]
                        hours    [wr:wr + N_WINDOWS] = meta["hour_utc"]
                        wr += N_WINDOWS
                    if USE_ONNX:
                        outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                        logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
                        emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
                    else:
                        out    = infer_fn(inputs=tf.convert_to_tensor(x))
                        logits = out["label"].numpy().astype(np.float32)
                        emb    = out["embedding"].numpy().astype(np.float32)
                    scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
                    embs  [br:wr]             = emb
                    for pos_idx, bc_idxs in proxy_map.items():
                        bc_arr = np.array(bc_idxs, dtype=np.int32)
                        scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)
                    del x, logits, emb, batch_audio
                    gc.collect()
            meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames,
                                     "site": sites, "hour_utc": hours})
            return meta_df, scores, embs
        
        print("Perch inference engine defined")
        
        
        # ── Cache ─────────────────────────────────────────────────────────────────────
        print(f"USE_ONNX = {USE_ONNX}")
        
        EXTERNAL_CACHE_DIRS = [
            Path("/kaggle/input/notebooks/vyankteshdwivedi/notebook1b25083f0d"),
            Path("/kaggle/input/datasets/jaejohn/perch-meta"),
        ]
        CACHE_NAME_PAIRS = [
            ("perch_meta.parquet", "perch_arrays.npz"),
            ("full_perch_meta.parquet", "full_perch_arrays.npz"),
        ]
        CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
        CACHE_NPZ_LOCAL  = WORK_DIR / "perch_arrays.npz"
        
        def _find_external_cache():
            roots = [d for d in EXTERNAL_CACHE_DIRS if d.exists()]
            roots.append(Path("/kaggle/input"))
            seen = set()
            for root in roots:
                if not root.exists():
                    continue
                key = str(root)
                if key in seen:
                    continue
                seen.add(key)
                for meta_name, npz_name in CACHE_NAME_PAIRS:
                    meta = root / meta_name
                    npz = root / npz_name
                    if meta.exists() and npz.exists():
                        print("Using Perch cache:", meta, npz)
                        return meta, npz
                for meta_name, npz_name in CACHE_NAME_PAIRS:
                    for meta in sorted(root.rglob(meta_name)):
                        npz = meta.parent / npz_name
                        if npz.exists():
                            print("Using Perch cache:", meta, npz)
                            return meta, npz
            return None, None
        
        SCORE_KEYS = ["scores", "sc", "logits", "perch_scores", "preds", "arr_0"]
        EMB_KEYS   = ["embs", "emb", "embeddings", "features", "perch_embs", "arr_1"]
        
        def _pick_array(arr, candidates, shape_hint_cols):
            for k in candidates:
                if k in arr.files:
                    v = arr[k]
                    if getattr(v, "ndim", 0) == 2 and v.shape[1] == shape_hint_cols:
                        return v, k
                    print(f"Skipping cache key {k!r}: shape={getattr(v, 'shape', None)}, expected second dim={shape_hint_cols}")
            for k in arr.files:
                v = arr[k]
                if getattr(v, "ndim", 0) == 2 and v.shape[1] == shape_hint_cols:
                    return v, k
            raise KeyError(f"None of {candidates} found in npz. Available keys: {arr.files}")
        
        def _build_cache():
            print(f"Building Perch cache from {len(full_files)} training files…")
            train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
            train_paths = [p for p in train_paths if p.exists()]
            t0 = time.time()
            meta_built, sc_built, emb_built = run_perch(train_paths, batch_files=CFG["batch_files"], verbose=True)
            print(f"  Perch pass done in {time.time()-t0:.1f}s  scores={sc_built.shape} embs={emb_built.shape}")
            meta_built.to_parquet(CACHE_META_LOCAL)
            np.savez(CACHE_NPZ_LOCAL, scores=sc_built.astype(np.float32),
                     embs=emb_built.astype(np.float32), primary_labels=np.array(PRIMARY_LABELS))
            print(f"  Cache saved to {WORK_DIR}")
            return CACHE_META_LOCAL, CACHE_NPZ_LOCAL
        
        ext_meta, ext_npz = _find_external_cache()
        if ext_meta is not None:
            CACHE_META, CACHE_NPZ = ext_meta, ext_npz
            print(f"Using external cache: {CACHE_META.parent}")
        elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
            CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
            print(f"Using local cache: {WORK_DIR}")
        else:
            print("No cache found — building from scratch")
            CACHE_META, CACHE_NPZ = _build_cache()
        
        meta_tr = pd.read_parquet(CACHE_META)
        _arr    = np.load(CACHE_NPZ)
        sc_tr_raw,  sk = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
        emb_tr_raw, ek = _pick_array(_arr, EMB_KEYS,   1536)
        sc_tr  = sc_tr_raw.astype(np.float32)
        emb_tr = emb_tr_raw.astype(np.float32)
        
        if "primary_labels" in _arr.files:
            if _arr["primary_labels"].tolist() != PRIMARY_LABELS:
                print("  WARNING: cached primary_labels differ — scores columns may not align!")
            else:
                print("  primary_labels schema OK")
        
        if "row_id" not in meta_tr.columns:
            if "end_sec" in meta_tr.columns:
                end_sec = meta_tr["end_sec"].astype(int)
            elif "window_idx" in meta_tr.columns:
                end_sec = (meta_tr["window_idx"].astype(int) + 1) * WINDOW_SEC
            else:
                assert len(meta_tr) % N_WINDOWS == 0, "cannot infer end_sec from cache row count"
                end_sec = np.tile(np.arange(WINDOW_SEC, WINDOW_SEC * N_WINDOWS + 1, WINDOW_SEC), len(meta_tr) // N_WINDOWS)
            meta_tr["row_id"] = (meta_tr["filename"].str.replace(".ogg", "", regex=False)
                                 + "_" + end_sec.astype(str))
        if "end_sec" not in meta_tr.columns:
            if "window_idx" in meta_tr.columns:
                meta_tr["end_sec"] = (meta_tr["window_idx"].astype(int) + 1) * WINDOW_SEC
            else:
                meta_tr["end_sec"] = meta_tr["row_id"].str.rsplit("_", n=1).str[-1].astype(int)
        assert len(meta_tr) == sc_tr.shape[0] == emb_tr.shape[0], (
            f"cache row count mismatch: meta={len(meta_tr)}, sc={sc_tr.shape}, emb={emb_tr.shape}"
        )
        assert meta_tr["row_id"].is_unique, "duplicate row_id in Perch cache metadata"
        meta_tr = meta_tr.copy()
        meta_tr["_cache_pos"] = np.arange(len(meta_tr))
        order = meta_tr.sort_values(["filename", "end_sec"])["_cache_pos"].to_numpy()
        meta_tr = meta_tr.iloc[order].drop(columns=["_cache_pos"]).reset_index(drop=True)
        sc_tr = sc_tr[order]
        emb_tr = emb_tr[order]
        
        row_id_to_index = full_rows.set_index("row_id")["index"]
        missing_rows = set(meta_tr["row_id"]) - set(row_id_to_index.index)
        if missing_rows:
            raise RuntimeError(f"Cache has {len(missing_rows)} row_ids not in labeled set.")
        
        Y_FULL_aligned = Y_SC[row_id_to_index.loc[meta_tr["row_id"]].to_numpy()]
        print(f"sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  Y_FULL_aligned: {Y_FULL_aligned.shape}")
        
        
        # ── Post-processing helpers ───────────────────────────────────────────────────
        def macro_auc(y_true, y_score):
            keep = y_true.sum(axis=0) > 0
            return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")
        
        def smooth_predictions(probs, n_windows=12, alpha=0.3):
            N, C = probs.shape
            assert N % n_windows == 0
            view = probs.reshape(-1, n_windows, C).copy()
            prev_w = np.concatenate([view[:, :1, :],  view[:, :-1, :]], axis=1)
            next_w = np.concatenate([view[:, 1:,  :], view[:, -1:, :]], axis=1)
            return ((1 - alpha) * view + 0.5 * alpha * (prev_w + next_w)).reshape(N, C)
        
        
        # ── UPGRADED prior tables — joint site-hour bucket ────────────────────────────
        def build_prior_tables(sc_df, Y_labels):
            sc_df = sc_df.reset_index(drop=True)
            global_p = Y_labels.mean(axis=0).astype(np.float32)
        
            site_keys = sorted(sc_df["site"].dropna().astype(str).unique())
            site_to_i = {k: i for i, k in enumerate(site_keys)}
            site_p = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
            site_n = np.zeros(len(site_keys), dtype=np.float32)
            for s in site_keys:
                i = site_to_i[s]
                mask = sc_df["site"].astype(str).values == s
                site_n[i] = mask.sum()
                site_p[i] = Y_labels[mask].mean(axis=0)
        
            hour_keys = sorted(sc_df["hour_utc"].dropna().astype(int).unique())
            hour_to_i = {h: i for i, h in enumerate(hour_keys)}
            hour_p = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
            hour_n = np.zeros(len(hour_keys), dtype=np.float32)
            for h in hour_keys:
                i = hour_to_i[h]
                mask = sc_df["hour_utc"].astype(int).values == h
                hour_n[i] = mask.sum()
                hour_p[i] = Y_labels[mask].mean(axis=0)
        
            # Joint site-hour bucket (new — tighter shrinkage factor 4)
            sh_keys = sorted(
                {
                    (str(s), int(h))
                    for s, h in zip(sc_df["site"].dropna(), sc_df["hour_utc"].dropna())
                    if not pd.isna(s) and not pd.isna(h)
                }
            )
            sh_to_i = {k: i for i, k in enumerate(sh_keys)}
            sh_p = np.zeros((len(sh_keys), Y_labels.shape[1]), dtype=np.float32)
            sh_n = np.zeros(len(sh_keys), dtype=np.float32)
            for (s, h) in sh_keys:
                i = sh_to_i[(s, h)]
                mask = (sc_df["site"].astype(str).values == s) & (
                    sc_df["hour_utc"].astype(int).values == h
                )
                sh_n[i] = mask.sum()
                sh_p[i] = Y_labels[mask].mean(axis=0)
        
            # ── Tweak D: Circular Gaussian smoothing on hour priors ──────────────────
            # Motivation: Raw per-hour prior tables are computed from hard count buckets
            # (e.g. 06:00 UTC and 07:00 UTC are treated as independent). Many species
            # have a smooth dusk/dawn peak that leaks across adjacent hours. Applying a
            # circular Gaussian kernel (wrap-around at hour 23→0) with sigma=1.5 hrs
            # produces a more realistic, continuous prior distribution and reduces
            # over-fitting to hours that happen to have more training samples.
            # This is done on the N_hours x N_classes hour_p matrix (axis=0 = hours).
            if len(hour_keys) >= 3:  # only smooth if we have enough distinct hours
                # Build a full 24-hour grid and embed hour_p into it for wrap-around
                _full_hour_p = np.zeros((24, hour_p.shape[1]), dtype=np.float32)
                for _h, _i in hour_to_i.items():
                    _full_hour_p[int(_h)] = hour_p[_i]
                # Wrap-aware: tile 3x, smooth the middle block, then extract
                _tiled = np.tile(_full_hour_p, (3, 1))  # shape: (72, N_CLASSES)
                _tiled_smooth = gaussian_filter1d(_tiled, sigma=1.5, axis=0, mode='wrap')
                _full_smooth = _tiled_smooth[24:48]  # extract the middle 24 hours
                # Write back only the hours that exist in the training set
                for _h, _i in hour_to_i.items():
                    hour_p[_i] = _full_smooth[int(_h)]
                hour_p = np.clip(hour_p, 0.0, 1.0)
        
            return {
                "global_p": global_p,
                "site_to_i": site_to_i,
                "site_p": site_p,
                "site_n": site_n,
                "hour_to_i": hour_to_i,
                "hour_p": hour_p,
                "hour_n": hour_n,
                "sh_to_i": sh_to_i,
                "sh_p": sh_p,
                "sh_n": sh_n,
            }
        
        
        def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
            eps = 1e-4; n = len(scores); out = scores.copy()
            p = np.tile(tables["global_p"], (n, 1))
            for i, h in enumerate(hours):
                h = int(h)
                if h in tables["hour_to_i"]:
                    j = tables["hour_to_i"][h]; nh = tables["hour_n"][j]; w = nh / (nh + 8.0)
                    p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]
            for i, s in enumerate(sites):
                s = str(s)
                if s in tables["site_to_i"]:
                    j = tables["site_to_i"][s]; ns = tables["site_n"][j]; w = ns / (ns + 8.0)
                    p[i] = w * tables["site_p"][j] + (1 - w) * p[i]
            if "sh_to_i" in tables:
                for i, (s, h) in enumerate(zip(sites, hours)):
                    key = (str(s), int(h))
                    if key in tables["sh_to_i"]:
                        j = tables["sh_to_i"][key]; nsh = tables["sh_n"][j]; w = nsh / (nsh + 4.0)
                        p[i] = w * tables["sh_p"][j] + (1 - w) * p[i]
            p = np.clip(p, eps, 1 - eps)
            out += lambda_prior * (np.log(p) - np.log1p(-p))
            return out.astype(np.float32)
        
        def file_confidence_scale(probs, n_windows=12, top_k=2, power=0.4):
            N, C = probs.shape
            view      = probs.reshape(-1, n_windows, C)
            sorted_v  = np.sort(view, axis=1)
            top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)
            return (view * np.power(top_k_mean, power)).reshape(N, C)
        
        def rank_aware_scaling(probs, n_windows=12, power=0.5):
            N, C = probs.shape
            view     = probs.reshape(-1, n_windows, C)
            file_max = view.max(axis=1, keepdims=True)
            return (view * np.power(file_max, power)).reshape(N, C)
        
        def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
            N, C = probs.shape
            result = probs.copy(); view = probs.reshape(-1, n_windows, C); out = result.reshape(-1, n_windows, C)
            for t in range(n_windows):
                conf = view[:, t, :].max(axis=-1, keepdims=True); alpha = base_alpha * (1.0 - conf)
                if t == 0:           neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
                elif t == n_windows-1: neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
                else:                  neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0
                out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg
            return result
        
        
        # ── MLP probes ────────────────────────────────────────────────────────────────
        from sklearn.decomposition import PCA
        from sklearn.preprocessing import StandardScaler
        from sklearn.neural_network import MLPClassifier
        from sklearn.isotonic import IsotonicRegression
        import torch.nn as nn
        import torch.nn.functional as F
        
        def build_class_freq_weights(Y, cap=10.0):
            pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
            freq = pos_count / Y.shape[0]
            weights = np.clip(1.0 / (freq ** 0.5), 1.0, cap)
            return (weights / weights.mean()).astype(np.float32)
        
        def build_sequential_features(scores_col, n_windows=12):
            x     = scores_col.reshape(-1, n_windows)
            prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
            next_ = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
            mean  = np.repeat(x.mean(axis=1), n_windows)
            max_  = np.repeat(x.max(axis=1),  n_windows)
            std   = np.repeat(x.std(axis=1),  n_windows)
            return prev.reshape(-1), next_.reshape(-1), mean, max_, std
        
        def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
            scaler = StandardScaler()
            emb_s = scaler.fit_transform(emb)
        
            pca = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
            Z = pca.fit_transform(emb_s).astype(np.float32)
        
            print(
                f"Embedding: {emb.shape} → PCA: {Z.shape}  "
                f"(variance retained: {pca.explained_variance_ratio_.sum():.2%})"
            )
        
            class_weights = build_class_freq_weights(Y, cap=10.0)
            probe_models = {}
            active = np.where(Y.sum(axis=0) >= min_pos)[0]
            MAX_ROWS = 3000
        
            for ci in tqdm(active, desc="MLP probes"):
                y = Y[:, ci]
                if y.sum() == 0 or y.sum() == len(y):
                    continue
        
                prev, next_, mean, max_, std = build_sequential_features(scores_raw[:, ci])
                X = np.hstack(
                    [
                        Z,
                        scores_raw[:, ci : ci + 1],
                        prev[:, None],
                        next_[:, None],
                        mean[:, None],
                        max_[:, None],
                        std[:, None],
                    ]
                )
        
                n_pos = int(y.sum())
                n_neg = len(y) - n_pos
                pos_idx = np.where(y == 1)[0]
                w = float(class_weights[ci])
                repeat = max(1, min(int(round(w * n_neg / max(n_pos, 1))), 8))
                if n_pos * repeat + len(y) > MAX_ROWS:
                    repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))
        
                X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
                y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])
        
                # ── Tweak E: Wider MLP for frequent classes ──────────────────────────
                # Motivation: All probes previously used (128, 64) regardless of how
                # many positive examples a class has. For classes with ≥50 positives
                # the decision boundary is complex enough that a wider (256, 128) net
                # improves fit without overfitting, because enough data exists to
                # regularise it. Rare classes (<50 pos) keep (128, 64) to avoid
                # overfit. This mirrors the CFG['mlp_params'] hidden_layer_sizes
                # (256, 128) that was already defined but never used here.
                _hidden = (256, 128) if n_pos >= 50 else (128, 64)  # Tweak E: reuses n_pos already computed above
                clf = MLPClassifier(
                    hidden_layer_sizes=_hidden,
                    activation="relu",
                    max_iter=300,
                    early_stopping=True,
                    validation_fraction=0.15,
                    n_iter_no_change=15,
                    random_state=42,
                    learning_rate_init=5e-4,
                    alpha=0.005,
                )
                clf.fit(X_bal, y_bal)
                probe_models[ci] = clf
        
            print(f"Trained {len(probe_models)} MLP probes")
            return probe_models, scaler, pca, alpha_blend
        
        
        class VectorizedMLPProbes(nn.Module):
            """Vectorized forward pass for a homogeneous group of MLP probes.
        
            All probes passed to __init__ MUST share the same layer shapes.
            Tweak E introduced two architectures ((128,64) for rare classes and
            (256,128) for frequent ones), so the caller must split probes by
            architecture before constructing this module — see
            apply_mlp_probes_vectorized for how this is handled.
            """
        
            def __init__(self, probe_models):
                super().__init__()
                self.valid_classes = sorted(probe_models.keys())
                V = len(self.valid_classes)
        
                if V == 0:
                    self.weights = nn.ParameterList()
                    self.biases = nn.ParameterList()
                    self.n_layers = 0
                    return
        
                sample = probe_models[self.valid_classes[0]]
                self.n_layers = len(sample.coefs_)
                self.weights = nn.ParameterList()
                self.biases = nn.ParameterList()
        
                for li in range(self.n_layers):
                    W = np.stack([probe_models[c].coefs_[li] for c in self.valid_classes], axis=0)
                    b = np.stack(
                        [probe_models[c].intercepts_[li] for c in self.valid_classes], axis=0
                    )
                    self.weights.append(
                        nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False)
                    )
                    self.biases.append(
                        nn.Parameter(torch.tensor(b, dtype=torch.float32), requires_grad=False)
                    )
        
            def forward(self, x):
                h = x
                for i in range(self.n_layers):
                    h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
                    if i < self.n_layers - 1:
                        h = torch.relu(h)
                return h.squeeze(-1)
        
        
        def _run_probe_group(group_models, valid_classes_group, scores_test, Z_test, N):
            """Run VectorizedMLPProbes for one homogeneous architecture group.
        
            All probes in group_models must share the same hidden-layer shapes.
            Returns preds array of shape (len(valid_classes_group), N).
            """
            Vg = len(valid_classes_group)
            raw_g = scores_test[:, valid_classes_group].T          # (Vg, N)
            n_files = N // N_WINDOWS
            raw_view_g = raw_g.reshape(Vg, n_files, N_WINDOWS)
        
            prev_g = np.concatenate([raw_view_g[:, :, :1], raw_view_g[:, :, :-1]], axis=2).reshape(Vg, N)
            nxt_g  = np.concatenate([raw_view_g[:, :, 1:], raw_view_g[:, :, -1:]], axis=2).reshape(Vg, N)
            mean_g = np.repeat(raw_view_g.mean(axis=2), N_WINDOWS, axis=1)
            mx_g   = np.repeat(raw_view_g.max(axis=2),  N_WINDOWS, axis=1)
            std_g  = np.repeat(raw_view_g.std(axis=2),  N_WINDOWS, axis=1)
        
            scalar_g  = np.stack([raw_g, prev_g, nxt_g, mean_g, mx_g, std_g], axis=-1).astype(np.float32)
            Z_exp_g   = np.broadcast_to(Z_test, (Vg, N, Z_test.shape[1]))
            X_g       = np.concatenate([Z_exp_g.astype(np.float32), scalar_g], axis=-1)
        
            vec_probe = VectorizedMLPProbes(group_models).eval()
            with torch.no_grad():
                preds_g = vec_probe(torch.tensor(X_g)).numpy()   # (Vg, N)
            return preds_g
        
        
        def apply_mlp_probes_vectorized(
            emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.4
        ):
            """Apply MLP probes to test embeddings and scores.
        
            Tweak E fix: probes are partitioned by their hidden-layer architecture
            (tuple of layer sizes) before vectorization. Each architecture group is
            stacked separately through VectorizedMLPProbes, then results are merged
            back into the output array. This avoids the shape-mismatch error that
            arises when mixing (128,64) and (256,128) probes in the same np.stack.
            """
            if len(probe_models) == 0:
                return scores_test.copy()
        
            Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
            N = len(scores_test)
            result = scores_test.copy()
        
            # ── Partition probes by architecture (layer output sizes) ─────────────────
            def _arch_key(clf):
                """Canonical shape key: tuple of each layer's output size."""
                return tuple(w.shape[1] for w in clf.coefs_)
        
            from collections import defaultdict
            groups = defaultdict(dict)       # arch_key → {class_idx: clf}
            for ci, clf in probe_models.items():
                groups[_arch_key(clf)][ci] = clf
        
            # ── Run each architecture group separately, then blend into result ────────
            for arch, group_models in groups.items():
                valid_classes_group = sorted(group_models.keys())
                preds_g = _run_probe_group(group_models, valid_classes_group, scores_test, Z_test, N)
                # preds_g shape: (Vg, N) — transpose to (N, Vg) for column assignment
                result[:, valid_classes_group] = (
                    (1.0 - alpha_blend) * scores_test[:, valid_classes_group]
                    + alpha_blend * preds_g.T
                )
        
            return result
        
        
        def calibrate_and_optimize_thresholds(oof_probs, Y_FULL, threshold_grid=None, n_windows=12):
            if threshold_grid is None: threshold_grid = [0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]
            n_samples, n_cls = oof_probs.shape; thresholds = np.full(n_cls, 0.5, dtype=np.float32)
            n_files = n_samples // n_windows
            file_oof = oof_probs.reshape(n_files, n_windows, n_cls).max(axis=1)
            file_y   = Y_FULL.reshape(n_files, n_windows, n_cls).max(axis=1)
            n_calibrated = 0
            for c in range(n_cls):
                y_true = file_y[:, c]; y_prob = file_oof[:, c]
                if y_true.sum() < 3: continue
                try:
                    ir = IsotonicRegression(out_of_bounds="clip"); ir.fit(y_prob, y_true); y_cal = ir.transform(y_prob)
                except: y_cal = y_prob
                best_f1, best_t = 0.0, 0.5
                for t in threshold_grid:
                    pred = (y_cal >= t).astype(int)
                    tp=((pred==1)&(y_true==1)).sum(); fp=((pred==1)&(y_true==0)).sum(); fn=((pred==0)&(y_true==1)).sum()
                    prec=tp/(tp+fp+1e-8); rec=tp/(tp+fn+1e-8); f1=2*prec*rec/(prec+rec+1e-8)
                    if f1 > best_f1: best_f1,best_t = f1,t
                thresholds[c] = best_t; n_calibrated += 1
            print(f"Calibrated {n_calibrated} classes | Mean threshold: {thresholds.mean():.3f} | Range: [{thresholds.min():.2f}, {thresholds.max():.2f}]")
            return thresholds
        
        def apply_per_class_thresholds(scores, thresholds):
            C = scores.shape[1]; scaled = np.copy(scores)
            for c in range(C):
                t = thresholds[c]; above = scores[:, c] > t
                scaled[above, c]  = 0.5 + 0.5 * (scores[above, c]  - t) / (1 - t + 1e-8)
                scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
            return np.clip(scaled, 0.0, 1.0)
        
        
        # ── SSM Architecture ─────────────────────────────────────────────────────────
        class SelectiveSSM(nn.Module):
            def __init__(self, d_model, d_state=16, d_conv=4):
                super().__init__(); self.d_model=d_model; self.d_state=d_state
                self.in_proj=nn.Linear(d_model,2*d_model,bias=False)
                self.conv1d=nn.Conv1d(d_model,d_model,d_conv,padding=d_conv-1,groups=d_model)
                self.dt_proj=nn.Linear(d_model,d_model,bias=True)
                A=torch.arange(1,d_state+1,dtype=torch.float32).unsqueeze(0).expand(d_model,-1)
                self.A_log=nn.Parameter(torch.log(A)); self.D=nn.Parameter(torch.ones(d_model))
                self.B_proj=nn.Linear(d_model,d_state,bias=False); self.C_proj=nn.Linear(d_model,d_state,bias=False)
                self.out_proj=nn.Linear(d_model,d_model,bias=False)
            def forward(self,x):
                B_sz,T,D=x.shape; xz=self.in_proj(x); x_ssm,z=xz.chunk(2,dim=-1)
                x_conv=F.silu(self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2))
                dt=F.softplus(self.dt_proj(x_conv)); A=-torch.exp(self.A_log)
                B=self.B_proj(x_conv); C=self.C_proj(x_conv)
                h=torch.zeros(B_sz,D,self.d_state,device=x.device); ys=[]
                for t in range(T):
                    dA=torch.exp(A[None]*dt[:,t,:,None]); dB=dt[:,t,:,None]*B[:,t,None,:]
                    h=h*dA+x[:,t,:,None]*dB; ys.append((h*C[:,t,None,:]).sum(-1))
                return torch.stack(ys,dim=1)+x*self.D[None,None,:]
        
        class LightProtoSSM(nn.Module):
            def __init__(self,d_input=1536,d_model=128,d_state=16,n_classes=234,n_windows=12,
                         dropout=0.15,n_sites=20,meta_dim=16,use_cross_attn=True,cross_attn_heads=2):
                super().__init__(); self.n_classes=n_classes; self.n_windows=n_windows; self.use_cross_attn=use_cross_attn
                self.input_proj=nn.Sequential(nn.Linear(d_input,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
                self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
                self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
                self.meta_proj=nn.Linear(2*meta_dim,d_model)
                self.ssm_fwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
                self.ssm_bwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
                self.ssm_merge=nn.ModuleList([nn.Linear(2*d_model,d_model) for _ in range(2)])
                self.ssm_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
                self.drop=nn.Dropout(dropout)
                if use_cross_attn:
                    self.cross_attn=nn.ModuleList([nn.MultiheadAttention(d_model,cross_attn_heads,dropout=dropout,batch_first=True) for _ in range(2)])
                    self.cross_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
                self.prototypes=nn.Parameter(torch.randn(n_classes,d_model)*0.02)
                self.proto_temp=nn.Parameter(torch.tensor(5.0))
                self.class_bias=nn.Parameter(torch.zeros(n_classes))
                self.fusion_alpha=nn.Parameter(torch.zeros(n_classes))
            def init_prototypes(self,emb_tensor,labels_tensor):
                with torch.no_grad():
                    h=self.input_proj(emb_tensor)
                    for c in range(self.n_classes):
                        mask=labels_tensor[:,c]>0.5
                        if mask.sum()>0: self.prototypes.data[c]=F.normalize(h[mask].mean(0),dim=0)
            def forward(self,emb,perch_logits=None,site_ids=None,hours=None):
                B,T,_=emb.shape; h=self.input_proj(emb)+self.pos_enc[:,:T,:]
                if site_ids is not None and hours is not None:
                    meta=self.meta_proj(torch.cat([self.site_emb(site_ids),self.hour_emb(hours)],dim=-1))
                    h=h+meta[:,None,:]
                for i,(fwd,bwd,merge,norm) in enumerate(zip(self.ssm_fwd,self.ssm_bwd,self.ssm_merge,self.ssm_norm)):
                    res=h; hf=fwd(h); hb=bwd(h.flip(1)).flip(1)
                    h=self.drop(merge(torch.cat([hf,hb],dim=-1))); h=norm(h+res)
                    if self.use_cross_attn:
                        attn_out,_=self.cross_attn[i](h,h,h); h=self.cross_norm[i](h+attn_out)
                h_n=F.normalize(h,dim=-1); p_n=F.normalize(self.prototypes,dim=-1)
                sim=torch.matmul(h_n,p_n.T)*F.softplus(self.proto_temp)+self.class_bias[None,None,:]
                if perch_logits is not None:
                    alpha=torch.sigmoid(self.fusion_alpha)[None,None,:]
                    out=alpha*sim+(1-alpha)*perch_logits
                else: out=sim
                return out
        
        class ResidualSSM(nn.Module):
            def __init__(self,d_input=1536,d_scores=234,d_model=64,d_state=8,n_classes=234,
                         n_windows=12,dropout=0.1,n_sites=20,meta_dim=8):
                super().__init__(); self.n_classes=n_classes
                self.input_proj=nn.Sequential(nn.Linear(d_input+d_scores,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
                self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
                self.meta_proj=nn.Linear(2*meta_dim,d_model)
                self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
                self.ssm_fwd=SelectiveSSM(d_model,d_state); self.ssm_bwd=SelectiveSSM(d_model,d_state)
                self.ssm_merge=nn.Linear(2*d_model,d_model); self.ssm_norm=nn.LayerNorm(d_model); self.ssm_drop=nn.Dropout(dropout)
                self.output_head=nn.Linear(d_model,n_classes)
                nn.init.zeros_(self.output_head.weight); nn.init.zeros_(self.output_head.bias)
            def forward(self,emb,first_pass,site_ids=None,hours=None):
                B,T,_=emb.shape; x=torch.cat([emb,first_pass],dim=-1)
                h=self.input_proj(x)+self.pos_enc[:,:T,:]
                if site_ids is not None and hours is not None:
                    meta=self.meta_proj(torch.cat([self.site_emb(site_ids.clamp(0,self.site_emb.num_embeddings-1)),
                                                    self.hour_emb(hours.clamp(0,23))],dim=-1))
                    h=h+meta.unsqueeze(1)
                res=h; hf=self.ssm_fwd(h); hb=self.ssm_bwd(h.flip(1)).flip(1)
                h=self.ssm_drop(self.ssm_merge(torch.cat([hf,hb],dim=-1))); h=self.ssm_norm(h+res)
                return self.output_head(h)
        
        def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full, n_epochs=40, patience=8, lr=1e-3, n_sites=20, verbose=False):
            n_files=len(emb_full)//N_WINDOWS; emb_f=emb_full.reshape(n_files,N_WINDOWS,-1)
            log_f=scores_full.reshape(n_files,N_WINDOWS,-1); lab_f=Y_full.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
            fnames=meta_full["filename"].unique(); sites_u=sorted(meta_full["site"].unique())
            site2i={s:i+1 for i,s in enumerate(sites_u)}
            site_ids=np.array([min(site2i.get(meta_full.loc[meta_full["filename"]==fn,"site"].iloc[0],0),n_sites-1) for fn in fnames],dtype=np.int64)
            hour_ids=np.array([int(meta_full.loc[meta_full["filename"]==fn,"hour_utc"].iloc[0])%24 for fn in fnames],dtype=np.int64)
            model=LightProtoSSM(n_classes=N_CLASSES,n_sites=n_sites,use_cross_attn=True,cross_attn_heads=2)
            model.init_prototypes(torch.tensor(emb_full,dtype=torch.float32),torch.tensor(Y_full,dtype=torch.float32))
            emb_t=torch.tensor(emb_f,dtype=torch.float32); log_t=torch.tensor(log_f,dtype=torch.float32)
            lab_t=torch.tensor(lab_f,dtype=torch.float32); site_t=torch.tensor(site_ids,dtype=torch.long)
            hour_t=torch.tensor(hour_ids,dtype=torch.long)
            pos_cnt=lab_t.sum(dim=(0,1)); total=lab_t.shape[0]*lab_t.shape[1]
            pos_weight=((total-pos_cnt)/(pos_cnt+1)).clamp(max=25.0)
            opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
            sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,epochs=n_epochs,steps_per_epoch=1,pct_start=0.1,anneal_strategy="cos")
            best_loss,best_state,wait=float("inf"),None,0
            swa_model=torch.optim.swa_utils.AveragedModel(model); swa_start=int(n_epochs*0.65)
            swa_sched=torch.optim.swa_utils.SWALR(opt,swa_lr=4e-4)
            for ep in range(n_epochs):
                model.train()
                out=model(emb_t,log_t,site_ids=site_t,hours=hour_t)
                loss=F.binary_cross_entropy_with_logits(out,lab_t,pos_weight=pos_weight[None,None,:])+0.15*F.mse_loss(out,log_t)
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
                if ep>=swa_start: swa_model.update_parameters(model); swa_sched.step()
                else: sched.step()
                if loss.item()<best_loss:
                    best_loss=loss.item(); best_state={k:v.clone() for k,v in model.state_dict().items()}; wait=0
                else:
                    wait+=1
                    if wait>=patience: break
            if ep>=swa_start:
                torch.optim.swa_utils.update_bn(emb_t.unsqueeze(0),swa_model); model=swa_model
            else: model.load_state_dict(best_state)
            model.eval(); return model,site2i
        
        def run_tta_proto(proto_model, emb_files, sc_files, site_t, hour_t, shifts=[0, 1, -1, 2, -2]):
            proto_model.eval()
            all_preds = []
        
            emb_t = torch.tensor(emb_files, dtype=torch.float32)
            sc_t = torch.tensor(sc_files, dtype=torch.float32)
        
            for shift in shifts:
                e = torch.roll(emb_t, shift, dims=1) if shift else emb_t
                s = torch.roll(sc_t, shift, dims=1) if shift else sc_t
                with torch.no_grad():
                    out = proto_model(e, s, site_ids=site_t, hours=hour_t).numpy()
                if shift:
                    out = np.roll(out, -shift, axis=1)
                all_preds.append(out)
        
            # ── Tweak F: Temporal flip as extra TTA pass ──────────────────────────────
            # Motivation: The SSM is causal-ish (bidirectional, but trained on a fixed
            # left-to-right sequence). Reversing the time axis (flip dims=1) forces the
            # backward SSM branch to act as the forward one and vice versa, providing
            # a structurally different prediction than any shift-based augmentation.
            # The output is flipped back before averaging, so temporal order is restored.
            # Cost: one extra forward pass (~same as adding a 6th shift).
            with torch.no_grad():
                out_flip = proto_model(
                    emb_t.flip(1), sc_t.flip(1), site_ids=site_t, hours=hour_t
                ).numpy()
            all_preds.append(out_flip[:, ::-1, :].copy())  # flip output back to forward order
        
            return np.mean(all_preds, axis=0)
        
        
        def train_residual_ssm(emb_full, first_pass_flat, Y_full, site_ids, hour_ids,
                                n_epochs=30, patience=8, lr=1e-3, correction_weight=0.30, verbose=False):
            n_files=len(emb_full)//N_WINDOWS; emb_f=emb_full.reshape(n_files,N_WINDOWS,-1)
            fp_f=first_pass_flat.reshape(n_files,N_WINDOWS,-1); lab_f=Y_full.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
            fp_prob=1.0/(1.0+np.exp(-np.clip(fp_f,-30,30))); residuals=lab_f-fp_prob
            n_val=max(1,int(n_files*0.15)); rng=torch.Generator(); rng.manual_seed(42)
            perm=torch.randperm(n_files,generator=rng).numpy(); val_i=perm[:n_val]; train_i=perm[n_val:]
            emb_t=torch.tensor(emb_f,dtype=torch.float32); fp_t=torch.tensor(fp_f,dtype=torch.float32)
            res_t=torch.tensor(residuals,dtype=torch.float32)
            site_t=torch.tensor(site_ids,dtype=torch.long); hour_t=torch.tensor(hour_ids,dtype=torch.long)
            model=ResidualSSM(n_classes=N_CLASSES)
            opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
            sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,epochs=n_epochs,steps_per_epoch=1,pct_start=0.1,anneal_strategy="cos")
            best_loss,best_state,wait=float("inf"),None,0
            for ep in range(n_epochs):
                model.train()
                corr=model(emb_t[train_i],fp_t[train_i],site_ids=site_t[train_i],hours=hour_t[train_i])
                loss=F.mse_loss(corr,res_t[train_i])
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sched.step()
                model.eval()
                with torch.no_grad():
                    val_corr=model(emb_t[val_i],fp_t[val_i],site_ids=site_t[val_i],hours=hour_t[val_i])
                    val_loss=F.mse_loss(val_corr,res_t[val_i])
                if val_loss.item()<best_loss:
                    best_loss=val_loss.item(); best_state={k:v.clone() for k,v in model.state_dict().items()}; wait=0
                else:
                    wait+=1
                    if wait>=patience: break
            model.load_state_dict(best_state); return model,correction_weight
        
        print("Sequence Models defined")
        
        
        # ── Test inference ────────────────────────────────────────────────────────────
        test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
        IS_DRY_RUN = len(test_paths) == 0
        if IS_DRY_RUN:
            n = CFG["dryrun_n_files"] or 20
            print(f"No hidden test — dry-run on {n} train files")
            test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
        else:
            print(f"Hidden test files: {len(test_paths)}")
        
        meta_te, sc_te, emb_te = run_perch(test_paths, CFG["batch_files"], verbose=CFG["verbose"])
        print(f"Test scores: {sc_te.shape}")
        
        
        # ── Full ProtoSSM pipeline ────────────────────────────────────────────────────
        def sigmoid(x):
            return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))
        
        t0 = time.time()
        proto_model, site2i_tr = train_light_proto_ssm(
            emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
            n_epochs=40, patience=8, lr=1e-3, verbose=False)
        print(f"ProtoSSM training: {time.time()-t0:.1f}s")
        
        n_test_files  = len(sc_te) // N_WINDOWS
        emb_te_f      = emb_te.reshape(n_test_files, N_WINDOWS, -1)
        sc_te_f       = sc_te.reshape(n_test_files, N_WINDOWS, -1)
        
        test_fnames   = meta_te.drop_duplicates("filename")["filename"].tolist()
        n_sites_cap   = 20
        test_site_ids = np.array([min(site2i_tr.get(meta_te.loc[meta_te["filename"]==fn,"site"].iloc[0],0),n_sites_cap-1)
                                   for fn in test_fnames], dtype=np.int64)
        test_hour_ids = np.array([int(meta_te.loc[meta_te["filename"]==fn,"hour_utc"].iloc[0])%24
                                   for fn in test_fnames], dtype=np.int64)
        
        proto_out = run_tta_proto(
            proto_model,
            emb_te_f,
            sc_te_f,
            site_t=torch.tensor(test_site_ids, dtype=torch.long),
            hour_t=torch.tensor(test_hour_ids, dtype=torch.long),
            shifts=[0, 1, -1, 2, -2],
        )
        proto_scores_flat = proto_out.reshape(-1, N_CLASSES).astype(np.float32)
        
        prior_tables   = build_prior_tables(sc, Y_SC)
        sc_te_adjusted = apply_prior(sc_te, sites=meta_te["site"].to_numpy(),
                                      hours=meta_te["hour_utc"].to_numpy(), tables=prior_tables, lambda_prior=POWEROPT_PRIOR_LAMBDA)
        
        probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
            emb=emb_tr, scores_raw=sc_tr, Y=Y_FULL_aligned, min_pos=5, pca_dim=64, alpha_blend=0.4)
        sc_te_adjusted = apply_mlp_probes_vectorized(emb_te, sc_te_adjusted, probe_models, emb_scaler, emb_pca, alpha_blend)
        
        # Mapped classes keep more ProtoSSM weight; unmapped classes trust adjusted SED/MLP/prior more.
        ENSEMBLE_W_PER_CLASS = np.where(MAPPED_MASK, 0.60, 0.35).astype(np.float32)
        first_pass_flat = (
            ENSEMBLE_W_PER_CLASS[None, :] * proto_scores_flat
            + (1.0 - ENSEMBLE_W_PER_CLASS)[None, :] * sc_te_adjusted
        )
        print(
            f"[LB0.948] Per-class first-pass weights: mapped={ENSEMBLE_W_PER_CLASS[MAPPED_MASK].mean():.2f} "
            f"unmapped={ENSEMBLE_W_PER_CLASS[~MAPPED_MASK].mean():.2f}"
        )
        
        n_tr_files    = len(sc_tr) // N_WINDOWS
        emb_tr_f      = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
        sc_tr_f       = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)
        
        tr_fnames     = meta_tr.drop_duplicates("filename")["filename"].tolist()
        tr_site_ids   = np.array([min(site2i_tr.get(meta_tr.loc[meta_tr["filename"]==fn,"site"].iloc[0],0),n_sites_cap-1)
                                   for fn in tr_fnames], dtype=np.int64)
        tr_hour_ids   = np.array([int(meta_tr.loc[meta_tr["filename"]==fn,"hour_utc"].iloc[0])%24
                                   for fn in tr_fnames], dtype=np.int64)
        
        proto_tr_out = run_tta_proto(proto_model, emb_tr_f, sc_tr_f,
            site_t=torch.tensor(tr_site_ids, dtype=torch.long),
            hour_t=torch.tensor(tr_hour_ids, dtype=torch.long),
            shifts=[0, 1, -1, 2, -2])
        proto_tr_flat = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)
        
        sc_tr_prior = apply_prior(sc_tr, sites=meta_tr["site"].to_numpy(),
                                   hours=meta_tr["hour_utc"].to_numpy(), tables=prior_tables, lambda_prior=POWEROPT_PRIOR_LAMBDA)
        sc_tr_mlp = apply_mlp_probes_vectorized(emb_tr, sc_tr_prior, probe_models, emb_scaler, emb_pca, alpha_blend)
        first_pass_tr = (
            ENSEMBLE_W_PER_CLASS[None, :] * proto_tr_flat
            + (1.0 - ENSEMBLE_W_PER_CLASS)[None, :] * sc_tr_mlp
        )
        
        train_probs_for_calib = sigmoid(first_pass_tr)
        PER_CLASS_THRESHOLDS = calibrate_and_optimize_thresholds(
            oof_probs=train_probs_for_calib,
            Y_FULL=Y_FULL_aligned,
            # Tweak 3: finer threshold grid — better per-class F1 calibration for rare species
            threshold_grid=(
                [round(t, 3) for t in np.arange(0.20, 0.45, 0.025)]
                + [round(t, 3) for t in np.arange(0.45, 0.75, 0.05)]
            ),
            n_windows=N_WINDOWS,
        )
        
        # ── Tweak C: Cross-validate ResidualSSM correction_weight ───────────────────
        # Motivation: The residual correction scale (0.30) was chosen by intuition.
        # Different values can shift OOF macro-AUC by 0.5–1.5 pts depending on how
        # well the ResidualSSM generalises. We do a fast sweep over a small grid on
        # the TRAINING set (same data used to fit the model, so this is optimistic,
        # but the residual model is trained on a 15%-held-out val split which limits
        # leakage). The best weight is then applied to the test correction.
        t0 = time.time()
        res_model, correction_weight = train_residual_ssm(
            emb_full=emb_tr,
            first_pass_flat=first_pass_tr,
            Y_full=Y_FULL_aligned,
            site_ids=tr_site_ids,
            hour_ids=tr_hour_ids,
            n_epochs=30,
            patience=8,
            lr=1e-3,
            correction_weight=0.30,  # initial value; overridden by grid search below
            verbose=False,
        )
        print(f"ResidualSSM training: {time.time() - t0:.1f}s")
        
        # --- Tweak C grid search: find best correction_weight on training residuals ---
        # Original EOS-4 / Model_6 grid. Keep this unchanged to preserve the score path.
        _CORRECTION_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
        _fp_prob_tr = sigmoid(first_pass_tr)  # (N_windows_total, N_CLASSES)
        _emb_tr_f_c = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
        _fp_tr_f_c = first_pass_tr.reshape(n_tr_files, N_WINDOWS, -1)
        res_model.eval()
        with torch.no_grad():
            _tr_correction = res_model(
                torch.tensor(_emb_tr_f_c, dtype=torch.float32),
                torch.tensor(_fp_tr_f_c, dtype=torch.float32),
                site_ids=torch.tensor(tr_site_ids, dtype=torch.long),
                hours=torch.tensor(tr_hour_ids, dtype=torch.long),
            ).numpy().reshape(-1, N_CLASSES).astype(np.float32)
        
        _best_auc, _best_w = -1.0, 0.30
        for _w in _CORRECTION_GRID:
            _trial_scores = first_pass_tr + _w * _tr_correction
            _trial_probs = sigmoid(_trial_scores)
            _auc = macro_auc(Y_FULL_aligned, _trial_probs)
            print(f"  correction_weight={_w:.2f}  OOF macro-AUC={_auc:.5f}")
            if _auc > _best_auc:
                _best_auc, _best_w = _auc, _w
        
        correction_weight = _best_w  # override with CV-selected value
        print(f"[Tweak C] Best correction_weight={correction_weight:.2f}  (AUC={_best_auc:.5f})")
        del _emb_tr_f_c, _fp_tr_f_c, _tr_correction, _fp_prob_tr
        # ---------------------------------------------------------------------------
        
        first_pass_te_f = first_pass_flat.reshape(n_test_files, N_WINDOWS, -1)
        res_model.eval()
        with torch.no_grad():
            test_correction = res_model(
                torch.tensor(emb_te_f,         dtype=torch.float32),
                torch.tensor(first_pass_te_f,  dtype=torch.float32),
                site_ids=torch.tensor(test_site_ids, dtype=torch.long),
                hours   =torch.tensor(test_hour_ids, dtype=torch.long),
            ).numpy()
        correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)
        final_scores    = first_pass_flat + correction_weight * correction_flat
        final_scores    = final_scores / temperatures[None, :]
        probs = sigmoid(final_scores)
        probs = file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=POWEROPT_FILE_CONFIDENCE_POWER)
        probs = rank_aware_scaling(probs,    n_windows=N_WINDOWS, power=POWEROPT_RANK_AWARE_POWER)
        probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=POWEROPT_DELTA_SMOOTH_ALPHA)
        probs = np.clip(probs, 0.0, 1.0)
        probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS) # ←now applied
        
        sub = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
        sub.insert(0, "row_id", meta_te["row_id"].values)

        ###############################################################################
        
        sub.to_csv("submission_protossm.csv", index=False)                  # original

        ###############################################################################

        if 'task2' in solutions and 'save PowerOptimization PSSM as' in solutions['task2']: 

            _subm_52p_csv = solutions['task2']['save PowerOptimization PSSM as']
            
            # The outer blend consumes this intermediate PSSM branch directly.
            # In public/dry-run mode the ProtoSSM rows can come from fallback audio,
            # while the other branches are aligned to sample_submission.csv. Align here
            # as well so the safe outer blend does not mix incompatible row sets.
            _task2_base = BASE_PATH if 'BASE_PATH' in globals() else BASE
            _task2_test_paths = list(Path(_task2_base).glob("test_soundscapes/*.ogg"))
            if len(_task2_test_paths) == 0:
                print("Dry-run detected: Aligning PowerOptimization PSSM rows with sample_submission.csv")
                _sample_task2 = pd.read_csv(Path(_task2_base) / "sample_submission.csv")
                _task2_cols = [c for c in _sample_task2.columns if c != "row_id"]
                _template_task2 = sub[_task2_cols].mean(axis=0).astype(np.float32)
                _sub_task2 = _sample_task2.copy()
                for _label_task2 in _task2_cols:
                    _sub_task2[_label_task2] = _template_task2[_label_task2]
            else:
                _sub_task2 = sub
            
            _sub_task2.to_csv(_subm_52p_csv, index=False)
            
            print('\n','#'*21, '\n _subm_52p_csv \n','#'*21, '\n')

        ###############################################################################
        
        print("ProtoSSM execution complete")
        print(f"Total wall time so far: {(time.time() - _WALL_START)/60:.1f} min")
        del emb_tr_f, sc_tr_f, proto_model, res_model
        gc.collect()
        print("Memory freed. Ready for SED cell.")

        if _runSED_once and not os.path.isfile("submission_sed.csv"):
        
            import librosa
            from scipy.ndimage import gaussian_filter1d
            
            N_MELS_SED = 256
            N_FFT_SED  = 2048
            HOP_SED    = 512
            FMIN_SED   = 20
            FMAX_SED   = 16000
            TOP_DB_SED = 80
            
            def find_sed_dir():
                hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
                if not hits:
                    raise FileNotFoundError("sed_fold0.onnx not found. Attach tuckerarrants/bc2026-distilled-sed-public.")
                return hits[0].parent
        
            def make_sed_session(path):
                so = ort.SessionOptions()
                so.intra_op_num_threads = 4
                so.inter_op_num_threads = 1
                so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
                return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])
            
            def audio_to_mel(chunks):
                mels = []
                for x in chunks:
                    s = librosa.feature.melspectrogram(y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
                                                        n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0)
                    s = librosa.power_to_db(s, top_db=TOP_DB_SED)
                    s = (s - s.mean()) / (s.std() + 1e-6)
                    mels.append(s)
                return np.stack(mels)[:, None].astype(np.float32)
        
            def file_to_sed_chunks(path):
                y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
                if y.ndim == 2: y = y.mean(axis=1)
                if sr0 != SR: y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
                n = 60 * SR
                if len(y) < n: y = np.pad(y, (0, n - len(y)))
                else:          y = y[:n]
                chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                ends   = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC
                return chunks, ends
            
            def sigmoid_sed(x):
                return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)
        
            # Use the same test files as Cell 1
            test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
            IS_DRY_RUN = len(test_paths) == 0
            if IS_DRY_RUN:
                dry_n = CFG["dryrun_n_files"] if "CFG" in dir() else 20
                test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:(dry_n or 20)]
            
            sed_dir = find_sed_dir()
            sed_fold_paths = sorted(sed_dir.glob("sed_fold*.onnx"),
                                     key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
            sed_sessions = [make_sed_session(p) for p in sed_fold_paths]
            
            print(f"SED dir: {sed_dir}")
            print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")
            
            sed_rows, sed_preds = [], []
        
            for i, path in enumerate(test_paths, 1):
                chunks, ends = file_to_sed_chunks(path)
                mel = audio_to_mel(chunks)
                p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)
            
                for sess in sed_sessions:
                    outs = sess.run(None, {sess.get_inputs()[0].name: mel})
                    clip_logits = outs[0]             # (12, 234)
                    frame_max   = outs[1].max(axis=1) # (12, 234)
                    p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)
            
                p_mean = p_sum / len(sed_sessions)
            
                if len(p_mean) > 1:
                    p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)
            
                stem = path.stem
                sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
                sed_preds.append(p_mean)
            
                if i == 1 or i % 50 == 0 or i == len(test_paths):
                    print(f"SED: {i}/{len(test_paths)}")
        
            sed_preds_arr = np.concatenate(sed_preds, axis=0)
            sed_sub = pd.DataFrame(np.clip(sed_preds_arr, 0.0, 1.0), columns=PRIMARY_LABELS)
            sed_sub.insert(0, "row_id", sed_rows)
            sed_sub.to_csv("submission_sed.csv", index=False)
            print(f"Distilled SED Processing Complete. Shape: {sed_sub.shape}")
        
        
        import os
        import numpy as np
        import pandas as pd
        from pathlib import Path
        
        PROTOSSM_CSV = "submission_protossm.csv"
        SED_CSV      = "submission_sed.csv"
        OUT_CSV      = "submission.csv"
        EPS = 1e-5
        
        df_proto = pd.read_csv(PROTOSSM_CSV)
        df_sed   = pd.read_csv(SED_CSV)
        
        cols = [c for c in df_proto.columns if c != "row_id"]
        
        # Align row order
        df_sed = df_sed.set_index("row_id").loc[df_proto["row_id"]].reset_index()
        p_proto = np.clip(df_proto[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
        p_sed   = np.clip(df_sed[cols].to_numpy(np.float32),   EPS, 1.0 - EPS)
        
        rank_proto = pd.DataFrame(p_proto).rank(axis=0, pct=True).to_numpy(np.float32)
        rank_sed   = pd.DataFrame(p_sed).rank(axis=0, pct=True).to_numpy(np.float32)

        # +++++++++++++++++++++++ xSED rank blend ++++++++++++++++++++++++++++++++++
        
        MODEL_NAME = "Karnakbayev_PowerOptimization_PSSM_Core"
        _this_model = next(m for m in solut["Models"] if m["Model"] == MODEL_NAME)

        # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

        if _this_model["xSED"] != []:
            PROTO_W, SED_W = [float(v) for v in _this_model.get("xSED", [0.585, 0.415])]
        else:
            PROTO_W, SED_W = [0.60, 0.40]

        # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
                                                                    
        _xsed_sum = PROTO_W + SED_W
        if _xsed_sum <= 0:
            raise ValueError(f"Invalid xSED weights for {MODEL_NAME}: {_this_model.get('xSED')}")
                 
        PROTO_W, SED_W = PROTO_W / _xsed_sum, SED_W / _xsed_sum
                 
        print(f"Executing xSED rank blend ({PROTO_W:.4f} Proto / {SED_W:.4f} SED)")
                 
        pred = (rank_proto * PROTO_W) + (rank_sed * SED_W)

        # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    
        row_ids  = df_proto["row_id"].astype(str).to_numpy()
        file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])

        # ── Gate 1: Noise suppression ─────────────────────────────────────────────────
        # If ProtoSSM is confident but SED strongly disagrees → trust ProtoSSM more
        fake_only = (p_proto > 0.50) & (p_sed < 0.05)
        pred = np.where(fake_only, (1.0 - 0.08) * pred + 0.08 * rank_proto, pred)
        
        # ── Gate 2: Temporal continuity (fat-tailed t-distribution kernel) ─────────────
        # 310-second context window to protect continuous calls across windows
        offs = np.arange(-3, 4, dtype=np.float32)
        proto_kernel = (1.0 + (offs / 1.20) ** 2 / 2.0) ** (-1.5)
        proto_kernel = (proto_kernel / proto_kernel.sum()).astype(np.float32)
        
        pa_ctx = p_proto.copy()
        for fid in pd.unique(file_ids):
            m  = file_ids == fid
            x  = p_proto[m]
            if len(x) > 1:
                xp = np.pad(x, ((3, 3), (0, 0)), mode="edge")
                pa_ctx[m] = sum(proto_kernel[i] * xp[i:i + len(x)] for i in range(7))
        
        xctx = pd.DataFrame(pa_ctx).rank(axis=0, pct=True).to_numpy(np.float32)
        proto_cont = (xctx > 0.88) & (rank_proto > 0.75) & (p_sed < 0.12) & (~fake_only)
        pred = np.where(proto_cont,
                        (1.0 - 0.15) * pred + 0.15 * np.maximum(rank_proto, xctx),
                        pred)
        
        # ── Gate 3: SED spike preservation ────────────────────────────────────────────
        # Brief high-confidence SED detections that ProtoSSM missed
        sed_only = (rank_sed > 0.95) & (rank_proto < 0.80) & (~fake_only) & (~proto_cont)
        pred = np.where(sed_only, (1.0 - 0.12) * pred + 0.12 * rank_sed, pred)
        sub = df_proto.copy()
        sub[cols] = pred.astype(np.float32)
        
        # ── Gate 4: Sonotype mirroring ────────────────────────────────────────────────
        # Max-pool across visually identical species groups
        MIRROR_PAIRS = (
            ("47158son15", "47158son16"),
            ("47158son09", "47158son12"),
            ("47158son02", "47158son14"),
            ("47158son13", "47158son21", "47158son22", "47158son23")
        )
        col_to_idx = {l: i for i, l in enumerate(cols)}
        
        mirror_count = 0
        for group in MIRROR_PAIRS:
            valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
            if len(valid_idx) >= 2:
                group_max = sub[cols].iloc[:, valid_idx].max(axis=1).to_numpy(np.float32)
                for idx in valid_idx:
                    sub.iloc[:, idx + 1] = group_max
                mirror_count += len(valid_idx)
        print(f"Sonotype mirroring applied to {mirror_count} columns.")
        
        # ── Gate 5: Adaptive rare-class thresholding ──────────────────────────────────
        BASE_PATH = BASE
        try:
            tax_df = pd.read_csv(BASE_PATH / "taxonomy.csv").set_index("primary_label")
            rare_classes = {"Amphibia", "Mammalia", "Reptilia"}
            rare_count = 0
            for ci, species in enumerate(cols):
                if species in tax_df.index and tax_df.loc[species, "class_name"] in rare_classes:
                    col_idx = ci + 1
                    vals = sub.iloc[:, col_idx].to_numpy(np.float32)
                    thr = vals.mean() + 0.05
                    sub.iloc[:, col_idx] = np.where(vals < thr, vals * 0.9, vals)
                    rare_count += 1
            print(f"Adaptive thresholding applied to {rare_count} rare species.")
        except Exception as e:
            print(f"Adaptive thresholding skipped: {e}")
        
        # ── Dry-run alignment ──────────────────────────────────────────────────────────
        test_paths = list(BASE_PATH.glob("test_soundscapes/*.ogg"))
        IS_DRY_RUN = len(test_paths) == 0
        if IS_DRY_RUN:
            print("Dry-run detected: Aligning rows with sample_submission.csv")
            sample_public = pd.read_csv(BASE_PATH / "sample_submission.csv")
            template = sub[cols].mean(axis=0).astype(np.float32)
            sub = sample_public.copy()
            for label in cols:
                sub[label] = template[label]
        
        sub.to_csv(__file_name_submission, index=False)
        print(f"Blend and post-processing complete. Saved {__file_name_submission} shape={sub.shape}")
        print("Ready for submission!")
        
        
        # Final submission diagnostics: does not alter submission.csv
        from pathlib import Path
        import numpy as np
        import pandas as pd
        from IPython.display import display, Markdown
        
        submission_path = Path(__file_name_submission)
        assert submission_path.exists(), f"{__file_name_submission} was not created. Run the blend cell first."
        
        sub_check = pd.read_csv(submission_path)
        prob_cols = [c for c in sub_check.columns if c != "row_id"]
        
        summary = pd.DataFrame({
            "check": [
                "rows",
                "columns",
                "class columns",
                "missing values",
                "min probability",
                "max probability",
                "duplicated row_id",
            ],
            "value": [
                len(sub_check),
                sub_check.shape[1],
                len(prob_cols),
                int(sub_check.isna().sum().sum()),
                float(sub_check[prob_cols].min().min()) if prob_cols else np.nan,
                float(sub_check[prob_cols].max().max()) if prob_cols else np.nan,
                int(sub_check["row_id"].duplicated().sum()) if "row_id" in sub_check.columns else "row_id missing",
            ]
        })
        
        display(Markdown("### ✅ Submission diagnostic summary"))
        display(summary)
        
        assert "row_id" in sub_check.columns, "row_id column is missing."
        assert len(prob_cols) > 0, "No class probability columns found."
        assert np.isfinite(sub_check[prob_cols].to_numpy()).all(), "Non-finite values found in probability columns."
        assert sub_check[prob_cols].min().min() >= 0.0, "Probability columns contain values below 0."
        assert sub_check[prob_cols].max().max() <= 1.0, "Probability columns contain values above 1."
        
        print(f"{__file_name_submission} passed basic diagnostics.")
    
    
    import pandas as pd, os, time, sys
    import numpy as np
    from pathlib import Path
    from warnings import filterwarnings; filterwarnings("ignore")
    
    def _read_submission_checked(path):
        df = pd.read_csv(path)
        assert "row_id" in df.columns, f"row_id column missing in {path}"
        assert not any(str(c).startswith("Unnamed") for c in df.columns), f"unexpected unnamed column in {path}: {df.columns.tolist()[:5]}"
        assert df["row_id"].is_unique, f"duplicate row_id values in {path}"
        prob_cols = [c for c in df.columns if c != "row_id"]
        assert prob_cols, f"no probability columns in {path}"
        values = df[prob_cols].to_numpy(dtype=np.float32)
        assert np.isfinite(values).all(), f"NaN/inf values in {path}"
        assert values.min() >= 0.0 and values.max() <= 1.0, f"probabilities outside [0, 1] in {path}"
        out = df.set_index("row_id")
        out.index = out.index.astype(str)
        out.index.name = "row_id"
        return out
    
    def direct_add_safe():
        print(f'Ensemble: {__ensemble_models},   LB: {__lbs},   weights: {__weights}')
        assert len(__files_subm) == len(__weights), "submission file / weight length mismatch"
        weight_sum = float(sum(__weights))
        assert weight_sum > 0, "ensemble weights must sum to a positive value"
        if not np.isclose(weight_sum, 1.0, atol=1e-6):
            print(f"Normalizing ensemble weights from sum={weight_sum:.6f}")
        norm_weights = [float(w) / weight_sum for w in __weights]
        dfs = [_read_submission_checked(path) for path in __files_subm]
        base_idx = dfs[0].index
        base_cols = dfs[0].columns
        for path, df in zip(__files_subm, dfs):
            assert df.columns.equals(base_cols), f"Column mismatch in {path}"
            missing = base_idx.difference(df.index)
            extra = df.index.difference(base_idx)
            assert len(missing) == 0 and len(extra) == 0, (
                f"row_id mismatch in {path}: missing={len(missing)}, extra={len(extra)}"
            )
        out = sum(w * df.loc[base_idx, base_cols] for w, df in zip(norm_weights, dfs))
        out.index.name = "row_id"
        values = out.to_numpy(dtype=np.float32)
        assert np.isfinite(values).all(), "NaN/inf in final blend"
        assert values.min() >= 0.0 and values.max() <= 1.0, "final probabilities outside [0, 1]"
        return out
    
    def direct_add2(): return direct_add_safe()
    def direct_add3(): return direct_add_safe()
    def direct():      return direct_add_safe()
    def rank_1():      raise RuntimeError("rank_1 is disabled inside the PowerOptimization branch; use direct only.")
    
    def _find_sample_submission_path():
        base_obj = globals().get("BASE", None)
        if base_obj is not None:
            p = Path(base_obj) / "sample_submission.csv"
            if p.exists():
                return p
        for p in [
            Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv"),
            Path("/kaggle/input/birdclef-2026/sample_submission.csv"),
        ]:
            if p.exists():
                return p
        root = Path("/kaggle/input")
        if root.exists():
            hits = sorted(root.rglob("sample_submission.csv"))
            for p in hits:
                if (p.parent / "taxonomy.csv").exists():
                    return p
        return None
    
    def _as_explicit_submission_table(pred):
        if isinstance(pred, pd.DataFrame) and "row_id" in pred.columns:
            df = pred.copy()
        elif isinstance(pred, pd.DataFrame) and pred.index.name == "row_id":
            df = pred.reset_index()
        else:
            raise AssertionError("final prediction must be a DataFrame with a row_id column or row_id index")
        assert "row_id" in df.columns, "row_id column missing after final conversion"
        assert not any(str(c).startswith("Unnamed") for c in df.columns), f"unexpected unnamed columns: {df.columns.tolist()[:5]}"
        df["row_id"] = df["row_id"].astype(str)
        assert df["row_id"].is_unique, "duplicate row_id in final submission"
        return df
    
    def _align_to_sample_submission_if_possible(df):
        sample_path = _find_sample_submission_path()
        if sample_path is None:
            return df
        sample = pd.read_csv(sample_path)
        assert "row_id" in sample.columns, f"sample_submission has no row_id: {sample_path}"
        sample["row_id"] = sample["row_id"].astype(str)
        assert sample["row_id"].is_unique, f"duplicate row_id in sample_submission: {sample_path}"
        sample_cols = sample.columns.tolist()
        missing_cols = [c for c in sample_cols if c not in df.columns]
        assert not missing_cols, f"final submission is missing sample columns: {missing_cols[:5]}"
        final_ids = set(df["row_id"])
        sample_ids = set(sample["row_id"])
        if final_ids == sample_ids:
            aligned = df.set_index("row_id").loc[sample["row_id"], sample_cols[1:]].reset_index()
            aligned.columns = sample_cols
            return aligned
        missing = sorted(sample_ids - final_ids)[:5]
        extra = sorted(final_ids - sample_ids)[:5]
        raise AssertionError(
            "final row_id set differs from sample_submission: "
            f"missing={len(sample_ids-final_ids)} first={missing}, extra={len(final_ids-sample_ids)} first={extra}"
        )
    
    def write_final_submission(pred, path="submission.csv"):
        df = _as_explicit_submission_table(pred)
        df = _align_to_sample_submission_if_possible(df)
        prob_cols = [c for c in df.columns if c != "row_id"]
        assert prob_cols, "no probability columns in final submission"
        values = df[prob_cols].to_numpy(dtype=np.float32)
        assert np.isfinite(values).all(), "NaN/inf in final submission"
        assert values.min() >= 0.0 and values.max() <= 1.0, "final probabilities outside [0, 1]"
        df.to_csv(path, index=False)
        check = pd.read_csv(path)
        assert check.columns.tolist() == df.columns.tolist(), "written submission columns changed on reload"
        assert len(check) == len(df), "written submission row count changed on reload"
        assert check["row_id"].is_unique, "duplicate row_id after reload"
        print(f"Wrote {path}: rows={len(df)}, cols={df.shape[1]}, min={values.min():.6f}, max={values.max():.6f}")
        return df
    
    
    if solut['type_add'] in {'rank', 'rank.1'} : f_add = rank_1
    if solut['type_add'] == 'direct'           : f_add = direct
    
    submission = f_add()
    final_submission = write_final_submission(submission, POWEROPT_PSSM_FULL_SUBM)

    final_submission.to_csv(POWEROPT_PSSM_FULL_SUBM, index=False)
    
    display(final_submission.head(3))


## Karnakbayev PowerOptimization EoS9 Branch

Dominant EoS9 branch. It uses the refreshed Model-74-style PowerOptimization path with the configured ProtoSSM/SED rank balance, stronger test prior, and EoS9 rank-aware scaling.


In [ ]:
if MODEL_POWEROPT_SZ2 in _ensemble_models:

    _file_name_submission = POWEROPT_SZ2_SUBM
    
    solut = {
     'type_add'  : 'direct',
     'Models'    : [
        {
         'Model' : 'Karnakbayev_PowerOptimization_LB0948_74',
         'subm'  : POWEROPT_SZ2_CORE_SUBM, 
         'weight': 1.0,
         'xSED'  : MODEL_REGISTRY[MODEL_POWEROPT_SZ2]['xSED'],
         'LB'    : '0.948'
        }]
    }
    
    __ensemble_models = [model['Model' ] for model in solut['Models']]
    __files_subm      = [model['subm'  ] for model in solut['Models']]
    __weights         = [model['weight'] for model in solut['Models']]
    __xsed            = [model['xSED'  ] for model in solut['Models']]
    __lbs             = [model['LB'    ] for model in solut['Models']]
    
    if 'Karnakbayev_PowerOptimization_LB0948_74' in __ensemble_models:
        
        __file_name_submission = solut['Models'][0]['subm']
    
        import subprocess, sys, os
        from pathlib import Path
        import random
        import numpy as np
        import torch
        
        INPUT_ROOT = Path("/kaggle/input")
        
        def find_optional_wheel(pattern):
            hits = sorted(INPUT_ROOT.rglob(pattern))
            return hits[0] if hits else None
    
        def install_optional_wheel(pattern):
            whl = find_optional_wheel(pattern)
            if whl is None:
                return False
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(whl)], check=True)
            return True
        
        try:
            import onnxruntime as ort
            _ONNX_AVAILABLE = True
            print("ONNX Runtime available")
        except ImportError:
            install_optional_wheel("onnxruntime-1.24.4-*.whl")
            try:
                import onnxruntime as ort
                _ONNX_AVAILABLE = True
                print("ONNX Runtime available")
            except ImportError:
                _ONNX_AVAILABLE = False
                print("ONNX not available, falling back to TF")
    
        try:
            import tensorflow as tf
        except ImportError:
            install_optional_wheel("tensorboard-2.20.0-*.whl")
            install_optional_wheel("tensorflow-2.20.0-*.whl")
            import tensorflow as tf
        
        def seed_everything(seed=42):
            random.seed(seed)
            os.environ['PYTHONHASHSEED'] = str(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        
        seed_everything(4)
        print("Global random seed set to 4")
        
        MODE = "submit"
        assert MODE in {"train", "submit"}
        print("MODE =", MODE)
        
        import os, re, gc, time, warnings
        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        warnings.filterwarnings("ignore")
        
        import pandas as pd
        import soundfile as sf
        import tensorflow as tf
        from sklearn.metrics import roc_auc_score
        from sklearn.model_selection import GroupKFold
        from scipy.ndimage import gaussian_filter1d
        from tqdm.auto import tqdm
        
        tf.experimental.numpy.experimental_enable_numpy_behavior()
        try: tf.config.set_visible_devices([], "GPU")
        except: pass
        
        _WALL_START = time.time()
        
        def find_competition_dir():
            candidates = [
                Path("/kaggle/input/competitions/birdclef-2026"),
                Path("/kaggle/input/birdclef-2026"),
            ]
            for p in candidates:
                if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
                    print("Using competition data:", p)
                    return p
            for p in Path("/kaggle/input").rglob("sample_submission.csv"):
                parent = p.parent
                if (parent / "taxonomy.csv").exists() and (parent / "train_soundscapes_labels.csv").exists():
                    print("Using competition data:", parent)
                    return parent
            raise FileNotFoundError("BirdCLEF competition data directory not found.")
    
        BASE      = find_competition_dir()
        MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
        WORK_DIR  = Path("/kaggle/working/cache")
        WORK_DIR.mkdir(parents=True, exist_ok=True)
        
        SR             = 32_000
        WINDOW_SEC     = 5
        WINDOW_SAMPLES = SR * WINDOW_SEC
        FILE_SAMPLES   = 60 * SR
        N_WINDOWS      = 12
        
        CFG = {
            "batch_files": 16,
            "oof_n_splits": 5   if MODE == "train" else 3,
            "dryrun_n_files": 20 if MODE == "train" else 0,
            "run_oof": MODE == "train",
            "verbose": MODE == "train",
            "proto_ssm_train": {
                "n_epochs":        80  if MODE == "train" else 40,
                "lr":              8e-4,
                "weight_decay":    1e-3,
                "val_ratio":       0.15,
                "patience":        20  if MODE == "train" else 8,
                "pos_weight_cap":  25.0,
                "distill_weight":  0.15,
                "proto_margin":    0.15,
                "label_smoothing": 0.03,
                "oof_n_splits":    5   if MODE == "train" else 3,
                "mixup_alpha":     0.4,
                "focal_gamma":     2.5,
                "swa_start_frac":  0.65,
                "swa_lr":          4e-4,
                "use_cosine_restart": True,
                "restart_period":  20,
            },
            "residual_ssm": {
                "d_model": 128, "d_state": 16, "n_ssm_layers": 2,
                "dropout": 0.1, "correction_weight": 0.35,
                "n_epochs": 40  if MODE == "train" else 20,
                "lr": 8e-4,
                "patience": 12  if MODE == "train" else 6,
            },
            "mlp_params": {
                "hidden_layer_sizes": (256, 128), "activation": "relu",
                "max_iter": 500  if MODE == "train" else 200,
                "early_stopping": True,
                "validation_fraction": 0.15,
                "n_iter_no_change": 20  if MODE == "train" else 10,
                "random_state": 42,
                "learning_rate_init": 5e-4,
                "alpha": 0.005,
            },
        }
        print("CFG loaded")
        
        
        # ── Data ──────────────────────────────────────────────────────────────────────
        taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
        sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
        soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
        
        PRIMARY_LABELS = sample_sub.columns[1:].tolist()
        N_CLASSES      = len(PRIMARY_LABELS)
        label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}
        
        FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")
        
        def parse_fname(name):
            m = FNAME_RE.match(name)
            if not m: return {"site": "unknown", "hour_utc": -1}
            _, site, _, hms = m.groups()
            return {"site": site, "hour_utc": int(hms[:2])}
        
        def union_labels(series):
            out = set()
            for x in series:
                if pd.notna(x):
                    for t in str(x).split(";"):
                        t = t.strip()
                        if t: out.add(t)
            return sorted(out)
        
        sc = (soundscape_labels
              .groupby(["filename", "start", "end"])["primary_label"]
              .apply(union_labels)
              .reset_index(name="label_list"))
        
        sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
        sc["row_id"]  = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)
        
        _meta = sc["filename"].apply(parse_fname).apply(pd.Series)
        sc = pd.concat([sc, _meta], axis=1)
        
        Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
        for i, lbls in enumerate(sc["label_list"]):
            for lbl in lbls:
                if lbl in label_to_idx:
                    Y_SC[i, label_to_idx[lbl]] = 1
        
        windows_per_file = sc.groupby("filename").size()
        full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
        sc["fully_labeled"] = sc["filename"].isin(full_files)
        
        full_rows = (sc[sc["fully_labeled"]]
                     .sort_values(["filename", "end_sec"])
                     .reset_index(drop=False))
        Y_FULL = Y_SC[full_rows["index"].to_numpy()]
        
        print(f"Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}")
        print(f"Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0) > 0).sum())}")
        
        
        # ── Perch backbone ────────────────────────────────────────────────────────────
        # Prefer no-DFT variant, fallback to standard
        ONNX_PERCH_PATH = next(INPUT_ROOT.glob("**/perch_v2_no_dft*.onnx"),
                           next(INPUT_ROOT.glob("**/perch_v2*.onnx"), Path("")))
        USE_ONNX = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()
        
        if MODEL_DIR.exists():
            birdclassifier = tf.saved_model.load(str(MODEL_DIR))
            infer_fn       = birdclassifier.signatures["serving_default"]
        else:
            birdclassifier = None
            infer_fn       = None
            print("TF Perch SavedModel not attached; using ONNX Perch path when available.")
        
        if not USE_ONNX and infer_fn is None:
            raise FileNotFoundError("No usable Perch backend found: attach ONNX Perch or google/bird-vocalization-classifier.")
        
        if USE_ONNX:
            _so = ort.SessionOptions()
            _so.intra_op_num_threads = 4
            ONNX_SESSION    = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                                    providers=["CPUExecutionProvider"])
            ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
            ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
            print(f"Using ONNX Perch: {ONNX_PERCH_PATH.name}")
        else:
            print("Using TF SavedModel Perch")
        
        def _find_perch_labels_path():
            preferred = MODEL_DIR / "assets" / "labels.csv"
            if preferred.exists():
                return preferred
            for p in sorted(Path("/kaggle/input").rglob("labels.csv")):
                try:
                    cols = set(pd.read_csv(p, nrows=0).columns)
                except Exception:
                    continue
                if {"inat2024_fsd50k", "scientific_name"} & cols:
                    print("Using Perch labels:", p)
                    return p
            raise FileNotFoundError("Perch labels.csv not found. Attach Perch ONNX labels or google/bird-vocalization-classifier.")
        
        def _load_perch_labels(path):
            df = pd.read_csv(path).reset_index().rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
            if "scientific_name" not in df.columns:
                for c in ["label", "labels", "name"]:
                    if c in df.columns:
                        df = df.rename(columns={c: "scientific_name"})
                        break
            assert "scientific_name" in df.columns, f"No scientific_name column in {path}"
            return df
        
        bc_labels = _load_perch_labels(_find_perch_labels_path())
        NO_LABEL = len(bc_labels)
        
        mapping = (taxonomy
                   .merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}),
                          on="scientific_name", how="left"))
        mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
        lbl2bc = mapping.set_index("primary_label")["bc_index"]
        
        BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
        MAPPED_MASK   = BC_INDICES != NO_LABEL
        MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
        MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)
        
        print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")
        
        import re as _re
        UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)
        CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
        TEXTURE_TAXA   = {"Amphibia", "Insecta"}
        
        proxy_map = {}
        unmapped_df = (taxonomy[taxonomy["primary_label"]
                       .isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].copy())
        
        for _, row in unmapped_df.iterrows():
            target = row["primary_label"]
            sci    = str(row["scientific_name"])
            genus  = sci.split()[0]
            hits = bc_labels[
                bc_labels["scientific_name"]
                .astype(str)
                .str.match(rf"^{_re.escape(genus)}\s", na=False)
            ]
            if len(hits) > 0:
                proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()
        
        PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
        proxy_map  = {
            idx: bc_idxs
            for idx, bc_idxs in proxy_map.items()
            if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
        }
        
        print(f"Unmapped: {len(UNMAPPED_POS)} | Proxy: {len(proxy_map)} | No signal: {len(UNMAPPED_POS)-len(proxy_map)}")
        
        
        # ── Per-taxon temperatures ────────────────────────────────────────────────────
        temperatures = np.ones(N_CLASSES, dtype=np.float32)
        for ci, label in enumerate(PRIMARY_LABELS):
            cls = CLASS_NAME_MAP.get(label, "Aves")
            temperatures[ci] = 0.95 if cls in TEXTURE_TAXA else 1.10
        
        
        # ── Perch inference engine ────────────────────────────────────────────────────
        import concurrent.futures
        
        def read_60s(path):
            y, sr = sf.read(path, dtype="float32", always_2d=False)
            if y.ndim == 2: y = y.mean(axis=1)
            if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
            else:                      y = y[:FILE_SAMPLES]
            return y
        
        def run_perch(paths, batch_files=16, verbose=True):
            paths  = [Path(p) for p in paths]
            n_rows = len(paths) * N_WINDOWS
            row_ids   = np.empty(n_rows, dtype=object)
            filenames = np.empty(n_rows, dtype=object)
            sites     = np.empty(n_rows, dtype=object)
            hours     = np.zeros(n_rows, dtype=np.int16)
            scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
            embs      = np.zeros((n_rows, 1536),      dtype=np.float32)
            wr  = 0
            itr = tqdm(range(0, len(paths), batch_files), desc="Perch") if verbose else range(0, len(paths), batch_files)
        
            with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
                next_paths   = paths[0:batch_files]
                future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
                for start in itr:
                    batch_paths  = next_paths
                    batch_n      = len(batch_paths)
                    batch_audio  = [f.result() for f in future_audio]
                    next_start = start + batch_files
                    if next_start < len(paths):
                        next_paths   = paths[next_start:next_start + batch_files]
                        future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
                    x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
                    br = wr
                    for bi, path in enumerate(batch_paths):
                        y    = batch_audio[bi]
                        meta = parse_fname(path.name)
                        stem = path.stem
                        x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                        row_ids  [wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                        filenames[wr:wr + N_WINDOWS] = path.name
                        sites    [wr:wr + N_WINDOWS] = meta["site"]
                        hours    [wr:wr + N_WINDOWS] = meta["hour_utc"]
                        wr += N_WINDOWS
                    if USE_ONNX:
                        outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                        logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
                        emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
                    else:
                        out    = infer_fn(inputs=tf.convert_to_tensor(x))
                        logits = out["label"].numpy().astype(np.float32)
                        emb    = out["embedding"].numpy().astype(np.float32)
                    scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
                    embs  [br:wr]             = emb
                    for pos_idx, bc_idxs in proxy_map.items():
                        bc_arr = np.array(bc_idxs, dtype=np.int32)
                        scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)
                    del x, logits, emb, batch_audio
                    gc.collect()
            meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames,
                                     "site": sites, "hour_utc": hours})
            return meta_df, scores, embs
        
        print("Perch inference engine defined")
        
        
        # ── Cache ─────────────────────────────────────────────────────────────────────
        print(f"USE_ONNX = {USE_ONNX}")
        
        EXTERNAL_CACHE_DIRS = [
            Path("/kaggle/input/notebooks/vyankteshdwivedi/notebook1b25083f0d"),
            Path("/kaggle/input/datasets/jaejohn/perch-meta"),
        ]
        CACHE_NAME_PAIRS = [
            ("perch_meta.parquet", "perch_arrays.npz"),
            ("full_perch_meta.parquet", "full_perch_arrays.npz"),
        ]
        CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
        CACHE_NPZ_LOCAL  = WORK_DIR / "perch_arrays.npz"
        
        def _find_external_cache():
            roots = [d for d in EXTERNAL_CACHE_DIRS if d.exists()]
            roots.append(Path("/kaggle/input"))
            seen = set()
            for root in roots:
                if not root.exists():
                    continue
                key = str(root)
                if key in seen:
                    continue
                seen.add(key)
                for meta_name, npz_name in CACHE_NAME_PAIRS:
                    meta = root / meta_name
                    npz = root / npz_name
                    if meta.exists() and npz.exists():
                        print("Using Perch cache:", meta, npz)
                        return meta, npz
                for meta_name, npz_name in CACHE_NAME_PAIRS:
                    for meta in sorted(root.rglob(meta_name)):
                        npz = meta.parent / npz_name
                        if npz.exists():
                            print("Using Perch cache:", meta, npz)
                            return meta, npz
            return None, None
        
        SCORE_KEYS = ["scores", "sc", "logits", "perch_scores", "preds", "arr_0"]
        EMB_KEYS   = ["embs", "emb", "embeddings", "features", "perch_embs", "arr_1"]
        
        def _pick_array(arr, candidates, shape_hint_cols):
            for k in candidates:
                if k in arr.files:
                    v = arr[k]
                    if getattr(v, "ndim", 0) == 2 and v.shape[1] == shape_hint_cols:
                        return v, k
                    print(f"Skipping cache key {k!r}: shape={getattr(v, 'shape', None)}, expected second dim={shape_hint_cols}")
            for k in arr.files:
                v = arr[k]
                if getattr(v, "ndim", 0) == 2 and v.shape[1] == shape_hint_cols:
                    return v, k
            raise KeyError(f"None of {candidates} found in npz. Available keys: {arr.files}")
        
        def _build_cache():
            print(f"Building Perch cache from {len(full_files)} training files…")
            train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
            train_paths = [p for p in train_paths if p.exists()]
            t0 = time.time()
            meta_built, sc_built, emb_built = run_perch(train_paths, batch_files=CFG["batch_files"], verbose=True)
            print(f"  Perch pass done in {time.time()-t0:.1f}s  scores={sc_built.shape} embs={emb_built.shape}")
            meta_built.to_parquet(CACHE_META_LOCAL)
            np.savez(CACHE_NPZ_LOCAL, scores=sc_built.astype(np.float32),
                     embs=emb_built.astype(np.float32), primary_labels=np.array(PRIMARY_LABELS))
            print(f"  Cache saved to {WORK_DIR}")
            return CACHE_META_LOCAL, CACHE_NPZ_LOCAL
        
        ext_meta, ext_npz = _find_external_cache()
        if ext_meta is not None:
            CACHE_META, CACHE_NPZ = ext_meta, ext_npz
            print(f"Using external cache: {CACHE_META.parent}")
        elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
            CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
            print(f"Using local cache: {WORK_DIR}")
        else:
            print("No cache found — building from scratch")
            CACHE_META, CACHE_NPZ = _build_cache()
        
        meta_tr = pd.read_parquet(CACHE_META)
        _arr    = np.load(CACHE_NPZ)
        sc_tr_raw,  sk = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
        emb_tr_raw, ek = _pick_array(_arr, EMB_KEYS,   1536)
        sc_tr  = sc_tr_raw.astype(np.float32)
        emb_tr = emb_tr_raw.astype(np.float32)
        
        if "primary_labels" in _arr.files:
            if _arr["primary_labels"].tolist() != PRIMARY_LABELS:
                print("  WARNING: cached primary_labels differ — scores columns may not align!")
            else:
                print("  primary_labels schema OK")
        
        if "row_id" not in meta_tr.columns:
            if "end_sec" in meta_tr.columns:
                end_sec = meta_tr["end_sec"].astype(int)
            elif "window_idx" in meta_tr.columns:
                end_sec = (meta_tr["window_idx"].astype(int) + 1) * WINDOW_SEC
            else:
                assert len(meta_tr) % N_WINDOWS == 0, "cannot infer end_sec from cache row count"
                end_sec = np.tile(np.arange(WINDOW_SEC, WINDOW_SEC * N_WINDOWS + 1, WINDOW_SEC), len(meta_tr) // N_WINDOWS)
            meta_tr["row_id"] = (meta_tr["filename"].str.replace(".ogg", "", regex=False)
                                 + "_" + end_sec.astype(str))
        if "end_sec" not in meta_tr.columns:
            if "window_idx" in meta_tr.columns:
                meta_tr["end_sec"] = (meta_tr["window_idx"].astype(int) + 1) * WINDOW_SEC
            else:
                meta_tr["end_sec"] = meta_tr["row_id"].str.rsplit("_", n=1).str[-1].astype(int)
        assert len(meta_tr) == sc_tr.shape[0] == emb_tr.shape[0], (
            f"cache row count mismatch: meta={len(meta_tr)}, sc={sc_tr.shape}, emb={emb_tr.shape}"
        )
        assert meta_tr["row_id"].is_unique, "duplicate row_id in Perch cache metadata"
        meta_tr = meta_tr.copy()
        meta_tr["_cache_pos"] = np.arange(len(meta_tr))
        order = meta_tr.sort_values(["filename", "end_sec"])["_cache_pos"].to_numpy()
        meta_tr = meta_tr.iloc[order].drop(columns=["_cache_pos"]).reset_index(drop=True)
        sc_tr = sc_tr[order]
        emb_tr = emb_tr[order]
        
        row_id_to_index = full_rows.set_index("row_id")["index"]
        missing_rows = set(meta_tr["row_id"]) - set(row_id_to_index.index)
        if missing_rows:
            raise RuntimeError(f"Cache has {len(missing_rows)} row_ids not in labeled set.")
        
        Y_FULL_aligned = Y_SC[row_id_to_index.loc[meta_tr["row_id"]].to_numpy()]
        print(f"sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  Y_FULL_aligned: {Y_FULL_aligned.shape}")
        
        
        # ── Post-processing helpers ───────────────────────────────────────────────────
        def macro_auc(y_true, y_score):
            keep = y_true.sum(axis=0) > 0
            return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")
        
        def smooth_predictions(probs, n_windows=12, alpha=0.3):
            N, C = probs.shape
            assert N % n_windows == 0
            view = probs.reshape(-1, n_windows, C).copy()
            prev_w = np.concatenate([view[:, :1, :],  view[:, :-1, :]], axis=1)
            next_w = np.concatenate([view[:, 1:,  :], view[:, -1:, :]], axis=1)
            return ((1 - alpha) * view + 0.5 * alpha * (prev_w + next_w)).reshape(N, C)
        
        
        # ── UPGRADED prior tables — joint site-hour bucket ────────────────────────────
        def build_prior_tables(sc_df, Y_labels):
            sc_df = sc_df.reset_index(drop=True)
            global_p = Y_labels.mean(axis=0).astype(np.float32)
        
            site_keys = sorted(sc_df["site"].dropna().astype(str).unique())
            site_to_i = {k: i for i, k in enumerate(site_keys)}
            site_p = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
            site_n = np.zeros(len(site_keys), dtype=np.float32)
            for s in site_keys:
                i = site_to_i[s]
                mask = sc_df["site"].astype(str).values == s
                site_n[i] = mask.sum()
                site_p[i] = Y_labels[mask].mean(axis=0)
        
            hour_keys = sorted(sc_df["hour_utc"].dropna().astype(int).unique())
            hour_to_i = {h: i for i, h in enumerate(hour_keys)}
            hour_p = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
            hour_n = np.zeros(len(hour_keys), dtype=np.float32)
            for h in hour_keys:
                i = hour_to_i[h]
                mask = sc_df["hour_utc"].astype(int).values == h
                hour_n[i] = mask.sum()
                hour_p[i] = Y_labels[mask].mean(axis=0)
        
            # Joint site-hour bucket (new — tighter shrinkage factor 4)
            sh_keys = sorted(
                {
                    (str(s), int(h))
                    for s, h in zip(sc_df["site"].dropna(), sc_df["hour_utc"].dropna())
                    if not pd.isna(s) and not pd.isna(h)
                }
            )
            sh_to_i = {k: i for i, k in enumerate(sh_keys)}
            sh_p = np.zeros((len(sh_keys), Y_labels.shape[1]), dtype=np.float32)
            sh_n = np.zeros(len(sh_keys), dtype=np.float32)
            for (s, h) in sh_keys:
                i = sh_to_i[(s, h)]
                mask = (sc_df["site"].astype(str).values == s) & (
                    sc_df["hour_utc"].astype(int).values == h
                )
                sh_n[i] = mask.sum()
                sh_p[i] = Y_labels[mask].mean(axis=0)
        
            # ── Tweak D: Circular Gaussian smoothing on hour priors ──────────────────
            # Motivation: Raw per-hour prior tables are computed from hard count buckets
            # (e.g. 06:00 UTC and 07:00 UTC are treated as independent). Many species
            # have a smooth dusk/dawn peak that leaks across adjacent hours. Applying a
            # circular Gaussian kernel (wrap-around at hour 23→0) with sigma=1.5 hrs
            # produces a more realistic, continuous prior distribution and reduces
            # over-fitting to hours that happen to have more training samples.
            # This is done on the N_hours x N_classes hour_p matrix (axis=0 = hours).
            if len(hour_keys) >= 3:  # only smooth if we have enough distinct hours
                # Build a full 24-hour grid and embed hour_p into it for wrap-around
                _full_hour_p = np.zeros((24, hour_p.shape[1]), dtype=np.float32)
                for _h, _i in hour_to_i.items():
                    _full_hour_p[int(_h)] = hour_p[_i]
                # Wrap-aware: tile 3x, smooth the middle block, then extract
                _tiled = np.tile(_full_hour_p, (3, 1))  # shape: (72, N_CLASSES)
                _tiled_smooth = gaussian_filter1d(_tiled, sigma=1.5, axis=0, mode='wrap')
                _full_smooth = _tiled_smooth[24:48]  # extract the middle 24 hours
                # Write back only the hours that exist in the training set
                for _h, _i in hour_to_i.items():
                    hour_p[_i] = _full_smooth[int(_h)]
                hour_p = np.clip(hour_p, 0.0, 1.0)
        
            return {
                "global_p": global_p,
                "site_to_i": site_to_i,
                "site_p": site_p,
                "site_n": site_n,
                "hour_to_i": hour_to_i,
                "hour_p": hour_p,
                "hour_n": hour_n,
                "sh_to_i": sh_to_i,
                "sh_p": sh_p,
                "sh_n": sh_n,
            }
        
        
        def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
            eps = 1e-4; n = len(scores); out = scores.copy()
            p = np.tile(tables["global_p"], (n, 1))
            for i, h in enumerate(hours):
                h = int(h)
                if h in tables["hour_to_i"]:
                    j = tables["hour_to_i"][h]; nh = tables["hour_n"][j]; w = nh / (nh + 8.0)
                    p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]
            for i, s in enumerate(sites):
                s = str(s)
                if s in tables["site_to_i"]:
                    j = tables["site_to_i"][s]; ns = tables["site_n"][j]; w = ns / (ns + 8.0)
                    p[i] = w * tables["site_p"][j] + (1 - w) * p[i]
            if "sh_to_i" in tables:
                for i, (s, h) in enumerate(zip(sites, hours)):
                    key = (str(s), int(h))
                    if key in tables["sh_to_i"]:
                        j = tables["sh_to_i"][key]; nsh = tables["sh_n"][j]; w = nsh / (nsh + 4.0)
                        p[i] = w * tables["sh_p"][j] + (1 - w) * p[i]
            p = np.clip(p, eps, 1 - eps)
            out += lambda_prior * (np.log(p) - np.log1p(-p))
            return out.astype(np.float32)
        
        def file_confidence_scale(probs, n_windows=12, top_k=2, power=POWEROPT_FILE_CONFIDENCE_POWER):
            N, C = probs.shape
            view      = probs.reshape(-1, n_windows, C)
            sorted_v  = np.sort(view, axis=1)
            top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)
            return (view * np.power(top_k_mean, power)).reshape(N, C)
        
        def rank_aware_scaling(probs, n_windows=12, power=0.5):
            N, C = probs.shape
            view     = probs.reshape(-1, n_windows, C)
            file_max = view.max(axis=1, keepdims=True)
            return (view * np.power(file_max, power)).reshape(N, C)
        
        def adaptive_delta_smooth(probs, n_windows=12, base_alpha=POWEROPT_DELTA_SMOOTH_ALPHA):
            N, C = probs.shape
            result = probs.copy(); view = probs.reshape(-1, n_windows, C); out = result.reshape(-1, n_windows, C)
            for t in range(n_windows):
                conf = view[:, t, :].max(axis=-1, keepdims=True); alpha = base_alpha * (1.0 - conf)
                if t == 0:           neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
                elif t == n_windows-1: neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
                else:                  neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0
                out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg
            return result
        
        
        # ── MLP probes ────────────────────────────────────────────────────────────────
        from sklearn.decomposition import PCA
        from sklearn.preprocessing import StandardScaler
        from sklearn.neural_network import MLPClassifier
        from sklearn.isotonic import IsotonicRegression
        import torch.nn as nn
        import torch.nn.functional as F
        
        def build_class_freq_weights(Y, cap=10.0):
            pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
            freq = pos_count / Y.shape[0]
            weights = np.clip(1.0 / (freq ** 0.5), 1.0, cap)
            return (weights / weights.mean()).astype(np.float32)
        
        def build_sequential_features(scores_col, n_windows=12):
            x     = scores_col.reshape(-1, n_windows)
            prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
            next_ = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
            mean  = np.repeat(x.mean(axis=1), n_windows)
            max_  = np.repeat(x.max(axis=1),  n_windows)
            std   = np.repeat(x.std(axis=1),  n_windows)
            return prev.reshape(-1), next_.reshape(-1), mean, max_, std
        
        def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
            scaler = StandardScaler()
            emb_s = scaler.fit_transform(emb)
        
            pca = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
            Z = pca.fit_transform(emb_s).astype(np.float32)
        
            print(
                f"Embedding: {emb.shape} → PCA: {Z.shape}  "
                f"(variance retained: {pca.explained_variance_ratio_.sum():.2%})"
            )
        
            class_weights = build_class_freq_weights(Y, cap=10.0)
            probe_models = {}
            active = np.where(Y.sum(axis=0) >= min_pos)[0]
            MAX_ROWS = 3000
        
            for ci in tqdm(active, desc="MLP probes"):
                y = Y[:, ci]
                if y.sum() == 0 or y.sum() == len(y):
                    continue
        
                prev, next_, mean, max_, std = build_sequential_features(scores_raw[:, ci])
                X = np.hstack(
                    [
                        Z,
                        scores_raw[:, ci : ci + 1],
                        prev[:, None],
                        next_[:, None],
                        mean[:, None],
                        max_[:, None],
                        std[:, None],
                    ]
                )
        
                n_pos = int(y.sum())
                n_neg = len(y) - n_pos
                pos_idx = np.where(y == 1)[0]
                w = float(class_weights[ci])
                repeat = max(1, min(int(round(w * n_neg / max(n_pos, 1))), 8))
                if n_pos * repeat + len(y) > MAX_ROWS:
                    repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))
        
                X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
                y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])
        
                # ── Tweak E: Wider MLP for frequent classes ──────────────────────────
                # Motivation: All probes previously used (128, 64) regardless of how
                # many positive examples a class has. For classes with ≥50 positives
                # the decision boundary is complex enough that a wider (256, 128) net
                # improves fit without overfitting, because enough data exists to
                # regularise it. Rare classes (<50 pos) keep (128, 64) to avoid
                # overfit. This mirrors the CFG['mlp_params'] hidden_layer_sizes
                # (256, 128) that was already defined but never used here.
                _hidden = (256, 128) if n_pos >= 50 else (128, 64)  # Tweak E: reuses n_pos already computed above
                clf = MLPClassifier(
                    hidden_layer_sizes=_hidden,
                    activation="relu",
                    max_iter=300,
                    early_stopping=True,
                    validation_fraction=0.15,
                    n_iter_no_change=15,
                    random_state=42,
                    learning_rate_init=5e-4,
                    alpha=0.005,
                )
                clf.fit(X_bal, y_bal)
                probe_models[ci] = clf
        
            print(f"Trained {len(probe_models)} MLP probes")
            return probe_models, scaler, pca, alpha_blend
        
        
        class VectorizedMLPProbes(nn.Module):
            """Vectorized forward pass for a homogeneous group of MLP probes.
        
            All probes passed to __init__ MUST share the same layer shapes.
            Tweak E introduced two architectures ((128,64) for rare classes and
            (256,128) for frequent ones), so the caller must split probes by
            architecture before constructing this module — see
            apply_mlp_probes_vectorized for how this is handled.
            """
        
            def __init__(self, probe_models):
                super().__init__()
                self.valid_classes = sorted(probe_models.keys())
                V = len(self.valid_classes)
        
                if V == 0:
                    self.weights = nn.ParameterList()
                    self.biases = nn.ParameterList()
                    self.n_layers = 0
                    return
        
                sample = probe_models[self.valid_classes[0]]
                self.n_layers = len(sample.coefs_)
                self.weights = nn.ParameterList()
                self.biases = nn.ParameterList()
        
                for li in range(self.n_layers):
                    W = np.stack([probe_models[c].coefs_[li] for c in self.valid_classes], axis=0)
                    b = np.stack(
                        [probe_models[c].intercepts_[li] for c in self.valid_classes], axis=0
                    )
                    self.weights.append(
                        nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False)
                    )
                    self.biases.append(
                        nn.Parameter(torch.tensor(b, dtype=torch.float32), requires_grad=False)
                    )
        
            def forward(self, x):
                h = x
                for i in range(self.n_layers):
                    h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
                    if i < self.n_layers - 1:
                        h = torch.relu(h)
                return h.squeeze(-1)
        
        
        def _run_probe_group(group_models, valid_classes_group, scores_test, Z_test, N):
            """Run VectorizedMLPProbes for one homogeneous architecture group.
        
            All probes in group_models must share the same hidden-layer shapes.
            Returns preds array of shape (len(valid_classes_group), N).
            """
            Vg = len(valid_classes_group)
            raw_g = scores_test[:, valid_classes_group].T          # (Vg, N)
            n_files = N // N_WINDOWS
            raw_view_g = raw_g.reshape(Vg, n_files, N_WINDOWS)
        
            prev_g = np.concatenate([raw_view_g[:, :, :1], raw_view_g[:, :, :-1]], axis=2).reshape(Vg, N)
            nxt_g  = np.concatenate([raw_view_g[:, :, 1:], raw_view_g[:, :, -1:]], axis=2).reshape(Vg, N)
            mean_g = np.repeat(raw_view_g.mean(axis=2), N_WINDOWS, axis=1)
            mx_g   = np.repeat(raw_view_g.max(axis=2),  N_WINDOWS, axis=1)
            std_g  = np.repeat(raw_view_g.std(axis=2),  N_WINDOWS, axis=1)
        
            scalar_g  = np.stack([raw_g, prev_g, nxt_g, mean_g, mx_g, std_g], axis=-1).astype(np.float32)
            Z_exp_g   = np.broadcast_to(Z_test, (Vg, N, Z_test.shape[1]))
            X_g       = np.concatenate([Z_exp_g.astype(np.float32), scalar_g], axis=-1)
        
            vec_probe = VectorizedMLPProbes(group_models).eval()
            with torch.no_grad():
                preds_g = vec_probe(torch.tensor(X_g)).numpy()   # (Vg, N)
            return preds_g
        
        
        def apply_mlp_probes_vectorized(
            emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.4
        ):
            """Apply MLP probes to test embeddings and scores.
        
            Tweak E fix: probes are partitioned by their hidden-layer architecture
            (tuple of layer sizes) before vectorization. Each architecture group is
            stacked separately through VectorizedMLPProbes, then results are merged
            back into the output array. This avoids the shape-mismatch error that
            arises when mixing (128,64) and (256,128) probes in the same np.stack.
            """
            if len(probe_models) == 0:
                return scores_test.copy()
        
            Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
            N = len(scores_test)
            result = scores_test.copy()
        
            # ── Partition probes by architecture (layer output sizes) ─────────────────
            def _arch_key(clf):
                """Canonical shape key: tuple of each layer's output size."""
                return tuple(w.shape[1] for w in clf.coefs_)
        
            from collections import defaultdict
            groups = defaultdict(dict)       # arch_key → {class_idx: clf}
            for ci, clf in probe_models.items():
                groups[_arch_key(clf)][ci] = clf
        
            # ── Run each architecture group separately, then blend into result ────────
            for arch, group_models in groups.items():
                valid_classes_group = sorted(group_models.keys())
                preds_g = _run_probe_group(group_models, valid_classes_group, scores_test, Z_test, N)
                # preds_g shape: (Vg, N) — transpose to (N, Vg) for column assignment
                result[:, valid_classes_group] = (
                    (1.0 - alpha_blend) * scores_test[:, valid_classes_group]
                    + alpha_blend * preds_g.T
                )
        
            return result
        
        
        def calibrate_and_optimize_thresholds(oof_probs, Y_FULL, threshold_grid=None, n_windows=12):
            if threshold_grid is None: threshold_grid = [0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]
            n_samples, n_cls = oof_probs.shape; thresholds = np.full(n_cls, 0.5, dtype=np.float32)
            n_files = n_samples // n_windows
            file_oof = oof_probs.reshape(n_files, n_windows, n_cls).max(axis=1)
            file_y   = Y_FULL.reshape(n_files, n_windows, n_cls).max(axis=1)
            n_calibrated = 0
            for c in range(n_cls):
                y_true = file_y[:, c]; y_prob = file_oof[:, c]
                if y_true.sum() < 3: continue
                try:
                    ir = IsotonicRegression(out_of_bounds="clip"); ir.fit(y_prob, y_true); y_cal = ir.transform(y_prob)
                except: y_cal = y_prob
                best_f1, best_t = 0.0, 0.5
                for t in threshold_grid:
                    pred = (y_cal >= t).astype(int)
                    tp=((pred==1)&(y_true==1)).sum(); fp=((pred==1)&(y_true==0)).sum(); fn=((pred==0)&(y_true==1)).sum()
                    prec=tp/(tp+fp+1e-8); rec=tp/(tp+fn+1e-8); f1=2*prec*rec/(prec+rec+1e-8)
                    if f1 > best_f1: best_f1,best_t = f1,t
                thresholds[c] = best_t; n_calibrated += 1
            print(f"Calibrated {n_calibrated} classes | Mean threshold: {thresholds.mean():.3f} | Range: [{thresholds.min():.2f}, {thresholds.max():.2f}]")
            return thresholds
        
        def apply_per_class_thresholds(scores, thresholds):
            C = scores.shape[1]; scaled = np.copy(scores)
            for c in range(C):
                t = thresholds[c]; above = scores[:, c] > t
                scaled[above, c]  = 0.5 + 0.5 * (scores[above, c]  - t) / (1 - t + 1e-8)
                scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
            return np.clip(scaled, 0.0, 1.0)
        
        
        # ── SSM Architecture ─────────────────────────────────────────────────────────
        class SelectiveSSM(nn.Module):
            def __init__(self, d_model, d_state=16, d_conv=4):
                super().__init__(); self.d_model=d_model; self.d_state=d_state
                self.in_proj=nn.Linear(d_model,2*d_model,bias=False)
                self.conv1d=nn.Conv1d(d_model,d_model,d_conv,padding=d_conv-1,groups=d_model)
                self.dt_proj=nn.Linear(d_model,d_model,bias=True)
                A=torch.arange(1,d_state+1,dtype=torch.float32).unsqueeze(0).expand(d_model,-1)
                self.A_log=nn.Parameter(torch.log(A)); self.D=nn.Parameter(torch.ones(d_model))
                self.B_proj=nn.Linear(d_model,d_state,bias=False); self.C_proj=nn.Linear(d_model,d_state,bias=False)
                self.out_proj=nn.Linear(d_model,d_model,bias=False)
            def forward(self,x):
                B_sz,T,D=x.shape; xz=self.in_proj(x); x_ssm,z=xz.chunk(2,dim=-1)
                x_conv=F.silu(self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2))
                dt=F.softplus(self.dt_proj(x_conv)); A=-torch.exp(self.A_log)
                B=self.B_proj(x_conv); C=self.C_proj(x_conv)
                h=torch.zeros(B_sz,D,self.d_state,device=x.device); ys=[]
                for t in range(T):
                    dA=torch.exp(A[None]*dt[:,t,:,None]); dB=dt[:,t,:,None]*B[:,t,None,:]
                    h=h*dA+x[:,t,:,None]*dB; ys.append((h*C[:,t,None,:]).sum(-1))
                return torch.stack(ys,dim=1)+x*self.D[None,None,:]
        
        class LightProtoSSM(nn.Module):
            def __init__(self,d_input=1536,d_model=128,d_state=16,n_classes=234,n_windows=12,
                         dropout=0.15,n_sites=20,meta_dim=16,use_cross_attn=True,cross_attn_heads=2):
                super().__init__(); self.n_classes=n_classes; self.n_windows=n_windows; self.use_cross_attn=use_cross_attn
                self.input_proj=nn.Sequential(nn.Linear(d_input,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
                self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
                self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
                self.meta_proj=nn.Linear(2*meta_dim,d_model)
                self.ssm_fwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
                self.ssm_bwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
                self.ssm_merge=nn.ModuleList([nn.Linear(2*d_model,d_model) for _ in range(2)])
                self.ssm_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
                self.drop=nn.Dropout(dropout)
                if use_cross_attn:
                    self.cross_attn=nn.ModuleList([nn.MultiheadAttention(d_model,cross_attn_heads,dropout=dropout,batch_first=True) for _ in range(2)])
                    self.cross_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
                self.prototypes=nn.Parameter(torch.randn(n_classes,d_model)*0.02)
                self.proto_temp=nn.Parameter(torch.tensor(5.0))
                self.class_bias=nn.Parameter(torch.zeros(n_classes))
                self.fusion_alpha=nn.Parameter(torch.zeros(n_classes))
            def init_prototypes(self,emb_tensor,labels_tensor):
                with torch.no_grad():
                    h=self.input_proj(emb_tensor)
                    for c in range(self.n_classes):
                        mask=labels_tensor[:,c]>0.5
                        if mask.sum()>0: self.prototypes.data[c]=F.normalize(h[mask].mean(0),dim=0)
            def forward(self,emb,perch_logits=None,site_ids=None,hours=None):
                B,T,_=emb.shape; h=self.input_proj(emb)+self.pos_enc[:,:T,:]
                if site_ids is not None and hours is not None:
                    meta=self.meta_proj(torch.cat([self.site_emb(site_ids),self.hour_emb(hours)],dim=-1))
                    h=h+meta[:,None,:]
                for i,(fwd,bwd,merge,norm) in enumerate(zip(self.ssm_fwd,self.ssm_bwd,self.ssm_merge,self.ssm_norm)):
                    res=h; hf=fwd(h); hb=bwd(h.flip(1)).flip(1)
                    h=self.drop(merge(torch.cat([hf,hb],dim=-1))); h=norm(h+res)
                    if self.use_cross_attn:
                        attn_out,_=self.cross_attn[i](h,h,h); h=self.cross_norm[i](h+attn_out)
                h_n=F.normalize(h,dim=-1); p_n=F.normalize(self.prototypes,dim=-1)
                sim=torch.matmul(h_n,p_n.T)*F.softplus(self.proto_temp)+self.class_bias[None,None,:]
                if perch_logits is not None:
                    alpha=torch.sigmoid(self.fusion_alpha)[None,None,:]
                    out=alpha*sim+(1-alpha)*perch_logits
                else: out=sim
                return out
        
        class ResidualSSM(nn.Module):
            def __init__(self,d_input=1536,d_scores=234,d_model=64,d_state=8,n_classes=234,
                         n_windows=12,dropout=0.1,n_sites=20,meta_dim=8):
                super().__init__(); self.n_classes=n_classes
                self.input_proj=nn.Sequential(nn.Linear(d_input+d_scores,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
                self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
                self.meta_proj=nn.Linear(2*meta_dim,d_model)
                self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
                self.ssm_fwd=SelectiveSSM(d_model,d_state); self.ssm_bwd=SelectiveSSM(d_model,d_state)
                self.ssm_merge=nn.Linear(2*d_model,d_model); self.ssm_norm=nn.LayerNorm(d_model); self.ssm_drop=nn.Dropout(dropout)
                self.output_head=nn.Linear(d_model,n_classes)
                nn.init.zeros_(self.output_head.weight); nn.init.zeros_(self.output_head.bias)
            def forward(self,emb,first_pass,site_ids=None,hours=None):
                B,T,_=emb.shape; x=torch.cat([emb,first_pass],dim=-1)
                h=self.input_proj(x)+self.pos_enc[:,:T,:]
                if site_ids is not None and hours is not None:
                    meta=self.meta_proj(torch.cat([self.site_emb(site_ids.clamp(0,self.site_emb.num_embeddings-1)),
                                                    self.hour_emb(hours.clamp(0,23))],dim=-1))
                    h=h+meta.unsqueeze(1)
                res=h; hf=self.ssm_fwd(h); hb=self.ssm_bwd(h.flip(1)).flip(1)
                h=self.ssm_drop(self.ssm_merge(torch.cat([hf,hb],dim=-1))); h=self.ssm_norm(h+res)
                return self.output_head(h)
        
        def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full, n_epochs=40, patience=8, lr=1e-3, n_sites=20, verbose=False):
            n_files=len(emb_full)//N_WINDOWS; emb_f=emb_full.reshape(n_files,N_WINDOWS,-1)
            log_f=scores_full.reshape(n_files,N_WINDOWS,-1); lab_f=Y_full.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
            fnames=meta_full["filename"].unique(); sites_u=sorted(meta_full["site"].unique())
            site2i={s:i+1 for i,s in enumerate(sites_u)}
            site_ids=np.array([min(site2i.get(meta_full.loc[meta_full["filename"]==fn,"site"].iloc[0],0),n_sites-1) for fn in fnames],dtype=np.int64)
            hour_ids=np.array([int(meta_full.loc[meta_full["filename"]==fn,"hour_utc"].iloc[0])%24 for fn in fnames],dtype=np.int64)
            model=LightProtoSSM(n_classes=N_CLASSES,n_sites=n_sites,use_cross_attn=True,cross_attn_heads=2)
            model.init_prototypes(torch.tensor(emb_full,dtype=torch.float32),torch.tensor(Y_full,dtype=torch.float32))
            emb_t=torch.tensor(emb_f,dtype=torch.float32); log_t=torch.tensor(log_f,dtype=torch.float32)
            lab_t=torch.tensor(lab_f,dtype=torch.float32); site_t=torch.tensor(site_ids,dtype=torch.long)
            hour_t=torch.tensor(hour_ids,dtype=torch.long)
            pos_cnt=lab_t.sum(dim=(0,1)); total=lab_t.shape[0]*lab_t.shape[1]
            pos_weight=((total-pos_cnt)/(pos_cnt+1)).clamp(max=25.0)
            opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
            sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,epochs=n_epochs,steps_per_epoch=1,pct_start=0.1,anneal_strategy="cos")
            best_loss,best_state,wait=float("inf"),None,0
            swa_model=torch.optim.swa_utils.AveragedModel(model); swa_start=int(n_epochs*0.65)
            swa_sched=torch.optim.swa_utils.SWALR(opt,swa_lr=4e-4)
            for ep in range(n_epochs):
                model.train()
                out=model(emb_t,log_t,site_ids=site_t,hours=hour_t)
                loss=F.binary_cross_entropy_with_logits(out,lab_t,pos_weight=pos_weight[None,None,:])+0.15*F.mse_loss(out,log_t)
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
                if ep>=swa_start: swa_model.update_parameters(model); swa_sched.step()
                else: sched.step()
                if loss.item()<best_loss:
                    best_loss=loss.item(); best_state={k:v.clone() for k,v in model.state_dict().items()}; wait=0
                else:
                    wait+=1
                    if wait>=patience: break
            if ep>=swa_start:
                torch.optim.swa_utils.update_bn(emb_t.unsqueeze(0),swa_model); model=swa_model
            else: model.load_state_dict(best_state)
            model.eval(); return model,site2i
        
        def run_tta_proto(proto_model, emb_files, sc_files, site_t, hour_t, shifts=[0, 1, -1, 2, -2]):
            proto_model.eval()
            all_preds = []
        
            emb_t = torch.tensor(emb_files, dtype=torch.float32)
            sc_t = torch.tensor(sc_files, dtype=torch.float32)
        
            for shift in shifts:
                e = torch.roll(emb_t, shift, dims=1) if shift else emb_t
                s = torch.roll(sc_t, shift, dims=1) if shift else sc_t
                with torch.no_grad():
                    out = proto_model(e, s, site_ids=site_t, hours=hour_t).numpy()
                if shift:
                    out = np.roll(out, -shift, axis=1)
                all_preds.append(out)
        
            # ── Tweak F: Temporal flip as extra TTA pass ──────────────────────────────
            # Motivation: The SSM is causal-ish (bidirectional, but trained on a fixed
            # left-to-right sequence). Reversing the time axis (flip dims=1) forces the
            # backward SSM branch to act as the forward one and vice versa, providing
            # a structurally different prediction than any shift-based augmentation.
            # The output is flipped back before averaging, so temporal order is restored.
            # Cost: one extra forward pass (~same as adding a 6th shift).
            with torch.no_grad():
                out_flip = proto_model(
                    emb_t.flip(1), sc_t.flip(1), site_ids=site_t, hours=hour_t
                ).numpy()
            all_preds.append(out_flip[:, ::-1, :].copy())  # flip output back to forward order
        
            return np.mean(all_preds, axis=0)
        
        
        def train_residual_ssm(emb_full, first_pass_flat, Y_full, site_ids, hour_ids,
                                n_epochs=30, patience=8, lr=1e-3, correction_weight=0.30, verbose=False):
            n_files=len(emb_full)//N_WINDOWS; emb_f=emb_full.reshape(n_files,N_WINDOWS,-1)
            fp_f=first_pass_flat.reshape(n_files,N_WINDOWS,-1); lab_f=Y_full.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
            fp_prob=1.0/(1.0+np.exp(-np.clip(fp_f,-30,30))); residuals=lab_f-fp_prob
            n_val=max(1,int(n_files*0.15)); rng=torch.Generator(); rng.manual_seed(42)
            perm=torch.randperm(n_files,generator=rng).numpy(); val_i=perm[:n_val]; train_i=perm[n_val:]
            emb_t=torch.tensor(emb_f,dtype=torch.float32); fp_t=torch.tensor(fp_f,dtype=torch.float32)
            res_t=torch.tensor(residuals,dtype=torch.float32)
            site_t=torch.tensor(site_ids,dtype=torch.long); hour_t=torch.tensor(hour_ids,dtype=torch.long)
            model=ResidualSSM(n_classes=N_CLASSES)
            opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
            sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,epochs=n_epochs,steps_per_epoch=1,pct_start=0.1,anneal_strategy="cos")
            best_loss,best_state,wait=float("inf"),None,0
            for ep in range(n_epochs):
                model.train()
                corr=model(emb_t[train_i],fp_t[train_i],site_ids=site_t[train_i],hours=hour_t[train_i])
                loss=F.mse_loss(corr,res_t[train_i])
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); sched.step()
                model.eval()
                with torch.no_grad():
                    val_corr=model(emb_t[val_i],fp_t[val_i],site_ids=site_t[val_i],hours=hour_t[val_i])
                    val_loss=F.mse_loss(val_corr,res_t[val_i])
                if val_loss.item()<best_loss:
                    best_loss=val_loss.item(); best_state={k:v.clone() for k,v in model.state_dict().items()}; wait=0
                else:
                    wait+=1
                    if wait>=patience: break
            model.load_state_dict(best_state); return model,correction_weight
        
        print("Sequence Models defined")
        
        
        # ── Test inference ────────────────────────────────────────────────────────────
        test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
        IS_DRY_RUN = len(test_paths) == 0
        if IS_DRY_RUN:
            n = CFG["dryrun_n_files"] or 20
            print(f"No hidden test — dry-run on {n} train files")
            test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
        else:
            print(f"Hidden test files: {len(test_paths)}")
        
        meta_te, sc_te, emb_te = run_perch(test_paths, CFG["batch_files"], verbose=CFG["verbose"])
        print(f"Test scores: {sc_te.shape}")
        
        
        # ── Full ProtoSSM pipeline ────────────────────────────────────────────────────
        def sigmoid(x):
            return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))
        
        t0 = time.time()
        proto_model, site2i_tr = train_light_proto_ssm(
            emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
            n_epochs=40, patience=8, lr=1e-3, verbose=False)
        print(f"ProtoSSM training: {time.time()-t0:.1f}s")
        
        n_test_files  = len(sc_te) // N_WINDOWS
        emb_te_f      = emb_te.reshape(n_test_files, N_WINDOWS, -1)
        sc_te_f       = sc_te.reshape(n_test_files, N_WINDOWS, -1)
        
        test_fnames   = meta_te.drop_duplicates("filename")["filename"].tolist()
        n_sites_cap   = 20
        test_site_ids = np.array([min(site2i_tr.get(meta_te.loc[meta_te["filename"]==fn,"site"].iloc[0],0),n_sites_cap-1)
                                   for fn in test_fnames], dtype=np.int64)
        test_hour_ids = np.array([int(meta_te.loc[meta_te["filename"]==fn,"hour_utc"].iloc[0])%24
                                   for fn in test_fnames], dtype=np.int64)
        
        proto_out = run_tta_proto(
            proto_model,
            emb_te_f,
            sc_te_f,
            site_t=torch.tensor(test_site_ids, dtype=torch.long),
            hour_t=torch.tensor(test_hour_ids, dtype=torch.long),
            shifts=[0, 1, -1, 2, -2],
        )
        proto_scores_flat = proto_out.reshape(-1, N_CLASSES).astype(np.float32)
        
        prior_tables   = build_prior_tables(sc, Y_SC)
        sc_te_adjusted = apply_prior(sc_te, sites=meta_te["site"].to_numpy(),
                                      hours=meta_te["hour_utc"].to_numpy(), tables=prior_tables, lambda_prior=POWEROPT_SZ2_TEST_PRIOR_LAMBDA)
        
        probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
            emb=emb_tr, scores_raw=sc_tr, Y=Y_FULL_aligned, min_pos=5, pca_dim=64, alpha_blend=0.4)
        sc_te_adjusted = apply_mlp_probes_vectorized(emb_te, sc_te_adjusted, probe_models, emb_scaler, emb_pca, alpha_blend)
        
        # Mapped classes keep more ProtoSSM weight; unmapped classes trust adjusted SED/MLP/prior more.
        ENSEMBLE_W_PER_CLASS = np.where(MAPPED_MASK, 0.60, 0.35).astype(np.float32)
        first_pass_flat = (
            ENSEMBLE_W_PER_CLASS[None, :] * proto_scores_flat
            + (1.0 - ENSEMBLE_W_PER_CLASS)[None, :] * sc_te_adjusted
        )
        print(
            f"[LB0.948] Per-class first-pass weights: mapped={ENSEMBLE_W_PER_CLASS[MAPPED_MASK].mean():.2f} "
            f"unmapped={ENSEMBLE_W_PER_CLASS[~MAPPED_MASK].mean():.2f}"
        )
        
        n_tr_files    = len(sc_tr) // N_WINDOWS
        emb_tr_f      = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
        sc_tr_f       = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)
        
        tr_fnames     = meta_tr.drop_duplicates("filename")["filename"].tolist()
        tr_site_ids   = np.array([min(site2i_tr.get(meta_tr.loc[meta_tr["filename"]==fn,"site"].iloc[0],0),n_sites_cap-1)
                                   for fn in tr_fnames], dtype=np.int64)
        tr_hour_ids   = np.array([int(meta_tr.loc[meta_tr["filename"]==fn,"hour_utc"].iloc[0])%24
                                   for fn in tr_fnames], dtype=np.int64)
        
        proto_tr_out = run_tta_proto(proto_model, emb_tr_f, sc_tr_f,
            site_t=torch.tensor(tr_site_ids, dtype=torch.long),
            hour_t=torch.tensor(tr_hour_ids, dtype=torch.long),
            shifts=[0, 1, -1, 2, -2])
        proto_tr_flat = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)
        
        sc_tr_prior = apply_prior(sc_tr, sites=meta_tr["site"].to_numpy(),
                                   hours=meta_tr["hour_utc"].to_numpy(), tables=prior_tables, lambda_prior=POWEROPT_SZ2_TRAIN_PRIOR_LAMBDA)
        sc_tr_mlp = apply_mlp_probes_vectorized(emb_tr, sc_tr_prior, probe_models, emb_scaler, emb_pca, alpha_blend)
        first_pass_tr = (
            ENSEMBLE_W_PER_CLASS[None, :] * proto_tr_flat
            + (1.0 - ENSEMBLE_W_PER_CLASS)[None, :] * sc_tr_mlp
        )
        
        train_probs_for_calib = sigmoid(first_pass_tr)
        PER_CLASS_THRESHOLDS = calibrate_and_optimize_thresholds(
            oof_probs=train_probs_for_calib,
            Y_FULL=Y_FULL_aligned,
            # Tweak 3: finer threshold grid — better per-class F1 calibration for rare species
            threshold_grid=(
                [round(t, 3) for t in np.arange(0.20, 0.45, 0.025)]
                + [round(t, 3) for t in np.arange(0.45, 0.75, 0.05)]
            ),
            n_windows=N_WINDOWS,
        )
        
        # ── Tweak C: Cross-validate ResidualSSM correction_weight ───────────────────
        # Motivation: The residual correction scale (0.30) was chosen by intuition.
        # Different values can shift OOF macro-AUC by 0.5–1.5 pts depending on how
        # well the ResidualSSM generalises. We do a fast sweep over a small grid on
        # the TRAINING set (same data used to fit the model, so this is optimistic,
        # but the residual model is trained on a 15%-held-out val split which limits
        # leakage). The best weight is then applied to the test correction.
        t0 = time.time()
        res_model, correction_weight = train_residual_ssm(
            emb_full=emb_tr,
            first_pass_flat=first_pass_tr,
            Y_full=Y_FULL_aligned,
            site_ids=tr_site_ids,
            hour_ids=tr_hour_ids,
            n_epochs=30,
            patience=8,
            lr=1e-3,
            correction_weight=0.30,  # initial value; overridden by grid search below
            verbose=False,
        )
        print(f"ResidualSSM training: {time.time() - t0:.1f}s")
        
        # --- Tweak C grid search: find best correction_weight on training residuals ---
        # Original EOS-4 / Model_6 grid. Keep this unchanged to preserve the score path.
        _CORRECTION_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
        _fp_prob_tr = sigmoid(first_pass_tr)  # (N_windows_total, N_CLASSES)
        _emb_tr_f_c = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
        _fp_tr_f_c = first_pass_tr.reshape(n_tr_files, N_WINDOWS, -1)
        res_model.eval()
        with torch.no_grad():
            _tr_correction = res_model(
                torch.tensor(_emb_tr_f_c, dtype=torch.float32),
                torch.tensor(_fp_tr_f_c, dtype=torch.float32),
                site_ids=torch.tensor(tr_site_ids, dtype=torch.long),
                hours=torch.tensor(tr_hour_ids, dtype=torch.long),
            ).numpy().reshape(-1, N_CLASSES).astype(np.float32)
        
        _best_auc, _best_w = -1.0, 0.30
        for _w in _CORRECTION_GRID:
            _trial_scores = first_pass_tr + _w * _tr_correction
            _trial_probs = sigmoid(_trial_scores)
            _auc = macro_auc(Y_FULL_aligned, _trial_probs)
            print(f"  correction_weight={_w:.2f}  OOF macro-AUC={_auc:.5f}")
            if _auc > _best_auc:
                _best_auc, _best_w = _auc, _w
        
        correction_weight = _best_w  # override with CV-selected value
        print(f"[Tweak C] Best correction_weight={correction_weight:.2f}  (AUC={_best_auc:.5f})")
        del _emb_tr_f_c, _fp_tr_f_c, _tr_correction, _fp_prob_tr
        # ---------------------------------------------------------------------------
        
        first_pass_te_f = first_pass_flat.reshape(n_test_files, N_WINDOWS, -1)
        res_model.eval()
        with torch.no_grad():
            test_correction = res_model(
                torch.tensor(emb_te_f,         dtype=torch.float32),
                torch.tensor(first_pass_te_f,  dtype=torch.float32),
                site_ids=torch.tensor(test_site_ids, dtype=torch.long),
                hours   =torch.tensor(test_hour_ids, dtype=torch.long),
            ).numpy()
        correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)
        final_scores    = first_pass_flat + correction_weight * correction_flat
        final_scores    = final_scores / temperatures[None, :]
        probs = sigmoid(final_scores)
        probs = file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=POWEROPT_FILE_CONFIDENCE_POWER)
        probs = rank_aware_scaling(probs,    n_windows=N_WINDOWS, power=POWEROPT_SZ2_RANK_AWARE_POWER)
        probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=POWEROPT_DELTA_SMOOTH_ALPHA)
        probs = np.clip(probs, 0.0, 1.0)
        probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)   # ← now applied
        
        sub = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
        sub.insert(0, "row_id", meta_te["row_id"].values)
        sub.to_csv("submission_protossm.csv", index=False)
        print("ProtoSSM execution complete")
        print(f"Total wall time so far: {(time.time() - _WALL_START)/60:.1f} min")
        del emb_tr_f, sc_tr_f, proto_model, res_model
        gc.collect()
        print("Memory freed. Ready for SED cell.")


        # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

        if False and 'task2' in solutions and 'save PSSM for PowerOptimization SZ2 as' in solutions['task2']: 

            _subm_74p_csv = solutions['task2']['save PSSM for PowerOptimization SZ2 as']
            
            sub.to_csv(_subm_74p_csv, index=False)
            
            print('\n','#'*21, '\n _subm_74p_csv \n','#'*21, '\n')
        
        # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


        if _runSED_once and not os.path.isfile("submission_sed.csv"):
        
        
            import librosa
            from scipy.ndimage import gaussian_filter1d
            
            N_MELS_SED = 256
            N_FFT_SED  = 2048
            HOP_SED    = 512
            FMIN_SED   = 20
            FMAX_SED   = 16000
            TOP_DB_SED = 80
            
            def find_sed_dir():
                hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
                if not hits:
                    raise FileNotFoundError("sed_fold0.onnx not found. Attach tuckerarrants/bc2026-distilled-sed-public.")
                return hits[0].parent
        
            def make_sed_session(path):
                so = ort.SessionOptions()
                so.intra_op_num_threads = 4
                so.inter_op_num_threads = 1
                so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
                return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])
            
            def audio_to_mel(chunks):
                mels = []
                for x in chunks:
                    s = librosa.feature.melspectrogram(y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
                                                        n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0)
                    s = librosa.power_to_db(s, top_db=TOP_DB_SED)
                    s = (s - s.mean()) / (s.std() + 1e-6)
                    mels.append(s)
                return np.stack(mels)[:, None].astype(np.float32)
        
            def file_to_sed_chunks(path):
                y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
                if y.ndim == 2: y = y.mean(axis=1)
                if sr0 != SR: y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
                n = 60 * SR
                if len(y) < n: y = np.pad(y, (0, n - len(y)))
                else:          y = y[:n]
                chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                ends   = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC
                return chunks, ends
            
            def sigmoid_sed(x):
                return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)
        
            # Use the same test files as Cell 1
            test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
            IS_DRY_RUN = len(test_paths) == 0
            if IS_DRY_RUN:
                dry_n = CFG["dryrun_n_files"] if "CFG" in dir() else 20
                test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:(dry_n or 20)]
            
            sed_dir = find_sed_dir()
            sed_fold_paths = sorted(sed_dir.glob("sed_fold*.onnx"),
                                     key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
            sed_sessions = [make_sed_session(p) for p in sed_fold_paths]
            
            print(f"SED dir: {sed_dir}")
            print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")
            
            sed_rows, sed_preds = [], []
            
            for i, path in enumerate(test_paths, 1):
                chunks, ends = file_to_sed_chunks(path)
                mel = audio_to_mel(chunks)
                p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)
        
                for sess in sed_sessions:
                    outs = sess.run(None, {sess.get_inputs()[0].name: mel})
                    clip_logits = outs[0]             # (12, 234)
                    frame_max   = outs[1].max(axis=1) # (12, 234)
                    p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)
            
                p_mean = p_sum / len(sed_sessions)
            
                if len(p_mean) > 1:
                    p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)
            
                stem = path.stem
                sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
                sed_preds.append(p_mean)
            
                if i == 1 or i % 50 == 0 or i == len(test_paths):
                    print(f"SED: {i}/{len(test_paths)}")
            
            sed_preds_arr = np.concatenate(sed_preds, axis=0)
            sed_sub = pd.DataFrame(np.clip(sed_preds_arr, 0.0, 1.0), columns=PRIMARY_LABELS)
            sed_sub.insert(0, "row_id", sed_rows)
            sed_sub.to_csv("submission_sed.csv", index=False)
            print(f"Distilled SED Processing Complete. Shape: {sed_sub.shape}")
        
        
        import os
        import numpy as np
        import pandas as pd
        from pathlib import Path
        
        PROTOSSM_CSV = "submission_protossm.csv"
        SED_CSV      = "submission_sed.csv"
        OUT_CSV      = "submission.csv"
        EPS = 1e-5
        
        df_proto = pd.read_csv(PROTOSSM_CSV)
        df_sed   = pd.read_csv(SED_CSV)
        
        cols = [c for c in df_proto.columns if c != "row_id"]
        
        # Align row order
        df_sed = df_sed.set_index("row_id").loc[df_proto["row_id"]].reset_index()
        p_proto = np.clip(df_proto[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
        p_sed   = np.clip(df_sed[cols].to_numpy(np.float32),   EPS, 1.0 - EPS)
        
        rank_proto = pd.DataFrame(p_proto).rank(axis=0, pct=True).to_numpy(np.float32)
        rank_sed   = pd.DataFrame(p_sed).rank(axis=0, pct=True).to_numpy(np.float32)
        
        # ── xSED rank blend ───────────────────────────────────────────────────────────
        # Active current-submission path: Power optimization Proto/SED blend controlled by xSED.
        MODEL_NAME = "Karnakbayev_PowerOptimization_LB0948_74"
        _this_model = next(m for m in solut["Models"] if m["Model"] == MODEL_NAME)
        PROTO_W, SED_W = [float(v) for v in _this_model.get("xSED", [0.600, 0.400])]
        _xsed_sum = PROTO_W + SED_W
        if _xsed_sum <= 0:
            raise ValueError(f"Invalid xSED weights for {MODEL_NAME}: {_this_model.get('xSED')}")
        PROTO_W, SED_W = PROTO_W / _xsed_sum, SED_W / _xsed_sum
        print(f"Executing xSED rank blend ({PROTO_W:.4f} Proto / {SED_W:.4f} SED)")
        pred = (rank_proto * PROTO_W) + (rank_sed * SED_W)
    
        row_ids  = df_proto["row_id"].astype(str).to_numpy()
        file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])
        
        # ── Gate 1: Noise suppression ─────────────────────────────────────────────────
        # If ProtoSSM is confident but SED strongly disagrees → trust ProtoSSM more
        fake_only = (p_proto > 0.50) & (p_sed < 0.05)
        pred = np.where(fake_only, (1.0 - 0.08) * pred + 0.08 * rank_proto, pred)
        
        # ── Gate 2: Temporal continuity (fat-tailed t-distribution kernel) ─────────────
        # 35-second context window to protect continuous calls across windows
        offs = np.arange(-3, 4, dtype=np.float32)
        proto_kernel = (1.0 + (offs / 1.20) ** 2 / 2.0) ** (-1.5)
        proto_kernel = (proto_kernel / proto_kernel.sum()).astype(np.float32)
        
        pa_ctx = p_proto.copy()
        for fid in pd.unique(file_ids):
            m  = file_ids == fid
            x  = p_proto[m]
            if len(x) > 1:
                xp = np.pad(x, ((3, 3), (0, 0)), mode="edge")
                pa_ctx[m] = sum(proto_kernel[i] * xp[i:i + len(x)] for i in range(7))
        
        xctx = pd.DataFrame(pa_ctx).rank(axis=0, pct=True).to_numpy(np.float32)

        # ++++++++++++++++++++++++++++++++++++ original +++++++++++++++++++++++++++++++++++++++
        
        #proto_cont = (xctx > 0.88) & (rank_proto > 0.75) & (p_sed < 0.12) & (~fake_only) #m.74

        # +++++++++++++++++++++++++++++++ Japanese Amendment ++++++++++++++++++++++++++++++++++

        proto_cont = (xctx > 0.88) & (rank_proto > 0.77) & (p_sed < 0.14) & (~fake_only) #m.74

        # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
        

        
        pred = np.where(proto_cont,
                        (1.0 - 0.15) * pred + 0.15 * np.maximum(rank_proto, xctx),
                        pred)
        
        # ── Gate 3: SED spike preservation ────────────────────────────────────────────
        # Brief high-confidence SED detections that ProtoSSM missed
        sed_only = (rank_sed > 0.95) & (rank_proto < 0.80) & (~fake_only) & (~proto_cont)
        pred = np.where(sed_only, (1.0 - 0.12) * pred + 0.12 * rank_sed, pred)
        sub = df_proto.copy()
        sub[cols] = pred.astype(np.float32)
        
        # ── Gate 4: Sonotype mirroring ────────────────────────────────────────────────
        # Max-pool across visually identical species groups
        MIRROR_PAIRS = (
            ("47158son15", "47158son16"),
            ("47158son09", "47158son12"),
            ("47158son02", "47158son14"),
            ("47158son13", "47158son21", "47158son22", "47158son23")
        )
        col_to_idx = {l: i for i, l in enumerate(cols)}
        
        mirror_count = 0
        for group in MIRROR_PAIRS:
            valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
            if len(valid_idx) >= 2:
                group_max = sub[cols].iloc[:, valid_idx].max(axis=1).to_numpy(np.float32)
                for idx in valid_idx:
                    sub.iloc[:, idx + 1] = group_max
                mirror_count += len(valid_idx)
        print(f"Sonotype mirroring applied to {mirror_count} columns.")
        
        # ── Gate 5: Adaptive rare-class thresholding ──────────────────────────────────
        BASE_PATH = BASE
        try:
            tax_df = pd.read_csv(BASE_PATH / "taxonomy.csv").set_index("primary_label")
            rare_classes = {"Amphibia", "Mammalia", "Reptilia"}
            rare_count = 0
            for ci, species in enumerate(cols):
                if species in tax_df.index and tax_df.loc[species, "class_name"] in rare_classes:
                    col_idx = ci + 1
                    vals = sub.iloc[:, col_idx].to_numpy(np.float32)
                    thr = vals.mean() + 0.05
                    sub.iloc[:, col_idx] = np.where(vals < thr, vals * 0.9, vals)
                    rare_count += 1
            print(f"Adaptive thresholding applied to {rare_count} rare species.")
        except Exception as e:
            print(f"Adaptive thresholding skipped: {e}")
        
        # ── Dry-run alignment ──────────────────────────────────────────────────────────
        test_paths = list(BASE_PATH.glob("test_soundscapes/*.ogg"))
        IS_DRY_RUN = len(test_paths) == 0
        if IS_DRY_RUN:
            print("Dry-run detected: Aligning rows with sample_submission.csv")
            sample_public = pd.read_csv(BASE_PATH / "sample_submission.csv")
            template = sub[cols].mean(axis=0).astype(np.float32)
            sub = sample_public.copy()
            for label in cols:
                sub[label] = template[label]
        
        sub.to_csv(__file_name_submission, index=False)
        print(f"Blend and post-processing complete. Saved {__file_name_submission} shape={sub.shape}")
        print("Ready for submission!")
        
        
        # Final submission diagnostics: does not alter submission.csv
        from pathlib import Path
        import numpy as np
        import pandas as pd
        from IPython.display import display, Markdown
        
        submission_path = Path(__file_name_submission)
        assert submission_path.exists(), f"{__file_name_submission} was not created. Run the blend cell first."
        
        sub_check = pd.read_csv(submission_path)
        prob_cols = [c for c in sub_check.columns if c != "row_id"]
        
        summary = pd.DataFrame({
            "check": [
                "rows",
                "columns",
                "class columns",
                "missing values",
                "min probability",
                "max probability",
                "duplicated row_id",
            ],
            "value": [
                len(sub_check),
                sub_check.shape[1],
                len(prob_cols),
                int(sub_check.isna().sum().sum()),
                float(sub_check[prob_cols].min().min()) if prob_cols else np.nan,
                float(sub_check[prob_cols].max().max()) if prob_cols else np.nan,
                int(sub_check["row_id"].duplicated().sum()) if "row_id" in sub_check.columns else "row_id missing",
            ]
        })
        
        display(Markdown("### Submission diagnostic summary"))
        display(summary)
        
        assert "row_id" in sub_check.columns, "row_id column is missing."
        assert len(prob_cols) > 0, "No class probability columns found."
        assert np.isfinite(sub_check[prob_cols].to_numpy()).all(), "Non-finite values found in probability columns."
        assert sub_check[prob_cols].min().min() >= 0.0, "Probability columns contain values below 0."
        assert sub_check[prob_cols].max().max() <= 1.0, "Probability columns contain values above 1."
        
        print(f"{__file_name_submission} passed basic diagnostics.")
    
    
    import pandas as pd, os, time, sys
    import numpy as np
    from pathlib import Path
    from warnings import filterwarnings; filterwarnings("ignore")
    
    def _read_submission_checked(path):
        df = pd.read_csv(path)
        assert "row_id" in df.columns, f"row_id column missing in {path}"
        assert not any(str(c).startswith("Unnamed") for c in df.columns), f"unexpected unnamed column in {path}: {df.columns.tolist()[:5]}"
        assert df["row_id"].is_unique, f"duplicate row_id values in {path}"
        prob_cols = [c for c in df.columns if c != "row_id"]
        assert prob_cols, f"no probability columns in {path}"
        values = df[prob_cols].to_numpy(dtype=np.float32)
        assert np.isfinite(values).all(), f"NaN/inf values in {path}"
        assert values.min() >= 0.0 and values.max() <= 1.0, f"probabilities outside [0, 1] in {path}"
        out = df.set_index("row_id")
        out.index = out.index.astype(str)
        out.index.name = "row_id"
        return out
    
    def direct_add_safe():
        print(f'Ensemble: {__ensemble_models},   LB: {__lbs},   weights: {__weights}')
        assert len(__files_subm) == len(__weights), "submission file / weight length mismatch"
        weight_sum = float(sum(__weights))
        assert weight_sum > 0, "ensemble weights must sum to a positive value"
        if not np.isclose(weight_sum, 1.0, atol=1e-6):
            print(f"Normalizing ensemble weights from sum={weight_sum:.6f}")
        norm_weights = [float(w) / weight_sum for w in __weights]
        dfs = [_read_submission_checked(path) for path in __files_subm]
        base_idx = dfs[0].index
        base_cols = dfs[0].columns
        for path, df in zip(__files_subm, dfs):
            assert df.columns.equals(base_cols), f"Column mismatch in {path}"
            missing = base_idx.difference(df.index)
            extra = df.index.difference(base_idx)
            assert len(missing) == 0 and len(extra) == 0, (
                f"row_id mismatch in {path}: missing={len(missing)}, extra={len(extra)}"
            )
        out = sum(w * df.loc[base_idx, base_cols] for w, df in zip(norm_weights, dfs))
        out.index.name = "row_id"
        values = out.to_numpy(dtype=np.float32)
        assert np.isfinite(values).all(), "NaN/inf in final blend"
        assert values.min() >= 0.0 and values.max() <= 1.0, "final probabilities outside [0, 1]"
        return out
    
    def direct_add2():
        return direct_add_safe()
    
    def direct_add3():
        return direct_add_safe()
    
    def direct():
        return direct_add_safe()
    
    def rank_1():
        return direct_add_safe()
    
    def _find_sample_submission_path():
        base_obj = globals().get("BASE", None)
        if base_obj is not None:
            p = Path(base_obj) / "sample_submission.csv"
            if p.exists():
                return p
        for p in [
            Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv"),
            Path("/kaggle/input/birdclef-2026/sample_submission.csv"),
        ]:
            if p.exists():
                return p
        root = Path("/kaggle/input")
        if root.exists():
            hits = sorted(root.rglob("sample_submission.csv"))
            for p in hits:
                if (p.parent / "taxonomy.csv").exists():
                    return p
        return None
    
    def _as_explicit_submission_table(pred):
        if isinstance(pred, pd.DataFrame) and "row_id" in pred.columns:
            df = pred.copy()
        elif isinstance(pred, pd.DataFrame) and pred.index.name == "row_id":
            df = pred.reset_index()
        else:
            raise AssertionError("final prediction must be a DataFrame with a row_id column or row_id index")
        assert "row_id" in df.columns, "row_id column missing after final conversion"
        assert not any(str(c).startswith("Unnamed") for c in df.columns), f"unexpected unnamed columns: {df.columns.tolist()[:5]}"
        df["row_id"] = df["row_id"].astype(str)
        assert df["row_id"].is_unique, "duplicate row_id in final submission"
        return df
    
    def _align_to_sample_submission_if_possible(df):
        sample_path = _find_sample_submission_path()
        if sample_path is None:
            return df
        sample = pd.read_csv(sample_path)
        assert "row_id" in sample.columns, f"sample_submission has no row_id: {sample_path}"
        sample["row_id"] = sample["row_id"].astype(str)
        assert sample["row_id"].is_unique, f"duplicate row_id in sample_submission: {sample_path}"
        sample_cols = sample.columns.tolist()
        missing_cols = [c for c in sample_cols if c not in df.columns]
        assert not missing_cols, f"final submission is missing sample columns: {missing_cols[:5]}"
        final_ids = set(df["row_id"])
        sample_ids = set(sample["row_id"])
        if final_ids == sample_ids:
            aligned = df.set_index("row_id").loc[sample["row_id"], sample_cols[1:]].reset_index()
            aligned.columns = sample_cols
            return aligned
        missing = sorted(sample_ids - final_ids)[:5]
        extra = sorted(final_ids - sample_ids)[:5]
        raise AssertionError(
            "final row_id set differs from sample_submission: "
            f"missing={len(sample_ids-final_ids)} first={missing}, extra={len(final_ids-sample_ids)} first={extra}"
        )
    
    def write_final_submission(pred, path="submission.csv"):
        df = _as_explicit_submission_table(pred)
        df = _align_to_sample_submission_if_possible(df)
        prob_cols = [c for c in df.columns if c != "row_id"]
        assert prob_cols, "no probability columns in final submission"
        values = df[prob_cols].to_numpy(dtype=np.float32)
        assert np.isfinite(values).all(), "NaN/inf in final submission"
        assert values.min() >= 0.0 and values.max() <= 1.0, "final probabilities outside [0, 1]"
        df.to_csv(path, index=False)
        check = pd.read_csv(path)
        assert check.columns.tolist() == df.columns.tolist(), "written submission columns changed on reload"
        assert len(check) == len(df), "written submission row count changed on reload"
        assert check["row_id"].is_unique, "duplicate row_id after reload"
        print(f"Wrote {path}: rows={len(df)}, cols={df.shape[1]}, min={values.min():.6f}, max={values.max():.6f}")
        return df
    
    
    if solut['type_add'] in {'rank', 'rank.1'} : f_add = rank_1
    if solut['type_add'] == 'direct' : f_add = direct
    
    submission = f_add()
    
    final_submission = write_final_submission(submission, POWEROPT_SZ2_SUBM)

    final_submission.to_csv(POWEROPT_SZ2_SUBM, index=False)
    
    final_submission.head(3)

## Final Blend Utilities

Every branch CSV is reloaded before blending. `sample_submission.csv` defines the canonical class-column order when available. The writer checks `row_id`, duplicate rows, class alignment, finite values, `[0, 1]` bounds, and writes with `index=False`.


In [ ]:
import pandas as pd, os, time, sys
import numpy as np
from pathlib import Path
from warnings import filterwarnings; filterwarnings("ignore")

def _read_submission_checked(path):
    df = pd.read_csv(path)
    assert "row_id" in df.columns, f"row_id column missing in {path}"
    assert not any(str(c).startswith("Unnamed") for c in df.columns), f"unexpected unnamed column in {path}: {df.columns.tolist()[:5]}"
    assert df["row_id"].is_unique, f"duplicate row_id values in {path}"
    prob_cols = [c for c in df.columns if c != "row_id"]
    assert prob_cols, f"no probability columns in {path}"
    values = df[prob_cols].to_numpy(dtype=np.float32)
    assert np.isfinite(values).all(), f"NaN/inf values in {path}"
    assert values.min() >= 0.0 and values.max() <= 1.0, f"probabilities outside [0, 1] in {path}"
    out = df.set_index("row_id")
    out.index = out.index.astype(str)
    out.index.name = "row_id"
    return out

def direct_add_safe():
    print(f'Ensemble: {_ensemble_models},   LB: {_lbs},   weights: {_weights}')
    assert len(_files_subm) == len(_weights), "submission file / weight length mismatch"
    weight_sum = float(sum(_weights))
    assert weight_sum > 0, "ensemble weights must sum to a positive value"
    if not np.isclose(weight_sum, 1.0, atol=1e-6):
        print(f"Normalizing ensemble weights from sum={weight_sum:.6f}")
    norm_weights = [float(w) / weight_sum for w in _weights]
    dfs = [_read_submission_checked(path) for path in _files_subm]

    sample_path = _find_sample_submission_path()
    if sample_path is not None:
        sample = pd.read_csv(sample_path)
        target_cols = sample.columns[1:].tolist()
        for path, df in zip(_files_subm, dfs):
            missing_cols = [c for c in target_cols if c not in df.columns]
            assert not missing_cols, f"{path} missing sample columns: {missing_cols[:5]}"
        dfs = [df.loc[:, target_cols] for df in dfs]

    base_idx = dfs[0].index
    base_cols = dfs[0].columns
    for path, df in zip(_files_subm, dfs):
        assert df.columns.equals(base_cols), f"Column mismatch in {path}"
        missing = base_idx.difference(df.index)
        extra = df.index.difference(base_idx)
        assert len(missing) == 0 and len(extra) == 0, (
            f"row_id mismatch in {path}: missing={len(missing)}, extra={len(extra)}"
        )
    out = sum(w * df.loc[base_idx, base_cols] for w, df in zip(norm_weights, dfs))
    out.index.name = "row_id"
    values = out.to_numpy(dtype=np.float32)
    assert np.isfinite(values).all(), "NaN/inf in final blend"
    assert values.min() >= 0.0 and values.max() <= 1.0, "final probabilities outside [0, 1]"
    return out

def direct_add2():
    return direct_add_safe()

def direct_add3():
    return direct_add_safe()

def direct():
    return direct_add_safe()

def rank_1():
    raise RuntimeError("rank_1 is disabled in this notebook; use direct_safe or TAX_SMOOTHING only.")



def _find_taxonomy_path_for_smoothing():
    candidates = [
        Path("/kaggle/input/competitions/birdclef-2026/taxonomy.csv"),
        Path("/kaggle/input/birdclef-2026/taxonomy.csv"),
    ]
    base_obj = globals().get("BASE", None)
    if base_obj is not None:
        candidates.insert(0, Path(base_obj) / "taxonomy.csv")
    base_path_obj = globals().get("BASE_PATH", None)
    if base_path_obj is not None:
        candidates.insert(0, Path(base_path_obj) / "taxonomy.csv")
    for path in candidates:
        if path.exists():
            return path
    root = Path("/kaggle/input")
    if root.exists():
        for path in sorted(root.rglob("taxonomy.csv")):
            return path
    return None


def f_TAX_SMOOTHING_POSTPROC(f_direct=direct, genus_alpha=None, class_alpha=None):
    submission = f_direct()
    genus_alpha = TAX_GENUS_ALPHA if genus_alpha is None else float(genus_alpha)
    class_alpha = TAX_CLASS_ALPHA if class_alpha is None else float(class_alpha)

    taxonomy_path = _find_taxonomy_path_for_smoothing()
    if taxonomy_path is None:
        msg = "taxonomy.csv not found; TAX_SMOOTHING cannot be applied"
        if globals().get("TAX_SMOOTHING_REQUIRE", True):
            raise FileNotFoundError(msg)
        print(msg + "; skipping taxonomy smoothing")
        submission.index.name = "row_id"
        return submission

    tax = pd.read_csv(taxonomy_path)
    species_to_genus = {}
    species_to_class = {}
    for _, row in tax.iterrows():
        label = str(row.get("primary_label", ""))
        sci = str(row.get("scientific_name", ""))
        class_name = str(row.get("class_name", ""))
        genus = sci.split(" ")[0] if " " in sci else sci
        if label:
            species_to_genus[label] = genus
            species_to_class[label] = class_name

    cols = list(submission.columns)
    genus_groups = {}
    class_groups = {}
    for col in cols:
        genus = species_to_genus.get(col, col)
        class_name = species_to_class.get(col, "")
        genus_groups.setdefault(genus, []).append(col)
        if class_name:
            class_groups.setdefault(class_name, []).append(col)

    multi_genus = {k: v for k, v in genus_groups.items() if len(v) > 1}
    multi_class = {k: v for k, v in class_groups.items() if len(v) > 1}
    print({
        "taxonomy_path": str(taxonomy_path),
        "genus_groups_used": len(multi_genus),
        "class_groups_used": len(multi_class),
        "genus_alpha": genus_alpha,
        "class_alpha": class_alpha,
    })

    probs = submission.to_numpy(dtype=np.float32, copy=True)
    col_to_idx = {col: i for i, col in enumerate(cols)}

    for members in multi_genus.values():
        idx = [col_to_idx[m] for m in members]
        group_mean = probs[:, idx].mean(axis=1, keepdims=True)
        probs[:, idx] = (1.0 - genus_alpha) * probs[:, idx] + genus_alpha * group_mean

    for members in multi_class.values():
        idx = [col_to_idx[m] for m in members]
        group_mean = probs[:, idx].mean(axis=1, keepdims=True)
        probs[:, idx] = (1.0 - class_alpha) * probs[:, idx] + class_alpha * group_mean

    probs = np.clip(probs, 0.0, 1.0)
    out = pd.DataFrame(probs, index=submission.index, columns=submission.columns)
    out.index.name = "row_id"
    return out

def _find_sample_submission_path():
    base_obj = globals().get("BASE", None)
    if base_obj is not None:
        p = Path(base_obj) / "sample_submission.csv"
        if p.exists():
            return p
    for p in [
        Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv"),
        Path("/kaggle/input/birdclef-2026/sample_submission.csv"),
    ]:
        if p.exists():
            return p
    root = Path("/kaggle/input")
    if root.exists():
        hits = sorted(root.rglob("sample_submission.csv"))
        for p in hits:
            if (p.parent / "taxonomy.csv").exists():
                return p
    return None

def _as_explicit_submission_table(pred):
    if isinstance(pred, pd.DataFrame) and "row_id" in pred.columns:
        df = pred.copy()
    elif isinstance(pred, pd.DataFrame) and pred.index.name == "row_id":
        df = pred.reset_index()
    else:
        raise AssertionError("final prediction must be a DataFrame with a row_id column or row_id index")
    assert "row_id" in df.columns, "row_id column missing after final conversion"
    assert not any(str(c).startswith("Unnamed") for c in df.columns), f"unexpected unnamed columns: {df.columns.tolist()[:5]}"
    df["row_id"] = df["row_id"].astype(str)
    assert df["row_id"].is_unique, "duplicate row_id in final submission"
    return df

def _align_to_sample_submission_if_possible(df):
    sample_path = _find_sample_submission_path()
    if sample_path is None:
        return df
    sample = pd.read_csv(sample_path)
    assert "row_id" in sample.columns, f"sample_submission has no row_id: {sample_path}"
    sample["row_id"] = sample["row_id"].astype(str)
    assert sample["row_id"].is_unique, f"duplicate row_id in sample_submission: {sample_path}"
    sample_cols = sample.columns.tolist()
    missing_cols = [c for c in sample_cols if c not in df.columns]
    assert not missing_cols, f"final submission is missing sample columns: {missing_cols[:5]}"
    final_ids = set(df["row_id"])
    sample_ids = set(sample["row_id"])
    if final_ids == sample_ids:
        aligned = df.set_index("row_id").loc[sample["row_id"], sample_cols[1:]].reset_index()
        aligned.columns = sample_cols
        return aligned
    missing = sorted(sample_ids - final_ids)[:5]
    extra = sorted(final_ids - sample_ids)[:5]
    raise AssertionError(
        "final row_id set differs from sample_submission: "
        f"missing={len(sample_ids-final_ids)} first={missing}, extra={len(final_ids-sample_ids)} first={extra}"
    )

def write_final_submission(pred, path="submission.csv"):
    df = _as_explicit_submission_table(pred)
    df = _align_to_sample_submission_if_possible(df)
    prob_cols = [c for c in df.columns if c != "row_id"]
    assert prob_cols, "no probability columns in final submission"
    values = df[prob_cols].to_numpy(dtype=np.float32)
    assert np.isfinite(values).all(), "NaN/inf in final submission"
    assert values.min() >= 0.0 and values.max() <= 1.0, "final probabilities outside [0, 1]"
    df.to_csv(path, index=False)
    check = pd.read_csv(path)
    assert check.columns.tolist() == df.columns.tolist(), "written submission columns changed on reload"
    assert len(check) == len(df), "written submission row count changed on reload"
    assert check["row_id"].is_unique, "duplicate row_id after reload"
    print(f"Wrote {path}: rows={len(df)}, cols={df.shape[1]}, min={values.min():.6f}, max={values.max():.6f}")
    return df


In [ ]:
if solutions["type_add"] in {"direct", "direct_safe"}:
    f_add = direct
elif solutions["type_add"] in {"rank", "rank.1"}:
    f_add = rank_1
elif solutions["type_add"] == "TAX_SMOOTHING":
    f_add = f_TAX_SMOOTHING_POSTPROC
else:
    raise ValueError(f"Unknown type_add: {solutions['type_add']}")


## Write Submission

The anchor output schema is:

```text
row_id, class_1, class_2, ...
```

This file is also saved as `submission_before_all_sidecars.csv` before optional rank corrections.


In [ ]:
submission = f_add()
final_submission = write_final_submission(submission, "submission.csv")
final_submission.to_csv("submission_before_all_sidecars.csv", index=False)


In [ ]:
# Anchor preview before optional rank corrections.
final_submission.head(3)


## Optional BirdNET Rank Correction

BirdNET is an optional class-gated side prediction controlled by the front settings. The current submission is anchor `A`; BirdNET is `S`; only masked cells move:

```text
B = A + W_class * M * (S - A)
```

The intended use is to keep non-bird taxa and uncovered classes fixed, assign a smaller cap to covered Aves classes already mapped by Perch, and assign a larger cap to covered Aves classes where Perch has no direct logit. Group perturbation budgets shrink the caps automatically when BirdNET would move the anchor too much.


In [ ]:
# Optional BirdNET sidecar: constrained rank-space correction of submission.csv.
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd


def _birdnet_find_competition_dir():
    base_obj = globals().get("BASE", None)
    if base_obj is not None:
        p = Path(base_obj)
        if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
            return p
    for p in [
        Path("/kaggle/input/competitions/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026"),
    ]:
        if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
            return p
    root = Path("/kaggle/input")
    if root.exists():
        for sample_path in sorted(root.rglob("sample_submission.csv")):
            parent = sample_path.parent
            if (parent / "taxonomy.csv").exists():
                return parent
    return None


def _find_first(patterns, explicit_patterns=None):
    roots = []
    explicit_root = str(globals().get("BIRDNET_MODEL_ROOT", "")).strip()
    if explicit_root:
        roots.append((Path(explicit_root), list(explicit_patterns or []) + list(patterns)))
    roots.append((Path("/kaggle/input"), list(patterns)))

    seen = set()
    for root, pats in roots:
        if not root.exists():
            continue
        root_key = str(root.resolve())
        if root_key in seen:
            continue
        seen.add(root_key)
        for pat in pats:
            hits = sorted(root.rglob(pat))
            if hits:
                return hits[0]
    return None


def _find_birdnet_model():
    return _find_first(
        [
            "**/birdnet_global_6k_v2.4_model_fp32*.tflite",
            "**/BirdNET_GLOBAL_6K_V2.4_Model_FP32*.tflite",
            "**/*birdnet*fp32*.tflite",
            "**/*birdnet*.tflite",
        ],
        explicit_patterns=["*.tflite", "**/*.tflite"],
    )


def _find_birdnet_labels():
    return _find_first(
        [
            "**/BirdNET_GLOBAL_6K_V2.4_Labels.txt",
            "**/birdnet*labels*.txt",
            "**/*birdnet*label*.txt",
        ],
        explicit_patterns=["BirdNET_GLOBAL_6K_V2.4_Labels.txt", "labels.txt", "*.txt", "**/*.txt"],
    )


def _safe_rank_cols(df):
    return df.rank(axis=0, method="average", pct=True).astype(np.float32)


def _topk_mask(arr, k):
    k = int(min(max(k, 1), arr.shape[1]))
    idx = np.argpartition(arr, -k, axis=1)[:, -k:]
    mask = np.zeros(arr.shape, dtype=bool)
    rows = np.arange(arr.shape[0])[:, None]
    mask[rows, idx] = True
    return mask


def _topk_overlap(A, B, k):
    k = int(min(max(k, 1), A.shape[1]))
    ai = np.argpartition(A, -k, axis=1)[:, -k:]
    bi = np.argpartition(B, -k, axis=1)[:, -k:]
    return float(np.mean([len(set(x).intersection(set(y))) / k for x, y in zip(ai, bi)]))


def _masked_rank_blend(anchor_df, side_df, class_meta):
    assert anchor_df.index.equals(side_df.index), "BirdNET row_id order differs from anchor"
    assert anchor_df.columns.equals(side_df.columns), "BirdNET class columns differ from anchor"
    assert list(class_meta["label"]) == list(anchor_df.columns), "class metadata does not match anchor columns"

    A = _safe_rank_cols(anchor_df).to_numpy(np.float32)
    S = _safe_rank_cols(side_df).to_numpy(np.float32)
    side_probs = side_df.to_numpy(np.float32)

    aves = class_meta["class_name"].eq("Aves").to_numpy(bool)
    perch_mapped = class_meta["perch_mapped"].to_numpy(bool)
    mapping_covered = class_meta["birdnet_mapped_or_proxy"].to_numpy(bool)
    score_covered = side_probs.max(axis=0) > float(BIRDNET_COVERAGE_MIN)
    covered = mapping_covered & score_covered

    mapped_cols = aves & perch_mapped & covered
    unmapped_cols = aves & (~perch_mapped) & covered
    non_bird_cols = (~aves) & covered

    mask_mapped = (
        _topk_mask(A, BIRDNET_MAPPED_ANCHOR_TOPK)
        | (_topk_mask(S, BIRDNET_MAPPED_SIDE_TOPK) & (A >= float(BIRDNET_MAPPED_TAU)))
    )
    mask_unmapped = (
        _topk_mask(A, BIRDNET_UNMAPPED_ANCHOR_TOPK)
        | (_topk_mask(S, BIRDNET_UNMAPPED_SIDE_TOPK) & (A >= float(BIRDNET_UNMAPPED_TAU)))
    )

    mask = np.zeros_like(A, dtype=bool)
    if mapped_cols.any():
        mask[:, mapped_cols] = mask_mapped[:, mapped_cols]
    if unmapped_cols.any():
        mask[:, unmapped_cols] = mask_unmapped[:, unmapped_cols]
    if non_bird_cols.any() and BIRDNET_NON_BIRD_WEIGHT_CAP > 0:
        mask[:, non_bird_cols] = mask_mapped[:, non_bird_cols]

    w_class = np.zeros(A.shape[1], dtype=np.float32)
    w_class[mapped_cols] = float(BIRDNET_MAPPED_WEIGHT_CAP)
    w_class[unmapped_cols] = float(BIRDNET_UNMAPPED_WEIGHT_CAP)
    w_class[non_bird_cols] = float(BIRDNET_NON_BIRD_WEIGHT_CAP)
    W = np.broadcast_to(w_class[None, :], A.shape)

    def _apply(scale):
        B = A.copy()
        if mask.any() and scale > 0:
            B[mask] = A[mask] + (S[mask] - A[mask]) * W[mask] * float(scale)
        delta = np.abs(B - A)
        def _group_mean(cols):
            return float(delta[:, cols].mean()) if cols.any() else 0.0
        return B, {
            "D_total": float(delta.mean()),
            "D_mapped": _group_mean(mapped_cols),
            "D_unmapped": _group_mean(unmapped_cols),
            "D_non_bird": _group_mean(non_bird_cols),
        }

    B0, d0 = _apply(1.0)
    scale = 1.0
    for key, budget in [
        ("D_total", BIRDNET_TOTAL_D_BUDGET),
        ("D_mapped", BIRDNET_MAPPED_D_BUDGET),
        ("D_unmapped", BIRDNET_UNMAPPED_D_BUDGET),
        ("D_non_bird", BIRDNET_NON_BIRD_D_BUDGET),
    ]:
        val = d0[key]
        if val > float(budget) > 0:
            scale = min(scale, float(budget) / val)
        elif float(budget) == 0 and val > 0:
            scale = 0.0

    B, d = _apply(scale)
    active = float(mask.mean())
    top3 = _topk_overlap(A, B, 3)
    top10 = _topk_overlap(A, B, 10)
    ok = (
        mask.any()
        and scale > 0.0
        and active <= float(BIRDNET_MAX_ACTIVE_FRACTION)
        and top3 >= float(BIRDNET_MIN_TOP3_OVERLAP)
        and top10 >= float(BIRDNET_MIN_TOP10_OVERLAP)
    )

    info = {
        "mapped_weight_cap": BIRDNET_MAPPED_WEIGHT_CAP,
        "unmapped_weight_cap": BIRDNET_UNMAPPED_WEIGHT_CAP,
        "effective_scale": float(scale),
        "effective_mapped_weight": float(BIRDNET_MAPPED_WEIGHT_CAP) * float(scale),
        "effective_unmapped_weight": float(BIRDNET_UNMAPPED_WEIGHT_CAP) * float(scale),
        "covered_classes": int(covered.sum()),
        "mapped_aves_classes": int(mapped_cols.sum()),
        "unmapped_aves_classes": int(unmapped_cols.sum()),
        "non_bird_covered_classes": int(non_bird_cols.sum()),
        "active_fraction": active,
        "top3_overlap": top3,
        "top10_overlap": top10,
        "ok": ok,
        **d,
    }
    diag = pd.DataFrame([info])
    diag.to_csv(BIRDNET_DIAGNOSTICS_CSV, index=False)
    if not ok:
        return None, diag, info
    blend = pd.DataFrame(B, index=anchor_df.index, columns=anchor_df.columns)
    return blend, diag, info

def _load_birdnet_interpreter(model_path):
    try:
        from tflite_runtime.interpreter import Interpreter
    except ImportError:
        from tensorflow.lite.python.interpreter import Interpreter
    interp = Interpreter(model_path=str(model_path), num_threads=4)
    interp.allocate_tensors()
    return interp


def _birdnet_input_array(chunk, input_shape):
    chunk = chunk.astype(np.float32, copy=False)
    if len(input_shape) == 3:
        return chunk[None, :, None]
    return chunk[None, :]


def _run_birdnet(paths, base, class_cols):
    import soundfile as sf
    import librosa as _librosa
    from scipy.ndimage import gaussian_filter1d as _gf1d
    from tqdm.auto import tqdm

    model_path = _find_birdnet_model()
    labels_path = _find_birdnet_labels()
    if model_path is None or labels_path is None:
        print("BirdNET TFLite model or label file not found; keeping anchor submission.")
        return None

    print("BirdNET model:", model_path)
    print("BirdNET labels:", labels_path)

    try:
        interp = _load_birdnet_interpreter(model_path)
    except Exception as exc:
        print(f"BirdNET interpreter load failed: {exc}; keeping anchor submission.")
        return None
    in_info = interp.get_input_details()[0]
    out_info = interp.get_output_details()[-1]

    labels_raw = [line.strip() for line in Path(labels_path).read_text().splitlines() if line.strip()]
    label_sci = [line.split("_", 1)[0].strip() for line in labels_raw]

    taxonomy = pd.read_csv(base / "taxonomy.csv")
    taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
    taxonomy["scientific_name"] = taxonomy["scientific_name"].astype(str)
    taxonomy["class_name"] = taxonomy["class_name"].astype(str)
    sci_to_primary = taxonomy.set_index("scientific_name")["primary_label"].to_dict()
    primary_to_sci = taxonomy.set_index("primary_label")["scientific_name"].to_dict()
    class_to_idx = {c: i for i, c in enumerate(class_cols)}

    bn_to_comp = {}
    for bn_i, sci in enumerate(label_sci):
        primary = sci_to_primary.get(sci)
        if primary in class_to_idx:
            bn_to_comp[bn_i] = class_to_idx[primary]

    direct_comp = set(bn_to_comp.values())
    bn_proxy = {}
    for c, ci in class_to_idx.items():
        if ci in direct_comp:
            continue
        sci = primary_to_sci.get(c, "")
        genus = sci.split(" ", 1)[0]
        if not genus:
            continue
        idxs = [i for i, bn_sci in enumerate(label_sci) if bn_sci.startswith(genus + " ")]
        if idxs:
            bn_proxy[ci] = idxs

    print(f"BirdNET mapping: {len(bn_to_comp)} direct labels, {len(bn_proxy)} genus proxies")

    class_meta = pd.DataFrame({"label": list(class_cols)})
    tax_by_primary = taxonomy.set_index("primary_label")
    class_meta["class_name"] = [str(tax_by_primary.loc[c, "class_name"]) if c in tax_by_primary.index else "" for c in class_cols]
    class_meta["birdnet_direct"] = False
    class_meta["birdnet_proxy"] = False
    if direct_comp:
        class_meta.loc[list(direct_comp), "birdnet_direct"] = True
    if bn_proxy:
        class_meta.loc[list(bn_proxy.keys()), "birdnet_proxy"] = True
    class_meta["birdnet_mapped_or_proxy"] = class_meta["birdnet_direct"] | class_meta["birdnet_proxy"]

    perch_mapped = np.zeros(len(class_cols), dtype=bool)
    if "MAPPED_MASK" in globals() and len(MAPPED_MASK) == len(class_cols):
        perch_mapped = np.asarray(MAPPED_MASK, dtype=bool).copy()
    elif "MAPPED_POS" in globals():
        _pos = np.asarray(MAPPED_POS, dtype=int)
        perch_mapped[_pos[(_pos >= 0) & (_pos < len(class_cols))]] = True
    else:
        perch_mapped[:] = True
        print("Perch mapped mask not found; treating classes as mapped for conservative BirdNET weights")
    class_meta["perch_mapped"] = perch_mapped

    sr = 48_000
    chunk_sec = 3
    chunk_samples = sr * chunk_sec
    n_chunks = 20
    n_windows = 12
    win_to_chunks = []
    for w in range(n_windows):
        ws, we = w * 5, (w + 1) * 5
        win_to_chunks.append([j for j in range(n_chunks) if 3 * j < we and 3 * (j + 1) > ws])

    n_rows = len(paths) * n_windows
    row_ids = np.empty(n_rows, dtype=object)
    scores = np.zeros((n_rows, len(class_cols)), dtype=np.float32)
    wr = 0

    for path in tqdm([Path(x) for x in paths], desc="BirdNET"):
        y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
        if getattr(y, "ndim", 1) == 2:
            y = y.mean(axis=1)
        if sr0 != sr:
            y = _librosa.resample(y, orig_sr=sr0, target_sr=sr)
        target_len = 60 * sr
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len]

        chunks = y.reshape(n_chunks, chunk_samples)
        chunk_probs = np.zeros((n_chunks, len(labels_raw)), dtype=np.float32)
        for j, chunk in enumerate(chunks):
            interp.set_tensor(in_info["index"], _birdnet_input_array(chunk, in_info["shape"]))
            interp.invoke()
            logits = interp.get_tensor(out_info["index"])[0]
            chunk_probs[j] = 1.0 / (1.0 + np.exp(-np.clip(logits, -50, 50)))

        stem = path.stem
        for w, clist in enumerate(win_to_chunks):
            wp = chunk_probs[clist].max(axis=0)
            r = wr + w
            row_ids[r] = f"{stem}_{(w + 1) * 5}"
            for bn_i, ci in bn_to_comp.items():
                scores[r, ci] = max(scores[r, ci], wp[bn_i])
            for ci, bn_idxs in bn_proxy.items():
                scores[r, ci] = max(scores[r, ci], float(wp[bn_idxs].max()))
        wr += n_windows

    if wr > 0:
        scores_3d = scores[:wr].reshape(len(paths), n_windows, len(class_cols))
        scores[:wr] = np.clip(_gf1d(scores_3d, sigma=0.65, axis=1, mode="nearest").reshape(wr, -1), 0.0, 1.0)

    out = pd.DataFrame(scores[:wr], columns=class_cols)
    out.insert(0, "row_id", row_ids[:wr].astype(str))
    return out, class_meta


if not RUN_BIRDNET_SIDECAR:
    print("BirdNET sidecar disabled; keeping anchor submission.")
else:
    base = _birdnet_find_competition_dir()
    if base is None:
        print("Competition data not found; keeping anchor submission.")
    else:
        anchor_full = pd.read_csv("submission.csv")
        assert "row_id" in anchor_full.columns, "submission.csv has no row_id column"
        anchor_full["row_id"] = anchor_full["row_id"].astype(str)
        anchor_cols = [c for c in anchor_full.columns if c != "row_id"]
        anchor = anchor_full.set_index("row_id")[anchor_cols]

        test_paths = sorted((base / "test_soundscapes").glob("*.ogg"))
        if not test_paths:
            dry_paths = sorted((base / "train_soundscapes").glob("*.ogg"))[:int(BIRDNET_DRYRUN_FILES)]
            print(f"No test soundscapes visible; BirdNET dry-run files={len(dry_paths)}")
            test_paths = dry_paths

        if not test_paths:
            print("No audio files found for BirdNET; keeping anchor submission.")
        else:
            t0 = time.time()
            birdnet_result = _run_birdnet(test_paths, base, anchor_cols)
            if birdnet_result is None:
                print("BirdNET prediction unavailable; keeping anchor submission.")
            else:
                birdnet_full, birdnet_class_meta = birdnet_result
                birdnet_full.to_csv(BIRDNET_SIDECAR_SUBM, index=False)
                birdnet_full["row_id"] = birdnet_full["row_id"].astype(str)
                side = birdnet_full.set_index("row_id")[anchor_cols]
                missing = anchor.index.difference(side.index)
                extra = side.index.difference(anchor.index)
                if len(missing) or len(extra):
                    print(
                        "BirdNET row_id mismatch; keeping anchor submission. "
                        f"missing={len(missing)}, extra={len(extra)}"
                    )
                else:
                    side = side.loc[anchor.index, anchor_cols]
                    blend, diag, best_info = _masked_rank_blend(anchor, side, birdnet_class_meta)
                    print(diag.to_string(index=False))
                    if blend is None:
                        print("No BirdNET mask passed overlap/active guards; keeping anchor submission.")
                    else:
                        final_submission = write_final_submission(blend, "submission.csv")
                        print("Applied BirdNET sidecar:", best_info)
                print(f"BirdNET sidecar elapsed: {time.time() - t0:.1f}s")


## Optional PCEN/ConvNeXt Sidecar Correction

The active PCEN/log-mel ConvNeXt sidecar variant is selected in the control cell. In `dual_exp002b_exp002` mode, exp002b is applied first and exp002 is applied second, each with its own asset, OOF gate, timeout, output CSV, and diagnostics file. It is applied only after the EoS9 taxonomy-smoothed anchor submission has been written, and it is used as a local rank correction rather than a third global ensemble branch.

```text
A = R(p_anchor)
S = R(p_sidecar)
B = A + W_class * M * (S - A)
```

The mask `M` limits corrections to classes already plausible under the anchor or strongly proposed by the sidecar while still passing an anchor-rank threshold.

The OOF gate is class-wise:

```text
W_class[c] = gate_weight[c]
class c is active only if gate_weight[c] > 0
```

The gate is generated outside the notebook from train-soundscape OOF/replay artifacts. It compares the current anchor replay with the sidecar OOF under the same masked blend rule. A class is opened only when the best masked blend weight gives positive anchor-delta AUC after sparse-class and group-level shrinkage checks. Classes that do not pass remain exactly anchored because their `gate_weight` is zero.

The final writer then rechecks row order, class columns, finite values, and probability range. Dual mode is useful when the two OOF gates open different class groups; the sidecars are still sequential masked corrections, not extra global ensemble branches.


In [ ]:
# Optional custom sidecar: constrained rank-space correction of submission.csv.
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


def _exp002_find_competition_dir():
    for key in ["BASE", "BASE_PATH"]:
        base_obj = globals().get(key, None)
        if base_obj is not None:
            p = Path(base_obj)
            if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
                return p
    for p in [
        Path("/kaggle/input/competitions/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026"),
    ]:
        if (p / "sample_submission.csv").exists() and (p / "taxonomy.csv").exists():
            return p
    root = Path("/kaggle/input")
    if root.exists():
        for sample_path in sorted(root.rglob("sample_submission.csv")):
            parent = sample_path.parent
            if (parent / "taxonomy.csv").exists():
                return parent
    return None


def _exp002_test_soundscape_paths(base_dir):
    test_dir = Path(base_dir) / "test_soundscapes"
    if not test_dir.exists():
        return []
    return sorted(test_dir.glob("*.ogg"))


def _exp002_write_skip_diag(reason, mode="dry_run", applicable=False):
    info = {
        "mode": str(mode),
        "applicable": bool(applicable),
        "weight_cap": float(SIDECAR_EXP002_WEIGHT_CAP),
        "effective_weight": 0.0,
        "E": 0.0,
        "D": 0.0,
        "D_budget": float(SIDECAR_EXP002_D_BUDGET),
        "anchor_topk": int(SIDECAR_EXP002_ANCHOR_TOPK),
        "side_topk": int(SIDECAR_EXP002_SIDE_TOPK),
        "tau": float(SIDECAR_EXP002_TAU),
        "active_fraction": 0.0,
        "top3_overlap": 1.0,
        "top10_overlap": 1.0,
        "ok": False,
        "skip_reason": str(reason),
    }
    pd.DataFrame([info]).to_csv(SIDECAR_EXP002_DIAGNOSTICS_CSV, index=False)
    return info


def _exp002_is_asset_root(p):
    p = Path(p)
    return (
        p.exists()
        and (p / "infer.py").exists()
        and (p / "config.yaml").exists()
        and any(p.glob("fold*.pt"))
    )


def _exp002_asset_matches_variant(p):
    if not _exp002_is_asset_root(p):
        return False
    try:
        import yaml

        cfg = yaml.safe_load((Path(p) / "config.yaml").read_text()) or {}
        audio = cfg.get("audio", {}) or {}
        feature = cfg.get("feature", {}) or {}
        return (
            float(audio.get("context_seconds", -1)) == float(SIDECAR_EXP002_EXPECTED_CONTEXT_SECONDS)
            and float(audio.get("target_seconds", -1)) == float(SIDECAR_EXP002_EXPECTED_TARGET_SECONDS)
            and int(feature.get("image_time", -1)) == int(SIDECAR_EXP002_EXPECTED_IMAGE_TIME)
        )
    except Exception:
        return False


def _find_exp002_asset_root():
    configured = Path(str(globals().get("SIDECAR_EXP002_ROOT", "")).strip())
    candidates = [
        configured,
        configured / "sidecar_exp002",
        configured / "sidecar_exp002b",
        Path("/kaggle/input/datasets/pilkwang/birdclef26-sidecar-exp002b-5s-weakaudio"),
        Path("/kaggle/input/datasets/pilkwang/birdclef26-sidecar-exp002b-5s-weakaudio/sidecar_exp002b"),
        Path("/kaggle/input/datasets/pilkwang/birdclef26-sidecar-exp002b-5s-weakaudio/sidecar_exp002"),
        Path("/kaggle/input/birdclef26-sidecar-exp002b-5s-weakaudio"),
        Path("/kaggle/input/birdclef26-sidecar-exp002b-5s-weakaudio/sidecar_exp002b"),
        Path("/kaggle/input/birdclef26-sidecar-exp002b-5s-weakaudio/sidecar_exp002"),
        Path("/kaggle/input/datasets/pilkwang/birdclef26-sidecar-exp002-10s-weakaudio"),
        Path("/kaggle/input/datasets/pilkwang/birdclef26-sidecar-exp002-10s-weakaudio/sidecar_exp002"),
        Path("/kaggle/input/birdclef26-sidecar-exp002-10s-weakaudio"),
        Path("/kaggle/input/birdclef26-sidecar-exp002-10s-weakaudio/sidecar_exp002"),
        Path("/kaggle/input/datasets/pilkwang/birdclef26-sidecar-exp002/sidecar_exp002"),
        Path("/kaggle/input/birdclef26-sidecar-exp002/sidecar_exp002"),
        Path("/kaggle/input/birdclef26-sidecar-exp002"),
    ]
    structurally_valid = []
    for p in candidates:
        if str(p) and _exp002_is_asset_root(p):
            if _exp002_asset_matches_variant(p):
                return p
            structurally_valid.append(Path(p))
    root = Path("/kaggle/input")
    if root.exists():
        for p in sorted(list(root.rglob("sidecar_exp002")) + list(root.rglob("sidecar_exp002b"))):
            if _exp002_is_asset_root(p):
                if _exp002_asset_matches_variant(p):
                    return p
                structurally_valid.append(Path(p))
    if structurally_valid:
        print(
            f"Found sidecar asset roots, but none match {SIDECAR_EXP002_VARIANT}: "
            f"{[str(p) for p in structurally_valid[:5]]}"
        )
    return None


def _print_exp002_asset_sanity(asset_root):
    import yaml

    asset_root = Path(asset_root)
    cfg_path = asset_root / "config.yaml"
    assert cfg_path.exists(), f"config.yaml missing in {asset_root}"

    cfg = yaml.safe_load(cfg_path.read_text()) or {}
    audio = cfg.get("audio", {}) or {}
    feature = cfg.get("feature", {}) or {}
    checkpoints = sorted(p.name for p in asset_root.glob("fold*.pt"))
    gate_path = asset_root / str(SIDECAR_EXP002_GATE_CSV)

    print(f"[{SIDECAR_EXP002_LABEL} asset] root:", asset_root)
    print(f"[{SIDECAR_EXP002_LABEL} asset] experiment:", cfg.get("experiment"))
    print(f"[{SIDECAR_EXP002_LABEL} asset] audio:", audio)
    print(f"[{SIDECAR_EXP002_LABEL} asset] image_time:", feature.get("image_time"))
    print(f"[{SIDECAR_EXP002_LABEL} asset] gate_csv:", gate_path)
    print(f"[{SIDECAR_EXP002_LABEL} asset] gate_exists:", gate_path.exists())
    print(f"[{SIDECAR_EXP002_LABEL} asset] checkpoints:", checkpoints)
    print(f"[{SIDECAR_EXP002_LABEL} asset] requested_folds:", SIDECAR_EXP002_FOLDS)

    assert float(audio.get("context_seconds", -1)) == float(SIDECAR_EXP002_EXPECTED_CONTEXT_SECONDS), (
        f"Unexpected {SIDECAR_EXP002_LABEL} context_seconds: {audio.get('context_seconds')}"
    )
    assert float(audio.get("target_seconds", -1)) == float(SIDECAR_EXP002_EXPECTED_TARGET_SECONDS), (
        f"Unexpected {SIDECAR_EXP002_LABEL} target_seconds: {audio.get('target_seconds')}"
    )
    assert int(feature.get("image_time", -1)) == int(SIDECAR_EXP002_EXPECTED_IMAGE_TIME), (
        f"Unexpected {SIDECAR_EXP002_LABEL} image_time: {feature.get('image_time')}"
    )

    requested = {f"fold{int(f)}.pt" for f in (SIDECAR_EXP002_FOLDS or [])}
    missing = sorted(requested.difference(checkpoints))
    if missing:
        raise FileNotFoundError(f"Requested {SIDECAR_EXP002_LABEL} checkpoint(s) missing: {missing}")

    if SIDECAR_EXP002_USE_OOF_GATE and SIDECAR_EXP002_GATE_REQUIRE_FILE:
        assert gate_path.exists(), f"OOF gate file missing: {gate_path}"


def _remaining_notebook_time():
    start = globals().get("NOTEBOOK_START_TIME", None)
    limit = globals().get("KAGGLE_LIMIT_SEC", None)
    if start is None or limit is None:
        return None
    return float(limit) - (time.time() - float(start))


def _exp002_allowed_timeout():
    configured = globals().get("SIDECAR_EXP002_TIMEOUT_SEC", None)
    remaining = _remaining_notebook_time()
    if remaining is None:
        return configured

    reserve = float(globals().get("FINAL_RESERVE_SEC", 180))
    allowed_by_notebook = int(remaining - reserve)
    allowed = allowed_by_notebook if configured is None else min(int(configured), allowed_by_notebook)
    print(f"[{SIDECAR_EXP002_LABEL}] remaining notebook time = {remaining:.1f}s")
    print(f"[{SIDECAR_EXP002_LABEL}] allowed subprocess timeout = {allowed}s")
    if allowed < 120:
        raise RuntimeError(
            f"Not enough remaining notebook time for {SIDECAR_EXP002_LABEL} sidecar: "
            f"remaining={remaining:.1f}s, allowed_timeout={allowed}s"
        )
    return allowed


def _exp002_read_sub(path):
    df = pd.read_csv(path)
    assert "row_id" in df.columns, f"row_id missing: {path}"
    assert not any(str(c).startswith("Unnamed") for c in df.columns), f"unnamed columns in {path}"
    df["row_id"] = df["row_id"].astype(str)
    assert df["row_id"].is_unique, f"duplicate row_id: {path}"
    prob_cols = [c for c in df.columns if c != "row_id"]
    assert prob_cols, f"no probability columns in {path}"
    arr = df[prob_cols].to_numpy(np.float32)
    assert np.isfinite(arr).all(), f"NaN/inf in {path}"
    assert arr.min() >= 0.0 and arr.max() <= 1.0, (
        f"values outside [0,1] in {path}: min={arr.min()}, max={arr.max()}"
    )
    out = df.set_index("row_id")
    out.index.name = "row_id"
    return out


def _exp002_rank_cols(df):
    return df.rank(axis=0, method="average", pct=True).astype(np.float32)


def _exp002_topk_mask(arr, k):
    k = int(min(max(k, 1), arr.shape[1]))
    idx = np.argpartition(arr, -k, axis=1)[:, -k:]
    mask = np.zeros(arr.shape, dtype=bool)
    rows = np.arange(arr.shape[0])[:, None]
    mask[rows, idx] = True
    return mask


def _exp002_topk_overlap(A, B, k):
    k = int(min(max(k, 1), A.shape[1]))
    ai = np.argpartition(A, -k, axis=1)[:, -k:]
    bi = np.argpartition(B, -k, axis=1)[:, -k:]
    return float(np.mean([len(set(x).intersection(set(y))) / k for x, y in zip(ai, bi)]))


def _load_exp002_gate_weights(asset_root, class_cols):
    if not SIDECAR_EXP002_USE_OOF_GATE:
        return None
    gate_path = Path(asset_root) / str(SIDECAR_EXP002_GATE_CSV)
    if not gate_path.exists():
        msg = f"{SIDECAR_EXP002_LABEL} OOF gate CSV not found: {gate_path}"
        if SIDECAR_EXP002_GATE_REQUIRE_FILE:
            raise FileNotFoundError(msg)
        print(msg, "Using scalar sidecar weight.")
        return None

    gate = pd.read_csv(gate_path)
    assert "primary_label" in gate.columns, f"{SIDECAR_EXP002_LABEL} gate CSV missing primary_label"
    assert "gate_weight" in gate.columns, f"{SIDECAR_EXP002_LABEL} gate CSV missing gate_weight"
    gate["primary_label"] = gate["primary_label"].astype(str)
    gate["gate_weight"] = gate["gate_weight"].astype(float)
    assert gate["primary_label"].is_unique, f"duplicate primary_label in {SIDECAR_EXP002_LABEL} gate CSV"
    assert np.isfinite(gate["gate_weight"].to_numpy(np.float32)).all(), f"NaN/inf in {SIDECAR_EXP002_LABEL} gate CSV"
    gate_mode = str(gate["gate_mode"].iloc[0]) if "gate_mode" in gate.columns and len(gate) else "unknown"

    weight_map = dict(zip(gate["primary_label"], gate["gate_weight"]))
    missing = [str(col) for col in class_cols if str(col) not in weight_map]
    if missing and SIDECAR_EXP002_GATE_REQUIRE_FILE:
        raise ValueError(
            f"{SIDECAR_EXP002_LABEL} gate CSV is missing {len(missing)} classes. "
            f"First missing: {missing[:10]}"
        )

    weights = np.array([float(weight_map.get(str(col), 0.0)) for col in class_cols], dtype=np.float32)
    weights = np.clip(weights, 0.0, float(SIDECAR_EXP002_GATE_MAX_WEIGHT))
    assert np.isfinite(weights).all(), f"NaN/inf in {SIDECAR_EXP002_LABEL} gate weights"
    enabled_count = int((weights > 0).sum())
    if SIDECAR_EXP002_USE_OOF_GATE and SIDECAR_EXP002_REQUIRE and enabled_count == 0:
        raise ValueError(f"{SIDECAR_EXP002_LABEL} OOF gate has zero enabled classes")

    print(
        f"[{SIDECAR_EXP002_LABEL} gate]",
        f"path={gate_path}",
        f"mode={gate_mode}",
        f"classes={len(weights)}",
        f"enabled={enabled_count}",
        f"missing={len(missing)}",
        f"mean_w={weights.mean():.6f}",
        f"max_w={weights.max():.6f}",
    )
    return weights


def _exp002_masked_rank_blend(anchor_df, side_df, class_weights=None):
    assert anchor_df.index.equals(side_df.index), "sidecar row_id order differs from anchor"
    assert anchor_df.columns.equals(side_df.columns), "sidecar class columns differ from anchor"

    A = _exp002_rank_cols(anchor_df).to_numpy(np.float32)
    S = _exp002_rank_cols(side_df).to_numpy(np.float32)
    mask = _exp002_topk_mask(A, SIDECAR_EXP002_ANCHOR_TOPK) | (
        _exp002_topk_mask(S, SIDECAR_EXP002_SIDE_TOPK) & (A >= float(SIDECAR_EXP002_TAU))
    )

    E = float(np.mean(mask * np.abs(S - A)))
    mode = "scalar"
    nonzero_classes = A.shape[1]
    effective_weight = 0.0
    gate_weight_mean = 0.0
    gate_weight_max = 0.0
    gate_scale = 1.0

    if class_weights is None:
        w_budget = 0.0 if E <= 1e-12 else float(SIDECAR_EXP002_D_BUDGET) / E
        w = min(float(SIDECAR_EXP002_WEIGHT_CAP), w_budget)
        B = A.copy()
        if mask.any() and w > 0:
            B[mask] = A[mask] + w * (S[mask] - A[mask])
        effective_weight = float(w)
        gate_weight_mean = float(w)
        gate_weight_max = float(w)
    else:
        mode = "oof_gated"
        W = np.asarray(class_weights, dtype=np.float32)
        assert W.shape[0] == A.shape[1], "class_weights length mismatch"
        W = np.clip(W, 0.0, float(SIDECAR_EXP002_GATE_MAX_WEIGHT))
        nonzero_classes = int((W > 0).sum())

        WW = np.broadcast_to(W[None, :], A.shape)
        B0 = A.copy()
        if mask.any() and W.max() > 0:
            B0[mask] = A[mask] + WW[mask] * (S[mask] - A[mask])
        D0 = float(np.mean(np.abs(B0 - A)))

        if D0 > float(SIDECAR_EXP002_D_BUDGET) and D0 > 1e-12:
            gate_scale = float(SIDECAR_EXP002_D_BUDGET) / D0
            W = W * gate_scale
            WW = np.broadcast_to(W[None, :], A.shape)

        B = A.copy()
        if mask.any() and W.max() > 0:
            B[mask] = A[mask] + WW[mask] * (S[mask] - A[mask])

        effective_weight = float(W.mean())
        gate_weight_mean = float(W.mean())
        gate_weight_max = float(W.max())

    D = float(np.mean(np.abs(B - A)))
    active = float(mask.mean())
    top3 = _exp002_topk_overlap(A, B, 3)
    top10 = _exp002_topk_overlap(A, B, 10)
    ok = (
        D > 0.0
        and active <= float(SIDECAR_EXP002_MAX_ACTIVE_FRACTION)
        and top3 >= float(SIDECAR_EXP002_MIN_TOP3_OVERLAP)
        and top10 >= float(SIDECAR_EXP002_MIN_TOP10_OVERLAP)
    )

    info = {
        "mode": mode,
        "use_oof_gate": bool(class_weights is not None),
        "weight_cap": float(SIDECAR_EXP002_WEIGHT_CAP),
        "gate_max_weight": float(SIDECAR_EXP002_GATE_MAX_WEIGHT),
        "effective_weight": effective_weight,
        "gate_weight_mean": gate_weight_mean,
        "gate_weight_max": gate_weight_max,
        "gate_nonzero_classes": int(nonzero_classes),
        "gate_scale": float(gate_scale),
        "E": E,
        "D": D,
        "D_budget": float(SIDECAR_EXP002_D_BUDGET),
        "anchor_topk": int(SIDECAR_EXP002_ANCHOR_TOPK),
        "side_topk": int(SIDECAR_EXP002_SIDE_TOPK),
        "tau": float(SIDECAR_EXP002_TAU),
        "active_fraction": active,
        "top3_overlap": top3,
        "top10_overlap": top10,
        "ok": bool(ok),
        "test_files_count": int(globals().get("_EXP002_TEST_FILES_COUNT", -1)),
        "anchor_rows": int(anchor_df.shape[0]),
        "sidecar_rows": int(side_df.shape[0]),
    }
    diag = pd.DataFrame([info])
    diag.to_csv(SIDECAR_EXP002_DIAGNOSTICS_CSV, index=False)
    if not ok:
        return None, diag, info
    blend = pd.DataFrame(B, index=anchor_df.index, columns=anchor_df.columns)
    blend.index.name = "row_id"
    return blend, diag, info


def _run_exp002_inference(asset_root, base_dir):
    cmd = [
        sys.executable,
        str(asset_root / "infer.py"),
        "--data-dir",
        str(base_dir),
        "--checkpoint-dir",
        str(asset_root),
        "--config",
        str(asset_root / "config.yaml"),
        "--output",
        str(SIDECAR_EXP002_SUBM),
        "--device",
        str(SIDECAR_EXP002_DEVICE),
        "--batch-size",
        str(SIDECAR_EXP002_BATCH_SIZE),
    ]
    folds = list(SIDECAR_EXP002_FOLDS or [])
    if folds:
        cmd += ["--folds"] + [str(f) for f in folds]
    timeout = _exp002_allowed_timeout()
    print(f"Running {SIDECAR_EXP002_LABEL} sidecar inference:", " ".join(cmd))
    subprocess.run(
        cmd,
        check=True,
        timeout=timeout,
    )
    if not Path(SIDECAR_EXP002_SUBM).exists():
        raise FileNotFoundError(f"{SIDECAR_EXP002_LABEL} sidecar output was not created: {SIDECAR_EXP002_SUBM}")


def _exp002_rank_delta_summary(base_csv, final_csv):
    base = _exp002_read_sub(base_csv)
    final = _exp002_read_sub(final_csv).loc[base.index, base.columns]
    A = _exp002_rank_cols(base).to_numpy(np.float32)
    B = _exp002_rank_cols(final).to_numpy(np.float32)
    return {
        "D_vs_pre_sidecar_anchor": float(np.mean(np.abs(B - A))),
        "top3_overlap_vs_pre_sidecar_anchor": _exp002_topk_overlap(A, B, 3),
        "top10_overlap_vs_pre_sidecar_anchor": _exp002_topk_overlap(A, B, 10),
        "rows": int(base.shape[0]),
        "classes": int(base.shape[1]),
    }


def _apply_exp002_current_variant():
    t0 = time.time()
    try:
        asset_root = _find_exp002_asset_root()
        base_dir = _exp002_find_competition_dir()
        if asset_root is None:
            raise FileNotFoundError(f"{SIDECAR_EXP002_LABEL} sidecar asset not found; attach the configured sidecar dataset.")
        if base_dir is None:
            raise FileNotFoundError(f"BirdCLEF competition data not found for {SIDECAR_EXP002_LABEL} sidecar inference.")

        print(f"{SIDECAR_EXP002_LABEL} asset:", asset_root)
        print("competition data:", base_dir)
        _print_exp002_asset_sanity(asset_root)
        pd.read_csv("submission.csv").to_csv(SIDECAR_EXP002_ANCHOR_BEFORE_CSV, index=False)

        test_paths = _exp002_test_soundscape_paths(base_dir)
        globals()["_EXP002_TEST_FILES_COUNT"] = len(test_paths)
        if not test_paths:
            reason = f"No test_soundscapes .ogg files found; public/dry-run anchor rows cannot be matched by {SIDECAR_EXP002_LABEL} inference."
            info = _exp002_write_skip_diag(reason, mode="dry_run", applicable=False)
            print(reason, "Keeping current submission.")
            print(pd.DataFrame([info]).to_string(index=False))
            return False

        if SIDECAR_EXP002_FORCE_INFER:
            Path(SIDECAR_EXP002_SUBM).unlink(missing_ok=True)
            _run_exp002_inference(asset_root, base_dir)
        elif not Path(SIDECAR_EXP002_SUBM).exists():
            _run_exp002_inference(asset_root, base_dir)
        else:
            print(f"Using existing {SIDECAR_EXP002_SUBM}")

        anchor = _exp002_read_sub("submission.csv")
        side_raw = _exp002_read_sub(SIDECAR_EXP002_SUBM)
        missing_rows = anchor.index.difference(side_raw.index)
        extra_rows = side_raw.index.difference(anchor.index)
        missing_cols = [col for col in anchor.columns if col not in side_raw.columns]
        if len(missing_rows) or len(extra_rows) or missing_cols:
            raise ValueError(
                f"{SIDECAR_EXP002_LABEL} row/column mismatch with real test_soundscapes present: "
                f"missing_rows={len(missing_rows)}, extra_rows={len(extra_rows)}, "
                f"missing_cols={len(missing_cols)}"
            )
        side = side_raw.loc[anchor.index, anchor.columns]
        class_weights = _load_exp002_gate_weights(asset_root, anchor.columns.tolist())
        blend, diag, info = _exp002_masked_rank_blend(anchor, side, class_weights=class_weights)
        print(diag.to_string(index=False))
        if blend is None:
            msg = f"{SIDECAR_EXP002_LABEL} sidecar was computed but rejected by guards: {info}"
            if SIDECAR_EXP002_REQUIRE:
                raise RuntimeError(msg)
            print(msg, "Keeping current submission.")
            return False

        final_submission = write_final_submission(blend, "submission.csv")
        print(f"Applied {SIDECAR_EXP002_LABEL} sidecar:", info)
        return True
    except Exception as exc:
        if SIDECAR_EXP002_REQUIRE:
            raise
        print(f"{SIDECAR_EXP002_LABEL} sidecar failed; keeping current submission:", repr(exc))
        return False
    finally:
        print(f"{SIDECAR_EXP002_LABEL} sidecar elapsed: {time.time() - t0:.1f}s")


if not RUN_EXP002_SIDECAR:
    print(
        f"PCEN/ConvNeXt sidecar disabled; configured mode={SIDECAR_EXP002_VARIANT}, "
        f"active_variants={SIDECAR_EXP002_ACTIVE_VARIANTS}. Keeping current submission."
    )
else:
    _dual_mode = len(SIDECAR_EXP002_ACTIVE_VARIANTS) > 1
    _dual_anchor_csv = Path("submission_before_exp002_dual_sidecar.csv")
    if _dual_mode:
        pd.read_csv("submission.csv").to_csv(_dual_anchor_csv, index=False)
        print(f"Dual PCEN sidecar mode active: {SIDECAR_EXP002_ACTIVE_VARIANTS}")

    _applied_variants = []
    for _variant in SIDECAR_EXP002_ACTIVE_VARIANTS:
        _load_exp002_variant_settings(_variant)
        print(f"=== PCEN sidecar variant: {_variant} ({SIDECAR_EXP002_LABEL}) ===")
        _applied = _apply_exp002_current_variant()
        if _applied:
            _applied_variants.append(_variant)

    if _dual_mode:
        summary = _exp002_rank_delta_summary(_dual_anchor_csv, "submission.csv")
        summary.update({
            "mode": "dual_exp002b_exp002",
            "requested_variants": ",".join(SIDECAR_EXP002_ACTIVE_VARIANTS),
            "applied_variants": ",".join(_applied_variants),
            "dual_total_D_budget": SIDECAR_EXP002_DUAL_TOTAL_D_BUDGET,
            "ok": True,
        })
        if SIDECAR_EXP002_DUAL_TOTAL_D_BUDGET is not None:
            summary["ok"] = summary["D_vs_pre_sidecar_anchor"] <= float(SIDECAR_EXP002_DUAL_TOTAL_D_BUDGET)
        pd.DataFrame([summary]).to_csv(SIDECAR_EXP002_DUAL_DIAGNOSTICS_CSV, index=False)
        print(pd.DataFrame([summary]).to_string(index=False))
        if not summary["ok"]:
            msg = f"Dual PCEN sidecars exceeded total D budget: {summary}"
            if SIDECAR_EXP002_REQUIRE:
                raise RuntimeError(msg)
            print(msg, "Restoring pre-sidecar anchor.")
            pd.read_csv(_dual_anchor_csv).to_csv("submission.csv", index=False)


## Final Output Preview

This reads the actual `submission.csv` after all enabled rank corrections.


In [ ]:
final = pd.read_csv("submission.csv")
sample_path = _find_sample_submission_path()
if sample_path is not None:
    sample = pd.read_csv(sample_path)
    assert final.columns.tolist() == sample.columns.tolist(), "final columns differ from sample_submission"
    assert final["row_id"].astype(str).tolist() == sample["row_id"].astype(str).tolist(), "final row_id order differs from sample_submission"
prob_cols = [c for c in final.columns if c != "row_id"]
values = final[prob_cols].to_numpy(np.float32)
assert "row_id" in final.columns
assert final["row_id"].is_unique
assert not any(str(c).startswith("Unnamed") for c in final.columns)
assert np.isfinite(values).all()
assert values.min() >= 0.0 and values.max() <= 1.0
print({
    "rows": len(final),
    "cols": final.shape[1],
    "class_cols": len(prob_cols),
    "min": float(values.min()),
    "max": float(values.max()),
    "sample_aligned": sample_path is not None,
})

base_path = Path("submission_before_all_sidecars.csv")
if base_path.exists():
    base = pd.read_csv(base_path).set_index("row_id")
    final_rank = final.set_index("row_id")[base.columns].rank(axis=0, method="average", pct=True).astype(np.float32)
    base_rank = base.rank(axis=0, method="average", pct=True).astype(np.float32)
    def _preview_topk_overlap(A_df, B_df, k):
        A = A_df.to_numpy(np.float32)
        B = B_df.to_numpy(np.float32)
        idx_a = np.argpartition(A, -k, axis=1)[:, -k:]
        idx_b = np.argpartition(B, -k, axis=1)[:, -k:]
        return float(np.mean([len(set(a).intersection(set(b))) / k for a, b in zip(idx_a, idx_b)]))
    print({
        "final_D_vs_base_anchor": float(np.mean(np.abs(final_rank.to_numpy(np.float32) - base_rank.to_numpy(np.float32)))),
        "final_top3_overlap_vs_base": _preview_topk_overlap(base_rank, final_rank, 3),
        "final_top10_overlap_vs_base": _preview_topk_overlap(base_rank, final_rank, 10),
    })
final.head(3)


## Optional Reference Output Check

This compares the current final `submission.csv` against a reference CSV. If the reference is the raw EoS9 anchor, disable `RUN_BIRDNET_SIDECAR` and `RUN_EXP002_SIDECAR` first. With sidecars enabled, a non-zero diff is expected because the final file is intentionally rank-corrected.


In [ ]:
ORIGINAL_EOS9_SUBMISSION = None  # e.g. "/kaggle/input/original-eos9-output/submission.csv"

if ORIGINAL_EOS9_SUBMISSION:
    import numpy as np
    import pandas as pd

    old = pd.read_csv(ORIGINAL_EOS9_SUBMISSION).set_index("row_id")
    new = pd.read_csv("submission.csv").set_index("row_id")

    assert old.index.equals(new.index), "row_id order differs from original EoS9 output"
    assert old.columns.equals(new.columns), "class columns differ from original EoS9 output"

    diff = np.abs(old.to_numpy(np.float32) - new.to_numpy(np.float32))
    print("max_abs_diff:", float(diff.max()))
    print("mean_abs_diff:", float(diff.mean()))
    print("p99_abs_diff:", float(np.quantile(diff, 0.99)))
else:
    print("Reference EoS9 submission not provided; output check skipped.")
